# Hay & Nicolay Fine-Tuning Dataset Generator (v4)
### Hay (Retrieval Director) · Nicolay (Response Synthesizer)

**Project:** Nicolay — A Transparent RAG System for Lincoln Corpus Exploration  
**Article:** *Hutchinson, Daniel. "Nicolay: Transparent Retrieval-Augmented Generation for Historical Corpus Exploration." Digital Humanities Quarterly (forthcoming).*  
**GitHub:** [github.com/Dr-Hutchinson/nicolay](https://github.com/Dr-Hutchinson/nicolay)

---

## About This Notebook

This notebook generates, verifies, and approves new fine-tuning examples for both model components of the Nicolay RAG system: **Hay**, the retrieval director that produces weighted keyword sets and a HyDE (Hypothetical Document Embedding) initial answer, and **Nicolay**, the response synthesizer that reasons over retrieved corpus passages using a structured chain-of-thought schema.

Each section targets a specific failure mode identified through benchmark evaluation and dev-log analysis. The generate→verify→approve workflow runs inline per section, allowing human review at each stage before examples are committed to the dataset. A contamination tracker records which corpus chunks and queries were used in generation, enabling their exclusion from formal benchmark evaluation.

**Note on this documentation:** The expanded commentary in this public version — including this header, the model rationale note in Cell 0.1, the embedded dataset note in Cell 0.4, and structural annotations throughout — was drafted with the assistance of Claude (Anthropic), specifically Claude Sonnet 4.6, as part of article preparation. Code, prompts, dataset content, and editorial decisions are the author's own.

---

## Section Map

| Section | Task | Target Model |
|---|---|---|
| 0 | Setup & data loading | — |
| 1 | Synthesis discrimination | Hay |
| 2 | HyDE hypothesis framing | Hay |
| 3 | Race/citizenship sensitivity | Hay + Nicolay |
| 4 | Refusal recalibration | Hay |
| 5 | Hypothesis testing (corpus > initial_answer) | Nicolay |
| 6 | Single-document selective citation | Nicolay |
| 7 | Export & contamination report | — |

Sections 1, 2, and 4 target Hay exclusively. Sections 5 and 6 target Nicolay exclusively. Section 3 generates examples for both models around the same historiographically sensitive topic cluster.

---

## Design Principles

- **Option C priming:** Each generation prompt includes one high-quality format example (the "primer") plus explicit task instructions — enough to establish output schema without over-determining content.
- **Inline generate→verify→approve:** Each section runs generation, prints a verification display, and requires an explicit approval flag before examples are added to the output lists. No example enters the dataset silently.
- **Contamination tracking throughout:** Every chunk ID and query used in generation is logged. The Section 7 contamination report produces the exclusion list for benchmark evaluation.
- **Voyant-calibrated keywords:** Rare-term hints drawn from Voyant Tools corpus frequency analysis (63,489 words, 7,563 unique terms) are injected into generation prompts to guide keyword selection toward corpus-discriminating terms. The frequency table is embedded directly in Cell 0.3 — no external file required.

---

## Files Needed Before Running

- `hay_finetuning_dataset_fixed.jsonl` — repaired Hay v3 dataset (51 examples, 4 schema bugs corrected; see Cell 0.4)
- `nicolay_finetuning_data.jsonl` — current Nicolay dataset
- `lincoln_speech_corpus_repaired.json` — re-indexed Lincoln corpus JSON

*(Voyant word counts are embedded directly in Cell 0.3 — no upload required.)*

---

## API Requirements

- **Model:** `gpt-4o` (see Cell 0.1 for rationale)
- **Estimated cost:** Low — this notebook makes one API call per fine-tuning candidate (roughly 10–20 calls across all sections). Total cost typically under $0.50 USD.
- Set `OPENAI_API_KEY` as an environment variable (e.g., via Colab Secrets) before running.


---
## Section 0 — Setup & Data Loading

In [ ]:
# 0.1  Install / import
!pip install openai -q

import json, re, textwrap
from pathlib import Path
from openai import OpenAI
from IPython.display import display, HTML
import pandas as pd

# API key: set OPENAI_API_KEY as an environment variable (e.g., via Colab Secrets → "Add new secret").
# Do not hardcode keys in the notebook.
import os  # needed for os.environ.get below
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

# GENERATION_MODEL: gpt-4o is used here (not gpt-4o-mini) because this notebook
# produces TRAINING DATA. Output quality directly shapes fine-tuned model behavior,
# so the more capable model is appropriate despite higher cost per call.
# This is the inverse of the indexing notebook's metadata generation (Cell 7 of
# nicolay_reindex_5b.ipynb), which used gpt-4o-mini for cost efficiency on
# ~882 lower-stakes summarization calls.
GENERATION_MODEL = "gpt-4o"


In [ ]:
# 0.2  Voyant frequency table (embedded — no file upload needed)
# Corpus: 63,489 words · 7,563 unique terms
# Source: voyant_word_counts_complete.json

TOTAL_CORPUS_WORDS = 63489

# Full frequency dict: term (lowercase) → rawFreq
VOYANT_FREQ = {"slavery": 636, "judge": 606, "states": 580, "people": 435, "douglas": 419, "say": 372, "constitution": 361, "union": 334, "shall": 330, "state": 328, "right": 326, "government": 319, "question": 306, "slave": 300, "great": 266, "men": 262, "time": 256, "man": 254, "congress": 254, "said": 253, "free": 250, "country": 248, "new": 237, "united": 228, "territory": 209, "law": 208, "make": 198, "bill": 192, "missouri": 191, "way": 180, "think": 180, "nebraska": 179, "compromise": 179, "slaves": 170, "speech": 169, "know": 169, "principle": 165, "mr": 164, "war": 150, "wrong": 148, "public": 146, "north": 144, "good": 143, "says": 142, "let": 142, "subject": 140, "national": 140, "south": 138, "decision": 135, "laughter": 130, "years": 129, "power": 129, "fact": 128, "old": 127, "day": 126, "just": 125, "year": 123, "trumbull": 123, "true": 121, "far": 118, "territories": 117, "line": 116, "ask": 114, "provision": 112, "believe": 111, "regard": 108, "general": 108, "friends": 108, "case": 108, "answer": 107, "policy": 106, "white": 104, "passed": 103, "come": 103, "institution": 100, "end": 100, "political": 99, "party": 99, "court": 99, "property": 98, "vote": 97, "present": 97, "clay": 97, "applause": 97, "show": 96, "nation": 95, "having": 95, "charge": 95, "argument": 94, "attention": 90, "repeal": 89, "point": 89, "matter": 88, "favor": 88, "hold": 87, "world": 86, "act": 85, "place": 84, "long": 84, "read": 83, "like": 82, "federal": 82, "evidence": 82, "declaration": 81, "voted": 79, "little": 79, "course": 79, "convention": 79, "came": 79, "laws": 78, "constitutional": 78, "brought": 78, "purpose": 76, "peace": 76, "illinois": 76, "fathers": 76, "citizens": 76, "thought": 75, "authority": 75, "reason": 74, "trade": 73, "view": 72, "wish": 71, "resolutions": 71, "opinion": 70, "making": 70, "better": 70, "negro": 69, "liberty": 69, "half": 69, "emancipation": 69, "called": 69, "self": 68, "mexico": 68, "held": 68, "cheers": 68, "senate": 67, "lincoln": 67, "life": 67, "got": 66, "supreme": 65, "negroes": 65, "legislation": 65, "forward": 65, "taken": 64, "stand": 64, "understand": 63, "large": 63, "house": 63, "equal": 63, "rights": 62, "left": 62, "tell": 61, "support": 61, "ground": 61, "condition": 61, "washington": 60, "set": 60, "scott": 60, "independence": 60, "exclude": 60, "agitation": 60, "service": 59, "dred": 59, "language": 58, "truth": 57, "principles": 57, "persons": 57, "measures": 57, "means": 57, "times": 56, "placed": 56, "object": 56, "different": 56, "beginning": 56, "territorial": 55, "number": 55, "district": 55, "quite": 54, "moral": 54, "mind": 54, "history": 54, "decided": 54, "certain": 54, "want": 53, "portion": 53, "hope": 53, "original": 52, "labor": 52, "foreign": 52, "suppose": 51, "president": 51, "kansas": 51, "election": 51, "best": 51, "army": 51, "proposition": 50, "ought": 50, "measure": 50, "deny": 50, "adopted": 50, "necessity": 49, "taking": 48, "republican": 48, "questions": 48, "proper": 48, "amendment": 48, "words": 47, "things": 47, "framed": 47, "form": 47, "existing": 47, "become": 47, "west": 46, "unless": 46, "sacred": 46, "occasion": 46, "limits": 46, "hand": 46, "given": 46, "difference": 46, "american": 46, "stated": 45, "spirit": 45, "responsible": 45, "precisely": 45, "majority": 45, "control": 45, "consider": 45, "black": 45, "springfield": 44, "possible": 44, "land": 44, "interest": 44, "intended": 44, "forgery": 44, "effect": 44, "cause": 44, "ago": 44, "action": 44, "word": 43, "southern": 43, "really": 43, "prove": 43, "known": 43, "henry": 43, "understood": 42, "speeches": 42, "sort": 42, "representatives": 42, "report": 42, "necessary": 42, "nearly": 42, "force": 42, "declared": 42, "controversy": 42, "senator": 41, "position": 41, "legislature": 41, "justice": 41, "equality": 41, "doing": 41, "contrary": 41, "consideration": 41, "admission": 41, "terms": 40, "sentiment": 40, "probably": 40, "platform": 40, "parties": 40, "human": 40, "fully": 40, "department": 40, "bring": 40, "used": 39, "understanding": 39, "navy": 39, "members": 39, "future": 39, "extension": 39, "expressly": 39, "existence": 39, "continue": 39, "texas": 38, "secretary": 38, "rebellion": 38, "points": 38, "opposed": 38, "military": 38, "merely": 38, "increase": 38, "important": 38, "express": 38, "care": 38, "believed": 38, "belief": 38, "virginia": 37, "took": 37, "side": 37, "owned": 37, "ordinance": 37, "fugitive": 37, "extent": 37, "douglas's": 37, "doubt": 37, "doctrine": 37, "days": 37, "clear": 37, "cases": 37, "carolina": 37, "utah": 36, "sense": 36, "received": 36, "opinions": 36, "formed": 36, "claim": 36, "actual": 36, "views": 35, "till": 35, "spread": 35, "session": 35, "rule": 35, "return": 35, "proposed": 35, "mere": 35, "mean": 35, "length": 35, "interrogatories": 35, "hands": 35, "established": 35, "consent": 35, "common": 35, "body": 35, "use": 34, "turn": 34, "second": 34, "reply": 34, "relation": 34, "prohibition": 34, "popular": 34, "period": 34, "pass": 34, "loud": 34, "desire": 34, "countries": 34, "continued": 34, "connection": 34, "1850": 34, "work": 33, "soon": 33, "section": 33, "race": 33, "propose": 33, "proclamation": 33, "organization": 33, "live": 33, "kentucky": 33, "issue": 33, "insist": 33, "freedom": 33, "following": 33, "especially": 33, "entirely": 33, "constantly": 33, "asked": 33, "struggle": 32, "society": 32, "result": 32, "leave": 32, "including": 32, "heard": 32, "gave": 32, "duty": 32, "able": 32, "voting": 31, "submitted": 31, "showing": 31, "saying": 31, "powers": 31, "places": 31, "office": 31, "institutions": 31, "god": 31, "friend": 31, "faith": 31, "executive": 31, "direct": 31, "congressional": 31, "answered": 31, "adoption": 31, "admit": 31, "statement": 30, "single": 30, "prevent": 30, "population": 30, "organized": 30, "legal": 30, "judgment": 30, "forever": 30, "evil": 30, "entire": 30, "danger": 30, "character": 30, "carry": 30, "basis": 30, "attempt": 30, "yes": 29, "treasury": 29, "social": 29, "provisions": 29, "plan": 29, "pay": 29, "naval": 29, "nations": 29, "met": 29, "meeting": 29, "meet": 29, "living": 29, "judge's": 29, "giving": 29, "expressed": 29, "domestic": 29, "claims": 29, "change": 29, "boundary": 29, "beyond": 29, "away": 29, "african": 29, "administration": 29, "acts": 29, "vast": 28, "treaty": 28, "saw": 28, "repeat": 28, "reference": 28, "real": 28, "provided": 28, "parts": 28, "open": 28, "maintain": 28, "hour": 28, "fairly": 28, "elected": 28, "efforts": 28, "democratic": 28, "cries": 28, "columbia": 28, "class": 28, "acquisition": 28, "western": 27, "ultimate": 27, "thousand": 27, "mentioned": 27, "high": 27, "going": 27, "gives": 27, "finally": 27, "divided": 27, "debt": 27, "certainly": 27, "article": 27, "altogether": 27, "additional": 27, "satisfactory": 26, "remember": 26, "remaining": 26, "promise": 26, "perfect": 26, "oppose": 26, "march": 26, "makes": 26, "local": 26, "knew": 26, "introduced": 26, "facts": 26, "face": 26, "directly": 26, "destroy": 26, "demand": 26, "decide": 26, "death": 26, "carried": 26, "audience": 26, "adopt": 26, "abolition": 26, "votes": 25, "stricken": 25, "settlement": 25, "proviso": 25, "presence": 25, "practice": 25, "originally": 25, "opposition": 25, "northern": 25, "needed": 25, "nature": 25, "lecompton": 25, "leaving": 25, "judges": 25, "included": 25, "idea": 25, "fort": 25, "false": 25, "exists": 25, "exactly": 25, "element": 25, "dead": 25, "civil": 25, "chief": 25, "annual": 25, "advantage": 25, "admitted": 25, "toombs": 24, "told": 24, "strong": 24, "speak": 24, "sovereignty": 24, "save": 24, "require": 24, "remain": 24, "races": 24, "prosperity": 24, "prohibit": 24, "plainly": 24, "piece": 24, "perfectly": 24, "ones": 24, "member": 24, "june": 24, "jefferson": 24, "instrument": 24, "greater": 24, "goes": 24, "fair": 24, "division": 24, "difficulty": 24, "declare": 24, "date": 24, "created": 24, "conclusion": 24, "compromises": 24, "bound": 24, "blow": 24, "address": 24, "somewhat": 23, "short": 23, "senators": 23, "revolution": 23, "respect": 23, "reported": 23, "renewed": 23, "proof": 23, "presented": 23, "past": 23, "order": 23, "ohio": 23, "money": 23, "look": 23, "kept": 23, "introduction": 23, "follow": 23, "deal": 23, "coming": 23, "clause": 23, "california": 23, "america": 23, "agree": 23, "africa": 23, "acquired": 23, "according": 23, "trumbull's": 22, "sufficient": 22, "spoken": 22, "shows": 22, "shown": 22, "sentiments": 22, "seen": 22, "restriction": 22, "rest": 22, "republicans": 22, "republic": 22, "receipts": 22, "protection": 22, "particular": 22, "owners": 22, "ottawa": 22, "insisted": 22, "home": 22, "holding": 22, "head": 22, "feeling": 22, "considered": 22, "close": 22, "cease": 22, "capital": 22, "arguments": 22, "allow": 22, "affairs": 22, "whig": 21, "surely": 21, "substantially": 21, "struck": 21, "soldiers": 21, "river": 21, "prohibited": 21, "practical": 21, "person": 21, "notice": 21, "needs": 21, "mississippi": 21, "influence": 21, "getting": 21, "follows": 21, "fight": 21, "establish": 21, "distinctly": 21, "destruction": 21, "courts": 21, "considerable": 21, "choose": 21, "alike": 21, "add": 21, "title": 20, "thirty": 20, "sure": 20, "supposed": 20, "sum": 20, "spoke": 20, "speaking": 20, "seven": 20, "scarcely": 20, "run": 20, "mutual": 20, "man's": 20, "knows": 20, "honor": 20, "generally": 20, "followed": 20, "fit": 20, "extend": 20, "equally": 20, "enemy": 20, "county": 20, "circumstances": 20, "chicago": 20, "agreed": 20, "1854": 20, "wilmot": 19, "willing": 19, "went": 19, "ways": 19, "universal": 19, "term": 19, "submit": 19, "small": 19, "shape": 19, "saving": 19, "relations": 19, "rapidly": 19, "putting": 19, "pledged": 19, "oath": 19, "maine": 19, "louisiana": 19, "lawful": 19, "lanphier": 19, "lands": 19, "interior": 19, "indian": 19, "increased": 19, "importance": 19, "heretofore": 19, "governments": 19, "gentlemen": 19, "fraud": 19, "forth": 19, "fellow": 19, "extinction": 19, "experience": 19, "exact": 19, "elections": 19, "effort": 19, "earth": 19, "delivered": 19, "commerce": 19, "comes": 19, "clay's": 19, "buy": 19, "britain": 19, "bringing": 19, "began": 19, "appear": 19, "aggregate": 19, "wise": 18, "whites": 18, "vessels": 18, "valuable": 18, "settled": 18, "revenue": 18, "restoration": 18, "operations": 18, "offered": 18, "near": 18, "months": 18, "month": 18, "millions": 18, "meant": 18, "meaning": 18, "main": 18, "looks": 18, "likely": 18, "leading": 18, "john": 18, "instance": 18, "greatly": 18, "gradual": 18, "field": 18, "fatal": 18, "fail": 18, "extended": 18, "expected": 18, "exist": 18, "execution": 18, "editor": 18, "east": 18, "discussion": 18, "cost": 18, "contest": 18, "construction": 18, "colored": 18, "century": 18, "born": 18, "balance": 18, "attempted": 18, "assumed": 18, "articles": 18, "appeal": 18, "allowed": 18, "aid": 18, "age": 18, "affirmed": 18, "trust": 17, "tried": 17, "thinks": 17, "thereof": 17, "tendency": 17, "takes": 17, "remove": 17, "provide": 17, "progress": 17, "plain": 17, "personal": 17, "peculiar": 17, "passing": 17, "paid": 17, "opponents": 17, "officers": 17, "obligations": 17, "notwithstanding": 17, "moment": 17, "mistake": 17, "maintained": 17, "july": 17, "introduce": 17, "intercourse": 17, "insurrection": 17, "individual": 17, "help": 17, "heart": 17, "harris": 17, "final": 17, "extract": 17, "expect": 17, "existed": 17, "entitled": 17, "enemies": 17, "demanded": 17, "conspiracy": 17, "cast": 17, "carefully": 17, "arms": 17, "added": 17, "1st": 17, "1861": 17, "written": 16, "wholly": 16, "wants": 16, "superior": 16, "source": 16, "simply": 16, "sectional": 16, "required": 16, "remains": 16, "region": 16, "record": 16, "reached": 16, "quotation": 16, "purposes": 16, "poor": 16, "occasions": 16, "obligation": 16, "numbers": 16, "minnesota": 16, "knowledge": 16, "intention": 16, "indiana": 16, "fixed": 16, "fighting": 16, "feel": 16, "farther": 16, "failed": 16, "extraordinary": 16, "engaged": 16, "dividing": 16, "disturb": 16, "distinguished": 16, "distant": 16, "disregard": 16, "difficulties": 16, "declaring": 16, "countrymen": 16, "confidence": 16, "citizen": 16, "charleston": 16, "calling": 16, "british": 16, "brings": 16, "became": 16, "asks": 16, "actually": 16, "acted": 16, "absolutely": 16, "yield": 15, "variety": 15, "try": 15, "temporary": 15, "tells": 15, "sustain": 15, "substance": 15, "statesman": 15, "sought": 15, "similar": 15, "silent": 15, "safely": 15, "rid": 15, "recommend": 15, "ready": 15, "propositions": 15, "previous": 15, "peaceful": 15, "objection": 15, "newspaper": 15, "need": 15, "mode": 15, "mexican": 15, "loyal": 15, "letter": 15, "led": 15, "intelligent": 15, "insurgents": 15, "information": 15, "individuals": 15, "inasmuch": 15, "impossible": 15, "globe": 15, "glad": 15, "france": 15, "fault": 15, "falsehood": 15, "fall": 15, "ending": 15, "early": 15, "disposed": 15, "difficult": 15, "democrats": 15, "dangerous": 15, "consequently": 15, "commend": 15, "chase": 15, "ceased": 15, "business": 15, "assumes": 15, "arkansas": 15, "application": 15, "afterward": 15, "worth": 14, "venture": 14, "treaties": 14, "test": 14, "tennessee": 14, "suggest": 14, "success": 14, "subjects": 14, "story": 14, "rightful": 14, "responsibility": 14, "repudiated": 14, "regarded": 14, "receive": 14, "reading": 14, "purchase": 14, "published": 14, "prohibiting": 14, "produce": 14, "presidential": 14, "permitted": 14, "origin": 14, "ordinary": 14, "opportunity": 14, "operation": 14, "message": 14, "limited": 14, "lay": 14, "larger": 14, "kind": 14, "judicial": 14, "joint": 14, "iowa": 14, "interrogatory": 14, "interfere": 14, "implied": 14, "immediate": 14, "hear": 14, "freeport": 14, "fourth": 14, "favorable": 14, "extreme": 14, "expenditures": 14, "exercise": 14, "europe": 14, "enter": 14, "distinct": 14, "derived": 14, "denied": 14, "defy": 14, "deemed": 14, "december": 14, "contained": 14, "commercial": 14, "colonies": 14, "choice": 14, "charges": 14, "branch": 14, "bonds": 14, "blacks": 14, "aware": 14, "avoid": 14, "argue": 14, "applied": 14, "answers": 14, "affect": 14, "1820": 14, "veracity": 13, "trying": 13, "truly": 13, "troops": 13, "treat": 13, "suppress": 13, "sumter": 13, "submitting": 13, "strength": 13, "steadily": 13, "stands": 13, "specific": 13, "secure": 13, "running": 13, "results": 13, "resources": 13, "recommendation": 13, "reach": 13, "properly": 13, "produced": 13, "proceedings": 13, "privilege": 13, "patriotic": 13, "particularly": 13, "paper": 13, "owner": 13, "opening": 13, "necessarily": 13, "natural": 13, "memory": 13, "matters": 13, "love": 13, "lies": 13, "late": 13, "insurgent": 13, "indispensable": 13, "independent": 13, "identical": 13, "gone": 13, "fifth": 13, "example": 13, "examine": 13, "efficiency": 13, "disbursements": 13, "determined": 13, "concerned": 13, "committee": 13, "colonization": 13, "coast": 13, "cheering": 13, "changed": 13, "blame": 13, "bills": 13, "assert": 13, "americans": 13, "amendments": 13, "accepted": 13, "abolitionists": 13, "abide": 13, "voice": 12, "various": 12, "unfriendly": 12, "turns": 12, "thrown": 12, "talk": 12, "suppression": 12, "successful": 12, "succeed": 12, "sound": 12, "soil": 12, "separation": 12, "separate": 12, "secured": 12, "sea": 12, "safety": 12, "respectively": 12, "respective": 12, "resolved": 12, "requires": 12, "replied": 12, "reduced": 12, "reasonable": 12, "quiet": 12, "quarter": 12, "purchased": 12, "proved": 12, "printed": 12, "preceding": 12, "portions": 12, "perceived": 12, "perceive": 12, "patriotism": 12, "occupy": 12, "noble": 12, "mouth": 12, "motives": 12, "minutes": 12, "mention": 12, "masters": 12, "loans": 12, "lie": 12, "later": 12, "knowing": 12, "interests": 12, "instead": 12, "inconsistent": 12, "hate": 12, "greatest": 12, "gentleman": 12, "framers": 12, "forget": 12, "fiscal": 12, "exercised": 12, "event": 12, "estimate": 12, "easy": 12, "easily": 12, "double": 12, "devoted": 12, "dealing": 12, "dare": 12, "cut": 12, "copy": 12, "constitutions": 12, "connected": 12, "conflict": 12, "complete": 12, "commenced": 12, "clearly": 12, "capacity": 12, "canvass": 12, "candidate": 12, "broad": 12, "break": 12, "blood": 12, "believes": 12, "behalf": 12, "bear": 12, "battle": 12, "appropriation": 12, "apply": 12, "acquiring": 12, "acknowledge": 12, "absolute": 12, "abolish": 12, "1863": 12, "yielded": 11, "whigs": 11, "wanted": 11, "vital": 11, "violence": 11, "value": 11, "uses": 11, "twice": 11, "trouble": 11, "total": 11, "systems": 11, "sympathy": 11, "stood": 11, "st": 11, "special": 11, "speaker": 11, "sources": 11, "sooner": 11, "sir": 11, "share": 11, "settlements": 11, "send": 11, "revolutionary": 11, "resist": 11, "remark": 11, "register": 11, "refer": 11, "reasons": 11, "readily": 11, "protect": 11, "preserve": 11, "ports": 11, "perpetual": 11, "pacific": 11, "official": 11, "november": 11, "names": 11, "miles": 11, "mighty": 11, "material": 11, "maryland": 11, "longer": 11, "logically": 11, "lives": 11, "leaves": 11, "lawfully": 11, "laborers": 11, "join": 11, "january": 11, "james": 11, "jackson": 11, "investigation": 11, "interested": 11, "insisting": 11, "inferior": 11, "increasing": 11, "immediately": 11, "grave": 11, "genius": 11, "generation": 11, "frame": 11, "forces": 11, "forbid": 11, "firm": 11, "fifty": 11, "feelings": 11, "excluding": 11, "enthusiastic": 11, "endure": 11, "eloquence": 11, "egypt": 11, "divide": 11, "disturbed": 11, "discovered": 11, "differently": 11, "devotion": 11, "democrat": 11, "democracy": 11, "defeated": 11, "deeply": 11, "decisions": 11, "dared": 11, "crisis": 11, "confident": 11, "confess": 11, "conference": 11, "compensation": 11, "community": 11, "climate": 11, "classes": 11, "claimed": 11, "chosen": 11, "cent": 11, "career": 11, "brief": 11, "borne": 11, "avowed": 11, "authorized": 11, "authorize": 11, "august": 11, "attend": 11, "assertion": 11, "assembled": 11, "arrangement": 11, "argues": 11, "argued": 11, "apparently": 11, "apparent": 11, "allowing": 11, "advanced": 11, "advance": 11, "adjustment": 11, "account": 11, "1803": 11, "withhold": 10, "voters": 10, "valid": 10, "uniform": 10, "text": 10, "tending": 10, "surplus": 10, "subsequent": 10, "spain": 10, "sides": 10, "september": 10, "roars": 10, "restore": 10, "respectfully": 10, "repeated": 10, "remarks": 10, "recognized": 10, "recognize": 10, "rapid": 10, "propounded": 10, "proportion": 10, "prior": 10, "print": 10, "principal": 10, "presume": 10, "prepared": 10, "precise": 10, "practically": 10, "possession": 10, "pointed": 10, "plea": 10, "pieces": 10, "perpetuate": 10, "permit": 10, "permanent": 10, "pennsylvania": 10, "partly": 10, "occupied": 10, "objects": 10, "numerous": 10, "note": 10, "native": 10, "minority": 10, "middle": 10, "massachusetts": 10, "marked": 10, "lines": 10, "light": 10, "liberties": 10, "laid": 10, "king": 10, "issued": 10, "insists": 10, "indians": 10, "inclined": 10, "implication": 10, "hoped": 10, "holds": 10, "hearts": 10, "happiness": 10, "governs": 10, "govern": 10, "fundamental": 10, "foundation": 10, "forbade": 10, "floor": 10, "finds": 10, "expediency": 10, "exclusively": 10, "exclusive": 10, "establishment": 10, "establishing": 10, "equivalent": 10, "entered": 10, "effective": 10, "education": 10, "earliest": 10, "duties": 10, "divine": 10, "dispute": 10, "discord": 10, "declares": 10, "declarations": 10, "debate": 10, "cotton": 10, "correct": 10, "conventions": 10, "contain": 10, "consummation": 10, "conclusive": 10, "circuit": 10, "children": 10, "causes": 10, "careful": 10, "campbell": 10, "book": 10, "blockade": 10, "benefit": 10, "author": 10, "arrest": 10, "arisen": 10, "appointed": 10, "amended": 10, "adopting": 10, "addition": 10, "absurd": 10, "abroad": 10, "1860": 10, "1819": 10, "wisdom": 9, "uttered": 9, "tremendous": 9, "testimony": 9, "teeth": 9, "task": 9, "surrendered": 9, "surrender": 9, "suppressed": 9, "suggestion": 9, "submission": 9, "strongly": 9, "step": 9, "standing": 9, "seventy": 9, "sent": 9, "sell": 9, "security": 9, "satisfied": 9, "room": 9, "resolution": 9, "repealed": 9, "remind": 9, "rejected": 9, "regulate": 9, "regions": 9, "refused": 9, "referred": 9, "recently": 9, "realize": 9, "range": 9, "quarrel": 9, "proves": 9, "prosperous": 9, "prospective": 9, "prominent": 9, "probable": 9, "pressed": 9, "press": 9, "postal": 9, "post": 9, "politicians": 9, "politically": 9, "originated": 9, "opposite": 9, "omit": 9, "occur": 9, "naked": 9, "motion": 9, "minds": 9, "manner": 9, "manifest": 9, "maintaining": 9, "logical": 9, "liberal": 9, "liar": 9, "lest": 9, "legislative": 9, "lead": 9, "largely": 9, "jurisdiction": 9, "investigate": 9, "internal": 9, "indorse": 9, "inaugural": 9, "impression": 9, "hostile": 9, "happy": 9, "hang": 9, "fugitives": 9, "frequently": 9, "foreigners": 9, "father": 9, "faithfully": 9, "eye": 9, "extensive": 9, "exceedingly": 9, "events": 9, "estimated": 9, "era": 9, "equals": 9, "entertain": 9, "enforced": 9, "enforce": 9, "endorsement": 9, "empire": 9, "driven": 9, "drawn": 9, "doubtless": 9, "dollars": 9, "documents": 9, "disunion": 9, "distinction": 9, "dissolution": 9, "disposition": 9, "direction": 9, "desirable": 9, "demands": 9, "continues": 9, "consequence": 9, "confederacy": 9, "conducted": 9, "concern": 9, "conceive": 9, "compensated": 9, "comments": 9, "chiefly": 9, "chase's": 9, "charter": 9, "chance": 9, "census": 9, "cash": 9, "benevolent": 9, "based": 9, "average": 9, "attended": 9, "appears": 9, "angry": 9, "addressed": 9, "1864": 9, "1856": 9, "york": 8, "working": 8, "witness": 8, "wild": 8, "wide": 8, "warning": 8, "wages": 8, "virtue": 8, "utterly": 8, "utter": 8, "urged": 8, "unfortunate": 8, "unconditional": 8, "turned": 8, "tribes": 8, "trial": 8, "thunders": 8, "throw": 8, "threatened": 8, "thomas": 8, "thinking": 8, "telegraph": 8, "teaches": 8, "sworn": 8, "sufficiently": 8, "succeeding": 8, "substantial": 8, "strictly": 8, "stop": 8, "steady": 8, "statute": 8, "spite": 8, "slightest": 8, "simple": 8, "showed": 8, "settle": 8, "secession": 8, "seat": 8, "saved": 8, "safe": 8, "rightfully": 8, "resisted": 8, "resistance": 8, "reports": 8, "remained": 8, "recent": 8, "ratified": 8, "railroad": 8, "pursuit": 8, "providing": 8, "providence": 8, "propriety": 8, "promised": 8, "profound": 8, "profess": 8, "producing": 8, "proclaim": 8, "proceeding": 8, "proceed": 8, "privileges": 8, "pleases": 8, "platforms": 8, "physical": 8, "perpetuity": 8, "perpetuation": 8, "permanently": 8, "pensions": 8, "payment": 8, "obliged": 8, "nullification": 8, "non": 8, "manifested": 8, "lord": 8, "lincoln's": 8, "lights": 8, "legislating": 8, "labors": 8, "invite": 8, "intend": 8, "instantly": 8, "indorsement": 8, "include": 8, "inclination": 8, "improvement": 8, "improved": 8, "hung": 8, "horses": 8, "homes": 8, "higher": 8, "herewith": 8, "guilty": 8, "gross": 8, "genuine": 8, "galesburgh": 8, "furnish": 8, "french": 8, "freely": 8, "frankly": 8, "fortunate": 8, "forbids": 8, "fix": 8, "filled": 8, "felt": 8, "fast": 8, "excuse": 8, "enlarging": 8, "employed": 8, "elements": 8, "door": 8, "document": 8, "died": 8, "detail": 8, "desired": 8, "deprived": 8, "depends": 8, "denounce": 8, "degree": 8, "defense": 8, "deep": 8, "deem": 8, "covert": 8, "correctly": 8, "convince": 8, "continent": 8, "constituted": 8, "consistent": 8, "conduct": 8, "concerns": 8, "conceived": 8, "competent": 8, "comparison": 8, "civilized": 8, "city": 8, "champion": 8, "catch": 8, "buchanan": 8, "broken": 8, "breaking": 8, "branches": 8, "bigler": 8, "bible": 8, "belong": 8, "bank": 8, "bad": 8, "atlantic": 8, "assumption": 8, "asking": 8, "ascertain": 8, "armies": 8, "armed": 8, "arising": 8, "apt": 8, "approbation": 8, "appeared": 8, "anybody": 8, "allude": 8, "affirmative": 8, "advocates": 8, "adjust": 8, "adapted": 8, "acres": 8, "acquisitions": 8, "acquire": 8, "accordingly": 8, "abstract": 8, "abolitionist": 8, "abolished": 8, "1849": 8, "1848": 8, "1812": 8, "zeal": 7, "worthy": 7, "wished": 7, "wisconsin": 7, "wife": 7, "webster": 7, "waters": 7, "violated": 7, "vessel": 7, "ventured": 7, "unwilling": 7, "unjust": 7, "undertake": 7, "unconstitutional": 7, "unable": 7, "turning": 7, "trusting": 7, "today": 7, "thrust": 7, "threw": 7, "thirteen": 7, "that's": 7, "thanks": 7, "taunt": 7, "sympathies": 7, "sustained": 7, "suitable": 7, "succession": 7, "struggled": 7, "stating": 7, "sole": 7, "societies": 7, "signs": 7, "shore": 7, "ship": 7, "settling": 7, "served": 7, "servants": 7, "seeks": 7, "sat": 7, "sales": 7, "rules": 7, "rise": 7, "rich": 7, "restored": 7, "resigned": 7, "repress": 7, "remarkable": 7, "relieve": 7, "regulations": 7, "regret": 7, "recollection": 7, "reasoning": 7, "rate": 7, "quoted": 7, "quarters": 7, "quality": 7, "previously": 7, "preservation": 7, "presents": 7, "plot": 7, "plant": 7, "personally": 7, "perform": 7, "pending": 7, "oregon": 7, "opposing": 7, "openly": 7, "omitted": 7, "omission": 7, "october": 7, "ocean": 7, "obligatory": 7, "notes": 7, "nominated": 7, "moved": 7, "merit": 7, "meetings": 7, "master": 7, "mass": 7, "maritime": 7, "maintenance": 7, "lower": 7, "lost": 7, "looked": 7, "liberia": 7, "liberate": 7, "letting": 7, "laying": 7, "latest": 7, "justly": 7, "job": 7, "island": 7, "insurrectionary": 7, "induced": 7, "indefinite": 7, "inconsistency": 7, "inauguration": 7, "hushed": 7, "hurrah": 7, "humanity": 7, "horse": 7, "honest": 7, "hit": 7, "hired": 7, "hard": 7, "gratifying": 7, "georgia": 7, "geographical": 7, "furnishes": 7, "friendly": 7, "freemen": 7, "framing": 7, "fought": 7, "forced": 7, "florida": 7, "firmly": 7, "finger": 7, "financial": 7, "fear": 7, "extracts": 7, "extending": 7, "expenses": 7, "expectation": 7, "exception": 7, "evident": 7, "eulogy": 7, "emigration": 7, "eloquent": 7, "elect": 7, "effected": 7, "easier": 7, "earnest": 7, "dwell": 7, "duly": 7, "drive": 7, "doubtful": 7, "disturbing": 7, "dissatisfied": 7, "disloyal": 7, "discharged": 7, "differences": 7, "despise": 7, "desires": 7, "design": 7, "deprive": 7, "denying": 7, "delaware": 7, "defend": 7, "debts": 7, "dear": 7, "currency": 7, "count": 7, "correspondence": 7, "convenient": 7, "content": 7, "contemplation": 7, "consultation": 7, "constitutes": 7, "conditions": 7, "concluded": 7, "completed": 7, "communication": 7, "commanded": 7, "combination": 7, "color": 7, "collision": 7, "claiming": 7, "charles": 7, "chances": 7, "cattle": 7, "carrying": 7, "briefly": 7, "breath": 7, "bondage": 7, "begin": 7, "becomes": 7, "beat": 7, "bargain": 7, "banks": 7, "ballot": 7, "attitude": 7, "assure": 7, "assurance": 7, "assuming": 7, "asserting": 7, "assail": 7, "aspect": 7, "approach": 7, "apprehension": 7, "ancient": 7, "alteration": 7, "alluded": 7, "alleges": 7, "alive": 7, "alarmed": 7, "alabama": 7, "agreement": 7, "advancement": 7, "admits": 7, "adding": 7, "acknowledged": 7, "accordance": 7, "abundant": 7, "ability": 7, "4th": 7, "30th": 7, "1852": 7, "1808": 7, "17th": 7, "1787": 7, "yesterday": 6, "wrote": 6, "world's": 6, "withdraw": 6, "william": 6, "weight": 6, "waste": 6, "wash": 6, "vociferous": 6, "violate": 6, "using": 6, "useful": 6, "untrue": 6, "undertakes": 6, "unanimous": 6, "unanimity": 6, "treating": 6, "treated": 6, "tongue": 6, "thirds": 6, "tear": 6, "taxation": 6, "sustaining": 6, "substitute": 6, "strike": 6, "stephen": 6, "stay": 6, "square": 6, "springing": 6, "spring": 6, "species": 6, "soul": 6, "somebody": 6, "sincerity": 6, "sincerely": 6, "silence": 6, "signed": 6, "signal": 6, "serve": 6, "seriously": 6, "series": 6, "sentence": 6, "seized": 6, "seeking": 6, "seek": 6, "sections": 6, "seceded": 6, "secede": 6, "scope": 6, "sanction": 6, "san": 6, "sake": 6, "russia": 6, "rose": 6, "roger": 6, "road": 6, "review": 6, "reverse": 6, "returns": 6, "returning": 6, "retraction": 6, "retain": 6, "restoring": 6, "respects": 6, "respected": 6, "respectable": 6, "requisite": 6, "representation": 6, "reporters": 6, "repealing": 6, "renew": 6, "rendered": 6, "render": 6, "removed": 6, "reject": 6, "reflection": 6, "reduce": 6, "red": 6, "recorded": 6, "recommended": 6, "recollect": 6, "reads": 6, "ratio": 6, "raised": 6, "raise": 6, "quote": 6, "quotations": 6, "questioned": 6, "pushed": 6, "push": 6, "promptly": 6, "project": 6, "production": 6, "process": 6, "proceeds": 6, "private": 6, "prevented": 6, "prevailed": 6, "preparing": 6, "practicable": 6, "powerful": 6, "postmaster": 6, "positions": 6, "politics": 6, "police": 6, "pointing": 6, "plausible": 6, "peru": 6, "perish": 6, "pensioners": 6, "partially": 6, "orleans": 6, "operating": 6, "opened": 6, "offices": 6, "offer": 6, "offensive": 6, "niche": 6, "news": 6, "nevada": 6, "neighbor": 6, "negative": 6, "necessities": 6, "naturally": 6, "nationalize": 6, "mountains": 6, "molony": 6, "modifications": 6, "model": 6, "mistaken": 6, "mile": 6, "michigan": 6, "merits": 6, "lovejoy": 6, "louis": 6, "looking": 6, "legislate": 6, "lee's": 6, "leader": 6, "justified": 6, "joyous": 6, "jacksonville": 6, "involve": 6, "invited": 6, "invention": 6, "invasion": 6, "introducing": 6, "intent": 6, "injustice": 6, "infer": 6, "induce": 6, "immense": 6, "humble": 6, "hostility": 6, "highly": 6, "helping": 6, "happens": 6, "gratitude": 6, "gradually": 6, "governed": 6, "glory": 6, "glorious": 6, "glance": 6, "garrison": 6, "freed": 6, "founded": 6, "forts": 6, "forms": 6, "forgotten": 6, "forged": 6, "forborne": 6, "fine": 6, "finding": 6, "ficklin": 6, "fearful": 6, "family": 6, "explicit": 6, "exciting": 6, "examination": 6, "evils": 6, "evidences": 6, "eternal": 6, "erroneous": 6, "ere": 6, "entertained": 6, "enacted": 6, "enable": 6, "effectually": 6, "dropped": 6, "drop": 6, "draw": 6, "doctrines": 6, "divert": 6, "discussed": 6, "directed": 6, "differed": 6, "designated": 6, "dependent": 6, "denounced": 6, "denial": 6, "definitely": 6, "definite": 6, "current": 6, "cruel": 6, "counties": 6, "council": 6, "corrupt": 6, "cordially": 6, "contract": 6, "contention": 6, "contended": 6, "contend": 6, "containing": 6, "constitute": 6, "constant": 6, "consequences": 6, "conscientious": 6, "confederation": 6, "compared": 6, "committed": 6, "commissioner": 6, "commission": 6, "comment": 6, "commanders": 6, "collateral": 6, "closed": 6, "china": 6, "cheerfully": 6, "charging": 6, "changing": 6, "carlin": 6, "cannon": 6, "cabinet": 6, "build": 6, "brown": 6, "border": 6, "bone": 6, "bond": 6, "boldly": 6, "bloody": 6, "benefits": 6, "belligerent": 6, "beg": 6, "becoming": 6, "avert": 6, "attempts": 6, "attack": 6, "astonishment": 6, "assume": 6, "assistance": 6, "assailed": 6, "arrayed": 6, "arose": 6, "appealed": 6, "annexation": 6, "anew": 6, "ancestors": 6, "amiable": 6, "alterations": 6, "allusion": 6, "allegiance": 6, "alas": 6, "agriculture": 6, "advocate": 6, "adherence": 6, "acting": 6, "acceptance": 6, "accept": 6, "abraham": 6, "9th": 6, "1862": 6, "1853": 6, "young": 5, "writing": 5, "won't": 5, "wisest": 5, "winter": 5, "westward": 5, "voter": 5, "voluntarily": 5, "void": 5, "violation": 5, "vigor": 5, "victory": 5, "vexed": 5, "unusual": 5, "unto": 5, "ultimately": 5, "triumph": 5, "transmitted": 5, "tract": 5, "traced": 5, "trace": 5, "touching": 5, "touch": 5, "tone": 5, "thirdly": 5, "thank": 5, "tendencies": 5, "tend": 5, "telling": 5, "taxed": 5, "taste": 5, "swear": 5, "suspended": 5, "supposes": 5, "supported": 5, "supply": 5, "summary": 5, "suggested": 5, "sugar": 5, "successive": 5, "successfully": 5, "subsequently": 5, "stuck": 5, "struggles": 5, "strikes": 5, "strict": 5, "sticks": 5, "standard": 5, "squatter": 5, "springs": 5, "speedy": 5, "specifically": 5, "situation": 5, "simultaneously": 5, "settlers": 5, "separated": 5, "sacrifices": 5, "sacrifice": 5, "route": 5, "rolls": 5, "revolt": 5, "revived": 5, "revival": 5, "returned": 5, "retained": 5, "restricting": 5, "restraint": 5, "request": 5, "republicanism": 5, "removal": 5, "reminds": 5, "remembered": 5, "remedy": 5, "religion": 5, "reliable": 5, "rejection": 5, "regular": 5, "regiments": 5, "records": 5, "reclaim": 5, "receiving": 5, "rebel": 5, "reasonably": 5, "ratification": 5, "quibbling": 5, "puts": 5, "purposely": 5, "publication": 5, "proud": 5, "prospect": 5, "proposing": 5, "promises": 5, "profitable": 5, "produces": 5, "primary": 5, "price": 5, "pretty": 5, "pretending": 5, "presidency": 5, "premises": 5, "precedent": 5, "possibly": 5, "possibility": 5, "possessed": 5, "port": 5, "planted": 5, "philosophy": 5, "perils": 5, "peril": 5, "pension": 5, "penned": 5, "pecuniary": 5, "payments": 5, "passion": 5, "pardon": 5, "page": 5, "overlooked": 5, "organizing": 5, "organic": 5, "oppressed": 5, "officer": 5, "occurred": 5, "obvious": 5, "objections": 5, "oaths": 5, "noblest": 5, "newspapers": 5, "nation's": 5, "narrow": 5, "napoleon": 5, "motive": 5, "monstrous": 5, "modification": 5, "mines": 5, "mineral": 5, "million": 5, "menace": 5, "maxim": 5, "map": 5, "mankind": 5, "mainly": 5, "machinery": 5, "loved": 5, "lose": 5, "localities": 5, "learn": 5, "latitude": 5, "lastly": 5, "ladies": 5, "lack": 5, "kings": 5, "joined": 5, "jersey": 5, "japan": 5, "issues": 5, "involving": 5, "invoked": 5, "invalid": 5, "intimation": 5, "intimated": 5, "intervention": 5, "international": 5, "intelligence": 5, "instances": 5, "inserted": 5, "innocent": 5, "injurious": 5, "inhabitants": 5, "informed": 5, "indorsed": 5, "indifference": 5, "incident": 5, "improve": 5, "immunities": 5, "humane": 5, "hostilities": 5, "hopes": 5, "honorable": 5, "homestead": 5, "hog": 5, "highest": 5, "health": 5, "hastily": 5, "haste": 5, "guaranties": 5, "growing": 5, "grow": 5, "greeted": 5, "granted": 5, "grant": 5, "grand": 5, "gin": 5, "generous": 5, "generations": 5, "gen": 5, "gained": 5, "furnished": 5, "fraternal": 5, "franklin": 5, "foundations": 5, "formation": 5, "foregoing": 5, "foes": 5, "fitting": 5, "ferry": 5, "fell": 5, "feeble": 5, "february": 5, "features": 5, "favorably": 5, "extremely": 5, "extinct": 5, "expression": 5, "expedient": 5, "exhibited": 5, "exclusion": 5, "excludes": 5, "excluded": 5, "excepted": 5, "establishes": 5, "enterprise": 5, "entering": 5, "enormity": 5, "enjoy": 5, "england": 5, "energies": 5, "enabling": 5, "enables": 5, "embracing": 5, "eighty": 5, "economical": 5, "eat": 5, "durable": 5, "dreaded": 5, "doubted": 5, "dissolved": 5, "discussions": 5, "discuss": 5, "discretion": 5, "discovery": 5, "discarding": 5, "discarded": 5, "detain": 5, "despotism": 5, "designed": 5, "dependence": 5, "delegates": 5, "degrees": 5, "deficiency": 5, "defeat": 5, "deeper": 5, "declined": 5, "decidedly": 5, "dealt": 5, "dark": 5, "crime": 5, "credit": 5, "creation": 5, "cranberry": 5, "councils": 5, "corresponding": 5, "controlling": 5, "contending": 5, "contemplated": 5, "consuls": 5, "constitutionally": 5, "consistency": 5, "consist": 5, "considering": 5, "confided": 5, "concur": 5, "concession": 5, "concerning": 5, "completion": 5, "complaint": 5, "complains": 5, "complain": 5, "communities": 5, "commit": 5, "commencement": 5, "commander": 5, "command": 5, "coin": 5, "closing": 5, "civilization": 5, "circulation": 5, "circle": 5, "chile": 5, "child": 5, "chestnut": 5, "cession": 5, "caused": 5, "candor": 5, "calls": 5, "bread": 5, "box": 5, "bore": 5, "borders": 5, "belonging": 5, "belonged": 5, "authorities": 5, "authoritative": 5, "augmented": 5, "association": 5, "assigned": 5, "assault": 5, "arrival": 5, "arise": 5, "approved": 5, "approaching": 5, "appreciate": 5, "applies": 5, "announce": 5, "anarchy": 5, "ample": 5, "amounts": 5, "amounted": 5, "almighty": 5, "aliens": 5, "agricultural": 5, "affirmation": 5, "adversaries": 5, "administered": 5, "administer": 5, "adheres": 5, "adaptation": 5, "abolitionism": 5, "abolishing": 5, "5th": 5, "22d": 5, "18th": 5, "1858": 5, "yielding": 4, "yard": 4, "yankees": 4, "workings": 4, "worked": 4, "woman": 4, "witnesses": 4, "withholding": 4, "widely": 4, "warrants": 4, "voluntary": 4, "vivid": 4, "vitally": 4, "virtually": 4, "virtual": 4, "violating": 4, "variance": 4, "vain": 4, "vacancy": 4, "unwise": 4, "unorganized": 4, "unlike": 4, "uninhabited": 4, "undoubtedly": 4, "understands": 4, "uncertain": 4, "unanimously": 4, "tyrants": 4, "trusted": 4, "troubles": 4, "trembled": 4, "totally": 4, "topics": 4, "tolerate": 4, "tobacco": 4, "timbers": 4, "throwing": 4, "throes": 4, "threaten": 4, "thraldom": 4, "thenceforward": 4, "tended": 4, "temper": 4, "tears": 4, "tax": 4, "taught": 4, "taney": 4, "talking": 4, "swelling": 4, "sweep": 4, "suspension": 4, "suspect": 4, "surveyed": 4, "supporters": 4, "supplies": 4, "sums": 4, "suit": 4, "suggestions": 4, "suffered": 4, "sudden": 4, "successes": 4, "strongest": 4, "stronger": 4, "strive": 4, "strife": 4, "stipulations": 4, "steps": 4, "statutes": 4, "station": 4, "statements": 4, "stake": 4, "spreading": 4, "spell": 4, "speedily": 4, "speaks": 4, "spake": 4, "southward": 4, "sold": 4, "sober": 4, "sketch": 4, "size": 4, "situated": 4, "significance": 4, "shadow": 4, "sensible": 4, "seeing": 4, "securities": 4, "secret": 4, "search": 4, "seamen": 4, "schedule": 4, "saturday": 4, "satisfy": 4, "satisfaction": 4, "sanguine": 4, "sale": 4, "ruthless": 4, "rush": 4, "rocky": 4, "rivers": 4, "rica": 4, "revolutionized": 4, "reviving": 4, "reversing": 4, "reversed": 4, "retracted": 4, "retired": 4, "residing": 4, "residence": 4, "reserved": 4, "requiring": 4, "reproach": 4, "repair": 4, "relinquish": 4, "relative": 4, "regulation": 4, "refusing": 4, "refuses": 4, "refuse": 4, "recurs": 4, "recommendations": 4, "recognition": 4, "reclaiming": 4, "reclaimed": 4, "rebuked": 4, "reaching": 4, "quotes": 4, "quantity": 4, "qualified": 4, "pursued": 4, "pure": 4, "publicly": 4, "proving": 4, "proportions": 4, "promote": 4, "profit": 4, "proceeded": 4, "pride": 4, "preventing": 4, "prevail": 4, "pretended": 4, "pretend": 4, "possess": 4, "positive": 4, "polk": 4, "politician": 4, "politic": 4, "pledge": 4, "playing": 4, "play": 4, "placing": 4, "piracy": 4, "pickens": 4, "pertaining": 4, "permitting": 4, "permits": 4, "permission": 4, "periods": 4, "percentage": 4, "people's": 4, "penetrate": 4, "paying": 4, "passage": 4, "partizan": 4, "parcel": 4, "pain": 4, "oyster": 4, "ownership": 4, "overthrown": 4, "overrun": 4, "odd": 4, "obtained": 4, "objected": 4, "obedience": 4, "noticing": 4, "northwestern": 4, "norfolk": 4, "night": 4, "newly": 4, "neutrality": 4, "negotiation": 4, "nearest": 4, "nearer": 4, "narrowed": 4, "named": 4, "muzzle": 4, "movements": 4, "mortals": 4, "moment's": 4, "modifying": 4, "modified": 4, "misunderstood": 4, "misunderstanding": 4, "misunderstand": 4, "misrepresent": 4, "misplaced": 4, "miserable": 4, "miscellaneous": 4, "minor": 4, "migration": 4, "midst": 4, "mclean": 4, "matheny": 4, "manifestly": 4, "management": 4, "manage": 4, "majesty's": 4, "maintainance": 4, "magnitude": 4, "lying": 4, "lullaby": 4, "lovers": 4, "lived": 4, "literally": 4, "literal": 4, "lion": 4, "limit": 4, "lieu": 4, "legally": 4, "learned": 4, "leads": 4, "lament": 4, "lacking": 4, "laborer": 4, "labored": 4, "knell": 4, "kindly": 4, "key": 4, "keeping": 4, "kane": 4, "justify": 4, "judicious": 4, "jonesboro": 4, "ipse": 4, "invented": 4, "introduces": 4, "intrinsically": 4, "interesting": 4, "intelligible": 4, "integrity": 4, "insurrections": 4, "insurrectionists": 4, "instructed": 4, "instituted": 4, "inquiry": 4, "inquire": 4, "injury": 4, "inferred": 4, "inference": 4, "infant": 4, "inevitably": 4, "indirectly": 4, "indicated": 4, "indentured": 4, "incoming": 4, "incidents": 4, "inch": 4, "inalienable": 4, "impulse": 4, "improving": 4, "impossibility": 4, "importation": 4, "implicated": 4, "imperfectly": 4, "impassable": 4, "illustration": 4, "illustrate": 4, "husband": 4, "humoredly": 4, "hours": 4, "hot": 4, "honors": 4, "honestly": 4, "hon": 4, "holders": 4, "hesitate": 4, "helped": 4, "heavy": 4, "heaven": 4, "hearing": 4, "hayti": 4, "harvests": 4, "harper's": 4, "hardly": 4, "harbor": 4, "guaranty": 4, "groundless": 4, "grief": 4, "grew": 4, "grateful": 4, "governor": 4, "governmental": 4, "gold": 4, "gladly": 4, "gifted": 4, "galesburg": 4, "gain": 4, "fruits": 4, "fruit": 4, "friendship": 4, "fresh": 4, "forgot": 4, "forfeited": 4, "footing": 4, "foot": 4, "fools": 4, "folly": 4, "fly": 4, "flag": 4, "fitted": 4, "finality": 4, "fiercely": 4, "fields": 4, "fidelity": 4, "feet": 4, "feature": 4, "favored": 4, "fatally": 4, "faster": 4, "famed": 4, "fame": 4, "fallen": 4, "faithful": 4, "extremes": 4, "expressive": 4, "expenditure": 4, "expended": 4, "expelled": 4, "expedition": 4, "exhibits": 4, "exhaustion": 4, "exhausted": 4, "executed": 4, "excuses": 4, "excited": 4, "excellent": 4, "examined": 4, "european": 4, "essential": 4, "escape": 4, "eradicate": 4, "equivalents": 4, "entrust": 4, "entertaining": 4, "enslaving": 4, "enslave": 4, "english": 4, "enforcement": 4, "energy": 4, "enduring": 4, "endeavoring": 4, "endangers": 4, "encouragement": 4, "enactment": 4, "emigrants": 4, "embraced": 4, "electors": 4, "effects": 4, "editorially": 4, "eastern": 4, "ears": 4, "earnestly": 4, "droves": 4, "dread": 4, "dr": 4, "dixit": 4, "disturbance": 4, "distinctions": 4, "discover": 4, "discourse": 4, "disadvantage": 4, "diminished": 4, "differs": 4, "differ": 4, "die": 4, "development": 4, "destroying": 4, "desperate": 4, "deserve": 4, "descendants": 4, "depend": 4, "departure": 4, "departments": 4, "departed": 4, "demur": 4, "deliberate": 4, "degrading": 4, "definition": 4, "deference": 4, "defending": 4, "deems": 4, "deeds": 4, "dedicated": 4, "deciding": 4, "dec": 4, "dealer": 4, "dares": 4, "crowd": 4, "criminal": 4, "create": 4, "covered": 4, "countenance": 4, "counted": 4, "costa": 4, "corn": 4, "cordial": 4, "contrast": 4, "contemplate": 4, "consular": 4, "consists": 4, "consequent": 4, "consciousness": 4, "connecticut": 4, "confusion": 4, "concurring": 4, "concurred": 4, "conclude": 4, "concert": 4, "composed": 4, "complimented": 4, "complaints": 4, "complained": 4, "comparatively": 4, "company": 4, "combating": 4, "colors": 4, "colorado": 4, "colonizing": 4, "col": 4, "cling": 4, "clamored": 4, "chopping": 4, "chooses": 4, "cherish": 4, "cheap": 4, "characterized": 4, "champions": 4, "central": 4, "ceded": 4, "cares": 4, "cared": 4, "captured": 4, "captive": 4, "capitol": 4, "capable": 4, "candid": 4, "canada": 4, "can't": 4, "burdens": 4, "burden": 4, "brother": 4, "breach": 4, "brave": 4, "bold": 4, "blot": 4, "blessed": 4, "birth": 4, "binding": 4, "bent": 4, "bench": 4, "believing": 4, "bearing": 4, "base": 4, "banking": 4, "aver": 4, "authors": 4, "attempting": 4, "asserted": 4, "assemblage": 4, "arrive": 4, "arming": 4, "arm": 4, "arguing": 4, "ardent": 4, "april": 4, "anxious": 4, "anticipated": 4, "answering": 4, "andre": 4, "amicable": 4, "alton": 4, "alter": 4, "alluding": 4, "allen": 4, "akin": 4, "agreeing": 4, "agitated": 4, "aggravated": 4, "afraid": 4, "aforesaid": 4, "afforded": 4, "afford": 4, "affirms": 4, "affirm": 4, "affair": 4, "advocating": 4, "advisable": 4, "advice": 4, "adjourned": 4, "adjoins": 4, "actions": 4, "acquiescence": 4, "acquiesced": 4, "achieved": 4, "accompanying": 4, "abuse": 4, "abundance": 4, "6th": 4, "3d": 4, "21st": 4, "1851": 4, "1845": 4, "1824": 4, "1807": 4, "1798": 4, "yeas": 3, "yates": 3, "writ": 3, "worse": 3, "works": 3, "wonderful": 3, "woe": 3, "wives": 3, "withal": 3, "wiser": 3, "wisely": 3, "winds": 3, "whereof": 3, "wen": 3, "weighed": 3, "week": 3, "wealth": 3, "weak": 3, "wars": 3, "wall": 3, "vulgarity": 3, "vocal": 3, "vicinity": 3, "vice": 3, "vermont": 3, "vastly": 3, "vacancies": 3, "urge": 3, "uproarious": 3, "unquestionably": 3, "unprecedented": 3, "unobjectionable": 3, "unmistakable": 3, "unlikely": 3, "uniformity": 3, "unholy": 3, "unhappy": 3, "unexplained": 3, "underlings": 3, "turkey": 3, "truthful": 3, "troublesome": 3, "triple": 3, "tries": 3, "transit": 3, "traffic": 3, "trader": 3, "touches": 3, "torn": 3, "topic": 3, "tongues": 3, "toil": 3, "tip": 3, "thousands": 3, "thoughts": 3, "thorough": 3, "thin": 3, "thereto": 3, "therefrom": 3, "tends": 3, "tempted": 3, "temporarily": 3, "tazewell": 3, "taylor": 3, "taxes": 3, "tardiness": 3, "tanto": 3, "swell": 3, "susceptible": 3, "surveyor": 3, "surprise": 3, "surface": 3, "supremacy": 3, "supporting": 3, "supervision": 3, "suggesting": 3, "successors": 3, "succeeded": 3, "substituting": 3, "structure": 3, "striving": 3, "stringency": 3, "strangely": 3, "straight": 3, "storm": 3, "stipulation": 3, "stilled": 3, "stead": 3, "statesmen": 3, "statesmanship": 3, "starved": 3, "started": 3, "start": 3, "stable": 3, "stability": 3, "sprung": 3, "spot": 3, "splitting": 3, "specially": 3, "space": 3, "sorry": 3, "sophistry": 3, "sophism": 3, "soldier": 3, "socially": 3, "sneaking": 3, "smallest": 3, "slaveholding": 3, "slaveholder": 3, "slanders": 3, "slander": 3, "sink": 3, "shoulders": 3, "ships": 3, "shipping": 3, "shifts": 3, "sherman": 3, "sever": 3, "seventh": 3, "setting": 3, "separately": 3, "sentences": 3, "sending": 3, "selling": 3, "selfish": 3, "selection": 3, "selected": 3, "seldom": 3, "seizure": 3, "seizes": 3, "securing": 3, "secures": 3, "sectionalism": 3, "secretaries": 3, "secondly": 3, "seceding": 3, "seceders": 3, "seated": 3, "seas": 3, "seacoast": 3, "seaboard": 3, "score": 3, "school": 3, "scheme": 3, "scattered": 3, "scaffolding": 3, "saith": 3, "sailor": 3, "sage": 3, "runaway": 3, "ruler": 3, "ruin": 3, "rising": 3, "richard": 3, "rhode": 3, "revenues": 3, "retract": 3, "retirement": 3, "resulting": 3, "restrict": 3, "restrain": 3, "rested": 3, "respecting": 3, "resort": 3, "resolve": 3, "resisting": 3, "republics": 3, "representative": 3, "repeatedly": 3, "reopened": 3, "removing": 3, "reminded": 3, "remembering": 3, "remainder": 3, "relieves": 3, "relief": 3, "relied": 3, "reliance": 3, "regulating": 3, "regularly": 3, "referring": 3, "redemption": 3, "reconstruction": 3, "recommends": 3, "recognizes": 3, "reckless": 3, "reader": 3, "ratify": 3, "rates": 3, "ran": 3, "railroads": 3, "rage": 3, "quasi": 3, "quarrels": 3, "qualities": 3, "purely": 3, "punishment": 3, "prudent": 3, "prudence": 3, "provoke": 3, "provides": 3, "proven": 3, "protest": 3, "protected": 3, "pronounced": 3, "pronounce": 3, "prolonged": 3, "prohibits": 3, "professional": 3, "proclaimed": 3, "pro": 3, "prize": 3, "printer": 3, "principally": 3, "pretext": 3, "pretenses": 3, "pressing": 3, "preserving": 3, "preserved": 3, "prescribed": 3, "prejudice": 3, "precious": 3, "preach": 3, "practiced": 3, "pour": 3, "populous": 3, "pledges": 3, "pleasure": 3, "plaza": 3, "playful": 3, "played": 3, "plausibility": 3, "pirates": 3, "pine": 3, "pinckney": 3, "phrase": 3, "phillips": 3, "pettit": 3, "penalty": 3, "peaceably": 3, "patriot": 3, "participated": 3, "parental": 3, "parallel": 3, "papers": 3, "owing": 3, "owed": 3, "owe": 3, "overthrow": 3, "outsiders": 3, "outside": 3, "outrage": 3, "outlet": 3, "outgoing": 3, "organ": 3, "operates": 3, "operate": 3, "offers": 3, "offenses": 3, "offense": 3, "odious": 3, "occupies": 3, "occupation": 3, "occasionally": 3, "obtaining": 3, "obscure": 3, "objectionable": 3, "obeyed": 3, "null": 3, "notify": 3, "noticed": 3, "norton": 3, "nominal": 3, "nobly": 3, "nicaragua": 3, "neutral": 3, "negotiated": 3, "negotiate": 3, "nays": 3, "navigation": 3, "naturalized": 3, "nationalization": 3, "movement": 3, "moon": 3, "mongrels": 3, "moneys": 3, "momentous": 3, "mixed": 3, "mitigate": 3, "mischief": 3, "misapply": 3, "minorities": 3, "minister": 3, "mingled": 3, "mingle": 3, "militia": 3, "menaced": 3, "memories": 3, "meets": 3, "matured": 3, "masses": 3, "martin": 3, "market": 3, "mark": 3, "manufactures": 3, "malice": 3, "major": 3, "mails": 3, "mail": 3, "magistrate": 3, "lump": 3, "lowest": 3, "lot": 3, "loss": 3, "loose": 3, "license": 3, "liberated": 3, "liberally": 3, "lexington": 3, "lesson": 3, "lesser": 3, "leaders": 3, "lawyers": 3, "lasted": 3, "largest": 3, "landed": 3, "lakes": 3, "la": 3, "knowingly": 3, "knees": 3, "keeps": 3, "keeper": 3, "judgments": 3, "journals": 3, "joliet": 3, "jealous": 3, "it's": 3, "irritation": 3, "iron": 3, "involved": 3, "invokes": 3, "invasions": 3, "intrusted": 3, "intimately": 3, "intervals": 3, "interrupted": 3, "interposition": 3, "interference": 3, "intellectual": 3, "insure": 3, "instruments": 3, "insignificant": 3, "injuries": 3, "initiated": 3, "inhuman": 3, "inhabiting": 3, "inhabit": 3, "ingenious": 3, "inform": 3, "influential": 3, "influences": 3, "inexcusable": 3, "industrial": 3, "inducing": 3, "indefinitely": 3, "increases": 3, "incorporated": 3, "includes": 3, "improperly": 3, "impracticable": 3, "imposts": 3, "imposed": 3, "implicitly": 3, "impatient": 3, "immortal": 3, "immigration": 3, "immigrants": 3, "illustrious": 3, "ill": 3, "ignorance": 3, "i'll": 3, "hypocrites": 3, "hurts": 3, "hurt": 3, "hurry": 3, "houses": 3, "horror": 3, "holder": 3, "hitherto": 3, "hire": 3, "hinder": 3, "hesitated": 3, "helps": 3, "heaviest": 3, "heads": 3, "headed": 3, "harsh": 3, "harmonize": 3, "harm": 3, "happen": 3, "hanover": 3, "hall": 3, "habit": 3, "guns": 3, "guilt": 3, "guarded": 3, "guard": 3, "guarantied": 3, "guarantee": 3, "grudgingly": 3, "grounds": 3, "grossly": 3, "gotten": 3, "glorification": 3, "gettysburg": 3, "gets": 3, "george": 3, "gallant": 3, "fuss": 3, "furnishing": 3, "functions": 3, "frank": 3, "fragments": 3, "fourths": 3, "fostering": 3, "forges": 3, "ford's": 3, "flour": 3, "firmness": 3, "finances": 3, "figures": 3, "favors": 3, "favoring": 3, "fate": 3, "farewell": 3, "familiar": 3, "facilitate": 3, "extort": 3, "external": 3, "extends": 3, "expressing": 3, "explain": 3, "expired": 3, "experienced": 3, "expecting": 3, "expectations": 3, "exhibit": 3, "exempt": 3, "exchanges": 3, "exceptions": 3, "excepting": 3, "exceeding": 3, "everybody": 3, "euclid": 3, "especial": 3, "escaping": 3, "error": 3, "eradicating": 3, "enthusiasm": 3, "enterprising": 3, "enlightened": 3, "enlarged": 3, "enforcing": 3, "endowment": 3, "endowed": 3, "ended": 3, "endangering": 3, "endangered": 3, "enactments": 3, "enact": 3, "employ": 3, "embolden": 3, "elegant": 3, "efficient": 3, "eels": 3, "educated": 3, "eastward": 3, "earthly": 3, "earnestness": 3, "earlier": 3, "dynasty": 3, "duration": 3, "drowned": 3, "drew": 3, "draft": 3, "dozen": 3, "doubting": 3, "domain": 3, "dollar": 3, "dogmas": 3, "divisions": 3, "distributed": 3, "distracted": 3, "distance": 3, "disrespect": 3, "disregarded": 3, "disprove": 3, "dislike": 3, "discussing": 3, "discards": 3, "disapproved": 3, "disappear": 3, "diligently": 3, "dignity": 3, "devised": 3, "developments": 3, "developed": 3, "determine": 3, "determination": 3, "detention": 3, "details": 3, "detailed": 3, "destroyed": 3, "destitution": 3, "designs": 3, "deserted": 3, "descent": 3, "deprives": 3, "deprecated": 3, "deportation": 3, "demonstrate": 3, "demerit": 3, "delivery": 3, "deliverance": 3, "delay": 3, "dekalb": 3, "defined": 3, "decrease": 3, "deceptive": 3, "deception": 3, "deceived": 3, "deceive": 3, "daring": 3, "customs": 3, "customary": 3, "cursed": 3, "cultivation": 3, "cuba": 3, "cub": 3, "cry": 3, "creditors": 3, "crazy": 3, "coupled": 3, "corps": 3, "corporal": 3, "copied": 3, "cook": 3, "conveys": 3, "continuing": 3, "continuance": 3, "continually": 3, "contested": 3, "contented": 3, "contends": 3, "contemporaneous": 3, "contemplates": 3, "contains": 3, "construe": 3, "constituting": 3, "conservative": 3, "consented": 3, "conflicting": 3, "confine": 3, "confidently": 3, "confer": 3, "condemnation": 3, "concurrence": 3, "concessions": 3, "conceivable": 3, "comprehend": 3, "compensating": 3, "compass": 3, "commissioners": 3, "commanding": 3, "combined": 3, "combat": 3, "collection": 3, "closely": 3, "cleared": 3, "clauses": 3, "claimants": 3, "circumstance": 3, "circuits": 3, "chords": 3, "cherished": 3, "check": 3, "cheapest": 3, "charged": 3, "chapter": 3, "chaplains": 3, "channel": 3, "chamber": 3, "chain": 3, "cessation": 3, "certainty": 3, "ceding": 3, "catching": 3, "cass": 3, "cargo": 3, "cancer": 3, "calhoun": 3, "calculated": 3, "burned": 3, "bureau": 3, "bull": 3, "brooks": 3, "brethren": 3, "bounties": 3, "bounds": 3, "borrowed": 3, "books": 3, "bloomington": 3, "bloodshed": 3, "blessings": 3, "blessing": 3, "bless": 3, "bleed": 3, "bitter": 3, "beware": 3, "betters": 3, "beneficent": 3, "belongs": 3, "belligerents": 3, "behind": 3, "begins": 3, "beautiful": 3, "beaten": 3, "bears": 3, "bayonet": 3, "bay": 3, "battles": 3, "barren": 3, "bark": 3, "ballots": 3, "baldwin": 3, "awhile": 3, "awarded": 3, "awake": 3, "available": 3, "avail": 3, "autumn": 3, "authorizing": 3, "authoritatively": 3, "attained": 3, "attacked": 3, "astounding": 3, "astonishing": 3, "assured": 3, "asserts": 3, "assembly": 3, "assaults": 3, "aside": 3, "ascribe": 3, "art": 3, "approve": 3, "approval": 3, "apprehend": 3, "appearance": 3, "anticipate": 3, "anti": 3, "announcing": 3, "angels": 3, "amending": 3, "amendatory": 3, "ambition": 3, "altered": 3, "aiding": 3, "agreeable": 3, "agents": 3, "afflicted": 3, "affection": 3, "affected": 3, "advocated": 3, "adversary": 3, "advantages": 3, "advantageous": 3, "advancing": 3, "adjudication": 3, "adjoining": 3, "adherents": 3, "addresses": 3, "additions": 3, "adaptations": 3, "active": 3, "acquiesce": 3, "accustomed": 3, "accurate": 3, "accords": 3, "accompanies": 3, "accident": 3, "accessible": 3, "accepting": 3, "abolishment": 3, "abandon": 3, "2d": 3, "1900": 3, "1857": 3, "1846": 3, "1821": 3, "1810": 3, "1800": 3, "1776": 3, "16th": 3, "youth": 2, "writer": 2, "write": 2, "wrest": 2, "wreath": 2, "wrapped": 2, "wounds": 2, "wouldn't": 2, "worn": 2, "workmen": 2, "won": 2, "women": 2, "wolf": 2, "witnessed": 2, "withheld": 2, "withered": 2, "wit": 2, "wishes": 2, "wing": 2, "windows": 2, "wills": 2, "willingly": 2, "wilderness": 2, "widows": 2, "wheedling": 2, "webster's": 2, "web": 2, "weapons": 2, "weakness": 2, "weakened": 2, "weaken": 2, "we'll": 2, "wax": 2, "wavered": 2, "watery": 2, "water": 2, "watching": 2, "watchful": 2, "watches": 2, "watch": 2, "warranted": 2, "warlike": 2, "warfare": 2, "waive": 2, "waging": 2, "volunteers": 2, "volumes": 2, "volume": 2, "voltaire": 2, "vocations": 2, "vocation": 2, "viz": 2, "visit": 2, "visible": 2, "virginians": 2, "virginian": 2, "violent": 2, "violations": 2, "violates": 2, "villainy": 2, "villainous": 2, "vigorously": 2, "vigilance": 2, "vested": 2, "verb": 2, "venerated": 2, "vaunted": 2, "vanquished": 2, "van": 2, "valiantly": 2, "valiant": 2, "vague": 2, "uttering": 2, "usually": 2, "useless": 2, "unvexed": 2, "unthought": 2, "unsoundness": 2, "unselfishly": 2, "unsatisfactory": 2, "unpaid": 2, "unnoticed": 2, "unnecessarily": 2, "unnatural": 2, "unknown": 2, "universe": 2, "universally": 2, "unimpaired": 2, "unhesitatingly": 2, "unforeseen": 2, "unfinished": 2, "unfairly": 2, "unfair": 2, "uneasiness": 2, "undoubting": 2, "undone": 2, "undisturbed": 2, "undistinguished": 2, "undertaken": 2, "undermining": 2, "undenied": 2, "undeniable": 2, "unconscious": 2, "unconditionally": 2, "uncommon": 2, "uncle": 2, "unchanged": 2, "unaware": 2, "unavoidable": 2, "unavailing": 2, "unalterably": 2, "unaccountable": 2, "types": 2, "twins": 2, "turks": 2, "truths": 2, "trulya": 2, "troubled": 2, "trot": 2, "trifling": 2, "trifled": 2, "tribunal": 2, "trials": 2, "trembling": 2, "tree": 2, "treasure": 2, "treasonable": 2, "treason": 2, "tread": 2, "travelled": 2, "transportation": 2, "transplanted": 2, "transmit": 2, "transformed": 2, "transform": 2, "transferred": 2, "transcript": 2, "transcending": 2, "tranquilized": 2, "tranquility": 2, "training": 2, "train": 2, "trailed": 2, "trading": 2, "traded": 2, "tracks": 2, "track": 2, "towit": 2, "touched": 2, "torch": 2, "top": 2, "tons": 2, "tomb": 2, "toleration": 2, "timidity": 2, "thunderstruck": 2, "thunder": 2, "throttled": 2, "throbbings": 2, "thrice": 2, "threats": 2, "threat": 2, "thoroughly": 2, "thompson": 2, "theretofore": 2, "themes": 2, "tested": 2, "terror": 2, "terrible": 2, "terminate": 2, "tenth": 2, "tenons": 2, "tendering": 2, "tender": 2, "temerity": 2, "teeming": 2, "technically": 2, "teaching": 2, "taunted": 2, "tatters": 2, "talks": 2, "tail": 2, "sympathisers": 2, "sympathised": 2, "syllogistic": 2, "syllable": 2, "swimmingly": 2, "swells": 2, "sweet": 2, "sweeping": 2, "swallowed": 2, "survive": 2, "surrounding": 2, "surrounded": 2, "surpassingly": 2, "surpassed": 2, "suppressing": 2, "supposition": 2, "supposing": 2, "supports": 2, "supplied": 2, "superintendency": 2, "superceded": 2, "sunny": 2, "sullen": 2, "suggests": 2, "suffrage": 2, "suffocation": 2, "suffering": 2, "suffer": 2, "suddenly": 2, "successor": 2, "successively": 2, "substituted": 2, "substantiate": 2, "subsidiary": 2, "subsided": 2, "stunned": 2, "studying": 2, "study": 2, "studied": 2, "strove": 2, "strode": 2, "string": 2, "striking": 2, "street": 2, "strangled": 2, "strange": 2, "stout": 2, "storms": 2, "stone's": 2, "stone": 2, "stock": 2, "stipulated": 2, "steamers": 2, "steam": 2, "status": 2, "statistics": 2, "statistical": 2, "stations": 2, "starving": 2, "starting": 2, "stares": 2, "stages": 2, "squatted": 2, "squadrons": 2, "squabble": 2, "sprang": 2, "spoiled": 2, "speed": 2, "speculating": 2, "specious": 2, "specimens": 2, "specify": 2, "specified": 2, "specie": 2, "speaker's": 2, "spanish": 2, "sovereign": 2, "southerners": 2, "sounds": 2, "soundness": 2, "sounded": 2, "souls": 2, "sorts": 2, "solved": 2, "solitude": 2, "solemn": 2, "solely": 2, "soiled": 2, "sobriquet": 2, "sneering": 2, "sneer": 2, "snaky": 2, "smith": 2, "smaller": 2, "skin": 2, "skilled": 2, "sixth": 2, "sitting": 2, "sit": 2, "sincere": 2, "silver": 2, "silly": 2, "silenced": 2, "signified": 2, "sighted": 2, "sight": 2, "sigh": 2, "siamese": 2, "shuffled": 2, "shrinking": 2, "shrink": 2, "shorten": 2, "shooters": 2, "shocks": 2, "shocked": 2, "shock": 2, "shiver": 2, "shining": 2, "shifted": 2, "shift": 2, "sheet": 2, "shed": 2, "shamed": 2, "shaken": 2, "shades": 2, "severe": 2, "sets": 2, "servant": 2, "seriousness": 2, "sentinel": 2, "sensibility": 2, "senses": 2, "senatorial": 2, "selfishness": 2, "seekers": 2, "seats": 2, "seal": 2, "scythe": 2, "scrap": 2, "scouted": 2, "scott's": 2, "scorned": 2, "schemes": 2, "scene": 2, "scale": 2, "saves": 2, "satisfactorily": 2, "sarcasm": 2, "sank": 2, "sanctioned": 2, "sam's": 2, "salvation": 2, "salle": 2, "sakes": 2, "saint": 2, "sailors": 2, "sagacious": 2, "safer": 2, "sad": 2, "runs": 2, "ruling": 2, "rufus": 2, "rot": 2, "rollick": 2, "robe": 2, "risked": 2, "ring": 2, "rightly": 2, "righteous": 2, "ridicule": 2, "reynolds": 2, "revive": 2, "revere": 2, "revenge": 2, "reunion": 2, "retrograde": 2, "retires": 2, "retaliates": 2, "resumption": 2, "resumed": 2, "resume": 2, "restricted": 2, "restraints": 2, "restraining": 2, "restrained": 2, "resting": 2, "respond": 2, "respectful": 2, "respectably": 2, "resolute": 2, "reside": 2, "reserve": 2, "reservations": 2, "resent": 2, "resembling": 2, "resemblance": 2, "repurify": 2, "repudiation": 2, "repudiate": 2, "reproached": 2, "reprieve": 2, "represented": 2, "reporter": 2, "repetition": 2, "repel": 2, "renown": 2, "rendering": 2, "rend": 2, "remunerative": 2, "remodeled": 2, "remembers": 2, "remarked": 2, "relish": 2, "release": 2, "relationship": 2, "rejecting": 2, "reinstate": 2, "regretted": 2, "regiment": 2, "refuting": 2, "refusal": 2, "refugees": 2, "reeled": 2, "reduction": 2, "redress": 2, "reddick": 2, "recreant": 2, "recovery": 2, "reconsider": 2, "recognizing": 2, "reckon": 2, "reciprocated": 2, "recess": 2, "recall": 2, "rebels": 2, "reasoned": 2, "realized": 2, "realization": 2, "reaccept": 2, "rational": 2, "rarely": 2, "rare": 2, "rapidity": 2, "ranks": 2, "rallying": 2, "rally": 2, "raising": 2, "quit": 2, "quincy": 2, "questioning": 2, "quelled": 2, "qualifying": 2, "pushing": 2, "pursuits": 2, "pursuance": 2, "purporting": 2, "purported": 2, "pulpit": 2, "pull": 2, "prussia": 2, "protracted": 2, "protested": 2, "prospects": 2, "props": 2, "propounding": 2, "propound": 2, "prompted": 2, "prompt": 2, "projector": 2, "progressing": 2, "progressed": 2, "profoundest": 2, "profitably": 2, "professes": 2, "products": 2, "productions": 2, "product": 2, "procured": 2, "procure": 2, "proclamations": 2, "problematical": 2, "problem": 2, "prizes": 2, "pretense": 2, "president's": 2, "presenting": 2, "presentation": 2, "prescribes": 2, "prepare": 2, "preparatory": 2, "preparations": 2, "preliminary": 2, "preferred": 2, "predominant": 2, "preconcert": 2, "precept": 2, "preceded": 2, "preamble": 2, "preacher": 2, "pre": 2, "prayers": 2, "pray": 2, "practices": 2, "pouring": 2, "potomac": 2, "postage": 2, "possessory": 2, "positively": 2, "populated": 2, "poised": 2, "poet": 2, "plowing": 2, "pleased": 2, "pleaded": 2, "platt": 2, "planting": 2, "plans": 2, "plagues": 2, "pitchfork": 2, "piled": 2, "pigeon": 2, "physically": 2, "philanthropy": 2, "phases": 2, "phase": 2, "pharaoh's": 2, "petty": 2, "persuasion": 2, "persisting": 2, "perpetuating": 2, "perpetuated": 2, "perjury": 2, "performed": 2, "performance": 2, "perchance": 2, "perceiving": 2, "peoria": 2, "pens": 2, "penetrating": 2, "penalties": 2, "peep": 2, "peculiarly": 2, "peculiarity": 2, "peacefully": 2, "patterns": 2, "patronage": 2, "patient": 2, "pathos": 2, "pathetic": 2, "patents": 2, "patent": 2, "pat": 2, "passions": 2, "passenger": 2, "partnership": 2, "partners": 2, "partizans": 2, "participate": 2, "partial": 2, "parted": 2, "parliamentary": 2, "parents": 2, "parent": 2, "pardons": 2, "pardoned": 2, "paraphrases": 2, "paragraph": 2, "pampered": 2, "palpable": 2, "palliation": 2, "pale": 2, "pains": 2, "pages": 2, "pacificator": 2, "owning": 2, "overslaugh": 2, "outlets": 2, "otis": 2, "other's": 2, "orphans": 2, "originating": 2, "originality": 2, "organize": 2, "organizations": 2, "orator": 2, "oration": 2, "option": 2, "operated": 2, "onset": 2, "onerous": 2, "omitting": 2, "omits": 2, "ometer": 2, "oldest": 2, "older": 2, "oil": 2, "oh": 2, "offend": 2, "occurs": 2, "occurring": 2, "occupying": 2, "obtain": 2, "obstinately": 2, "obstinacy": 2, "observed": 2, "obliterated": 2, "obey": 2, "novel": 2, "notions": 2, "notable": 2, "northward": 2, "northampton": 2, "nominee": 2, "nominations": 2, "nomination": 2, "nominate": 2, "noes": 2, "ninety": 2, "niggers": 2, "nigger": 2, "nerved": 2, "nelson": 2, "negroe's": 2, "negociating": 2, "neglect": 2, "nay": 2, "nathan": 2, "narrowest": 2, "namesthat": 2, "mystery": 2, "mysteries": 2, "muzzling": 2, "muskets": 2, "murfreesboro": 2, "murder": 2, "multiply": 2, "multiplied": 2, "muddy": 2, "mourns": 2, "mould": 2, "mothers": 2, "mortices": 2, "morocco": 2, "modify": 2, "modes": 2, "moderate": 2, "modelled": 2, "mistaking": 2, "mistakes": 2, "missourians": 2, "mission": 2, "misrepresented": 2, "misnamed": 2, "misery": 2, "miracle": 2, "minors": 2, "minded": 2, "microscope": 2, "metals": 2, "messages": 2, "merest": 2, "merchandise": 2, "men's": 2, "meliorate": 2, "melancholy": 2, "meade's": 2, "mchenry": 2, "mcclellan": 2, "mayo": 2, "maxims": 2, "maturity": 2, "maternal": 2, "materials": 2, "mastering": 2, "massacre": 2, "marrying": 2, "marks": 2, "markets": 2, "marion's": 2, "margins": 2, "marginal": 2, "mantle": 2, "mansion": 2, "manhood": 2, "manfully": 2, "mammon": 2, "malignant": 2, "malicious": 2, "male": 2, "majestic": 2, "magic": 2, "maddening": 2, "lyman": 2, "loyalty": 2, "louder": 2, "losses": 2, "loosened": 2, "logs": 2, "logic": 2, "location": 2, "located": 2, "locality": 2, "loan": 2, "loaf": 2, "lo": 2, "livery": 2, "listen": 2, "limitations": 2, "limb": 2, "lifted": 2, "lieutenant": 2, "licenced": 2, "liberation": 2, "liberates": 2, "liars": 2, "lever": 2, "level": 2, "lent": 2, "lengthy": 2, "lengths": 2, "leisure": 2, "legitimate": 2, "legislatures": 2, "lays": 2, "lawyer": 2, "lawless": 2, "laudable": 2, "lately": 2, "lasting": 2, "langdon": 2, "lain": 2, "laborious": 2, "knocking": 2, "knocked": 2, "knives": 2, "kindreds": 2, "kindred": 2, "kindness": 2, "kickings": 2, "kentuckians": 2, "keepers": 2, "kaskaskia": 2, "justifies": 2, "jurors": 2, "judging": 2, "judged": 2, "juan": 2, "jotted": 2, "jogging": 2, "jewel": 2, "jealousies": 2, "irritations": 2, "irritating": 2, "irresistible": 2, "irony": 2, "involuntarily": 2, "invoke": 2, "inviting": 2, "invests": 2, "inventive": 2, "intimate": 2, "intestine": 2, "interoceanic": 2, "intermission": 2, "intermarry": 2, "institute": 2, "instinctively": 2, "installments": 2, "installment": 2, "inspires": 2, "inspired": 2, "inspiration": 2, "insinuations": 2, "insinuated": 2, "inscrutable": 2, "insane": 2, "inoperative": 2, "inland": 2, "injured": 2, "injure": 2, "inherent": 2, "ingenuity": 2, "ingeniously": 2, "inflicted": 2, "inflexible": 2, "inferences": 2, "inexhaustible": 2, "inevitable": 2, "ineffectual": 2, "industry": 2, "indulged": 2, "inducements": 2, "indomitable": 2, "indisposition": 2, "indirect": 2, "indifferent": 2, "indication": 2, "indicate": 2, "incumbent": 2, "incomplete": 2, "income": 2, "inclining": 2, "incipient": 2, "inciding": 2, "inception": 2, "inaugurate": 2, "inadequate": 2, "improvements": 2, "improper": 2, "impressed": 2, "impose": 2, "imperilled": 2, "imperative": 2, "impassioned": 2, "immortality": 2, "imaginable": 2, "illustrates": 2, "ills": 2, "ignorant": 2, "idaho": 2, "hunting": 2, "humored": 2, "humbug": 2, "household": 2, "hotel": 2, "hosts": 2, "hospitals": 2, "horribly": 2, "horns": 2, "horned": 2, "hooted": 2, "honored": 2, "hogs": 2, "historically": 2, "hides": 2, "hid": 2, "hewing": 2, "heresies": 2, "henry's": 2, "henceforth": 2, "hemmed": 2, "hemispheres": 2, "hedged": 2, "heavens": 2, "hearers": 2, "hazzard": 2, "hazard": 2, "haughty": 2, "hath": 2, "hateful": 2, "hat": 2, "harrison": 2, "harris's": 2, "harmony": 2, "harmonized": 2, "harder": 2, "happened": 2, "hanging": 2, "handkerchief": 2, "hancock": 2, "halting": 2, "hallowed": 2, "habitually": 2, "habeas": 2, "guided": 2, "guess": 2, "guarantees": 2, "growth": 2, "grows": 2, "grocery": 2, "gripe": 2, "greeks": 2, "greedy": 2, "greece": 2, "gratz": 2, "gratulation": 2, "grasping": 2, "grants": 2, "granting": 2, "graceful": 2, "grabbed": 2, "governors": 2, "government's": 2, "governing": 2, "gov": 2, "god's": 2, "gloomy": 2, "gloom": 2, "glimpse": 2, "giddings": 2, "giant": 2, "gentiles": 2, "galena": 2, "gag": 2, "funded": 2, "functionaries": 2, "fun": 2, "fullness": 2, "fulfilment": 2, "frustrated": 2, "fruitful": 2, "fresher": 2, "frequent": 2, "fremont": 2, "freesoiler": 2, "freeing": 2, "freedom's": 2, "frankness": 2, "francisco": 2, "fraction": 2, "founders": 2, "foul": 2, "forming": 2, "formally": 2, "forgeries": 2, "forestall": 2, "foresaw": 2, "forbore": 2, "forbidding": 2, "forbearance": 2, "flutter": 2, "flushed": 2, "fluctuations": 2, "flies": 2, "flattered": 2, "flat": 2, "fitness": 2, "firing": 2, "finish": 2, "finest": 2, "fillmore": 2, "figure": 2, "figuratively": 2, "fiftieth": 2, "fifths": 2, "feud": 2, "fervently": 2, "female": 2, "feels": 2, "fears": 2, "fashion": 2, "farm": 2, "fanatic": 2, "falsifies": 2, "falls": 2, "fairness": 2, "failures": 2, "facilities": 2, "faces": 2, "fabric": 2, "extracted": 2, "extra": 2, "extinguished": 2, "extinguish": 2, "extensively": 2, "extant": 2, "expiration": 2, "experiment": 2, "expects": 2, "expatriation": 2, "expand": 2, "exorcised": 2, "exonerate": 2, "exigencies": 2, "exhibiting": 2, "exercising": 2, "execration": 2, "exchange": 2, "excesses": 2, "excess": 2, "exceeded": 2, "examples": 2, "exaggerated": 2, "eventful": 2, "evening": 2, "evanescent": 2, "eternity": 2, "eternally": 2, "estopped": 2, "estimation": 2, "estimates": 2, "essentially": 2, "essence": 2, "espouse": 2, "eschewed": 2, "eradicated": 2, "enumerated": 2, "entrapping": 2, "entrapped": 2, "entranced": 2, "enos": 2, "enlargement": 2, "enhance": 2, "engineers": 2, "engineering": 2, "engaging": 2, "engage": 2, "enduringly": 2, "ends": 2, "endorsing": 2, "endorsed": 2, "encouraging": 2, "encourage": 2, "enchanted": 2, "employing": 2, "emphatically": 2, "emphatic": 2, "emperors": 2, "emperor": 2, "eminently": 2, "eminent": 2, "eminence": 2, "emigrated": 2, "emergencies": 2, "embraces": 2, "embitter": 2, "embellishment": 2, "embarrassments": 2, "embarrassment": 2, "emancipating": 2, "elevation": 2, "elevating": 2, "elapsed": 2, "eighth": 2, "effecting": 2, "economy": 2, "eccentricities": 2, "earth's": 2, "earns": 2, "ear": 2, "dynasties": 2, "dwelt": 2, "dust": 2, "durability": 2, "dupage": 2, "dues": 2, "dry": 2, "drove": 2, "driving": 2, "drill": 2, "dress": 2, "drayman": 2, "dougherty": 2, "doomed": 2, "domination": 2, "dominates": 2, "dogmatically": 2, "dodge": 2, "divorced": 2, "divest": 2, "diversity": 2, "diversities": 2, "disunionists": 2, "disunionist": 2, "districts": 2, "distribution": 2, "distasteful": 2, "dissatisfaction": 2, "disregarding": 2, "disposal": 2, "displaced": 2, "displace": 2, "dishonest": 2, "discrimination": 2, "discoveries": 2, "discourage": 2, "disciplined": 2, "discipline": 2, "discern": 2, "discard": 2, "disasters": 2, "disagreement": 2, "disagreeable": 2, "disagree": 2, "dirty": 2, "diplomatic": 2, "diminishing": 2, "differing": 2, "dictum": 2, "devote": 2, "devising": 2, "determining": 2, "detection": 2, "destructive": 2, "destitute": 2, "desperation": 2, "despair": 2, "desirous": 2, "designation": 2, "designate": 2, "desert": 2, "deriving": 2, "derives": 2, "derive": 2, "derangement": 2, "deplored": 2, "deplorable": 2, "depended": 2, "denouncing": 2, "denmark": 2, "denies": 2, "demonstrating": 2, "demonstrates": 2, "demonstrated": 2, "demon": 2, "deliver": 2, "deliberately": 2, "delegated": 2, "delegate": 2, "delayed": 2, "defining": 2, "defied": 2, "deferring": 2, "deferred": 2, "defended": 2, "default": 2, "deepest": 2, "deed": 2, "deduction": 2, "deducted": 2, "deduce": 2, "dedicate": 2, "decreased": 2, "declamation": 2, "decisively": 2, "decides": 2, "decennial": 2, "deceitful": 2, "deceased": 2, "debtors": 2, "deals": 2, "dealers": 2, "david": 2, "darkness": 2, "damp": 2, "damages": 2, "dalliance": 2, "dakota": 2, "cuttlefish": 2, "cutting": 2, "custom": 2, "curtis": 2, "curious": 2, "cured": 2, "cure": 2, "crittenden": 2, "criticism": 2, "criticising": 2, "crises": 2, "credible": 2, "creator": 2, "creating": 2, "crawls": 2, "cranberries": 2, "cover": 2, "courses": 2, "country's": 2, "corroboration": 2, "correctness": 2, "corrected": 2, "corpus": 2, "cordiality": 2, "cooperating": 2, "cooperate": 2, "convulsions": 2, "convulsion": 2, "convinced": 2, "conviction": 2, "convicted": 2, "convertibility": 2, "controversies": 2, "controllers": 2, "contrivances": 2, "contributed": 2, "contradiction": 2, "contradicted": 2, "contradict": 2, "contracts": 2, "contracted": 2, "contempt": 2, "contact": 2, "construing": 2, "construes": 2, "construed": 2, "constructing": 2, "constrained": 2, "constituency": 2, "constituencies": 2, "conspiring": 2, "conspire": 2, "conspicuously": 2, "conspicuous": 2, "consistently": 2, "considerate": 2, "considerably": 2, "conservatism": 2, "conscious": 2, "conquest": 2, "conquered": 2, "connecting": 2, "connect": 2, "conkling": 2, "congenial": 2, "confounded": 2, "conformity": 2, "confiscate": 2, "confesses": 2, "conferred": 2, "confederate": 2, "conditionally": 2, "condemned": 2, "condemn": 2, "concocted": 2, "concealment": 2, "conceal": 2, "comprising": 2, "compose": 2, "compliance": 2, "complexion": 2, "completely": 2, "compelled": 2, "compeers": 2, "compatible": 2, "compare": 2, "companions": 2, "companies": 2, "communications": 2, "commotions": 2, "committing": 2, "commits": 2, "commented": 2, "commending": 2, "commended": 2, "commemorate": 2, "comfort": 2, "cometh": 2, "combatants": 2, "column": 2, "colony": 2, "colonial": 2, "colombia": 2, "collected": 2, "collect": 2, "coercion": 2, "coerced": 2, "clinton": 2, "clergyman": 2, "clenched": 2, "cleaver": 2, "citizenship": 2, "circulating": 2, "cincinnati": 2, "church": 2, "chose": 2, "cheese": 2, "cheer": 2, "channels": 2, "changes": 2, "chambers": 2, "challenging": 2, "chair": 2, "ceremony": 2, "center": 2, "cements": 2, "cede": 2, "ceaselessly": 2, "causing": 2, "causelessly": 2, "carved": 2, "carolinian": 2, "captors": 2, "captivating": 2, "captain": 2, "canonized": 2, "canonical": 2, "cancel": 2, "campbell's": 2, "calmly": 2, "buys": 2, "butcher's": 2, "burning": 2, "burn": 2, "buren": 2, "bullets": 2, "bullet": 2, "built": 2, "buffaloes": 2, "brown's": 2, "brow": 2, "brotherhood": 2, "brooklyn": 2, "brooding": 2, "broke": 2, "broils": 2, "britannic": 2, "brightly": 2, "bright": 2, "briefer": 2, "bridges": 2, "bred": 2, "breathed": 2, "breaks": 2, "brazil": 2, "bravely": 2, "bowie": 2, "bowen": 2, "bow": 2, "boundaries": 2, "bought": 2, "bosoms": 2, "borrowing": 2, "bolivar": 2, "bolder": 2, "bodies": 2, "board": 2, "blows": 2, "blowing": 2, "blockading": 2, "blank": 2, "blair": 2, "blackguardism": 2, "bit": 2, "biography": 2, "bind": 2, "billows": 2, "bier": 2, "bibles": 2, "betrayed": 2, "benighted": 2, "beneficial": 2, "below": 2, "bell": 2, "behemoth": 2, "begun": 2, "beginner": 2, "befall": 2, "bearded": 2, "bayou": 2, "battling": 2, "batteries": 2, "barrel": 2, "bare": 2, "barbarous": 2, "baptized": 2, "baptist": 2, "banned": 2, "ban": 2, "bagatelle": 2, "backers": 2, "ayes": 2, "axioms": 2, "axe": 2, "awakened": 2, "await": 2, "avowal": 2, "avow": 2, "averted": 2, "authorizes": 2, "auspices": 2, "augments": 2, "augmenting": 2, "augment": 2, "aught": 2, "attracted": 2, "attain": 2, "attacks": 2, "atrocities": 2, "atmosphere": 2, "astounded": 2, "associations": 2, "assist": 2, "assertions": 2, "assembling": 2, "assaulted": 2, "assailants": 2, "aspects": 2, "ashes": 2, "ascertaining": 2, "arts": 2, "arriving": 2, "arrived": 2, "arresting": 2, "arrested": 2, "array": 2, "arranged": 2, "aroused": 2, "armored": 2, "arises": 2, "area": 2, "ardently": 2, "arbitration": 2, "arbitrarily": 2, "aptly": 2, "appropriates": 2, "appropriately": 2, "appropriate": 2, "apprized": 2, "apprehensions": 2, "appointments": 2, "applications": 2, "appendix": 2, "apostrophised": 2, "apology": 2, "aphorism": 2, "apart": 2, "anxiety": 2, "antithesis": 2, "antietam": 2, "antagonists": 2, "antagonisms": 2, "antagonism": 2, "anomalous": 2, "annum": 2, "annexing": 2, "annexed": 2, "angles": 2, "anderson": 2, "anchor": 2, "amply": 2, "amend": 2, "alternative": 2, "allusions": 2, "alludes": 2, "alliance": 2, "alleged": 2, "allayed": 2, "aided": 2, "ah": 2, "agrees": 2, "aggressors": 2, "aggressions": 2, "aggravation": 2, "ages": 2, "affirming": 2, "affirmatively": 2, "affinity": 2, "admitting": 2, "admiral": 2, "administrative": 2, "administering": 2, "adjournment": 2, "adjourning": 2, "adjacent": 2, "adhesion": 2, "adhering": 2, "adhered": 2, "adams": 2, "adam": 2, "actuated": 2, "actor": 2, "activity": 2, "acquiescing": 2, "acquainted": 2, "acquaintances": 2, "acknowledging": 2, "achievement": 2, "ace": 2, "accused": 2, "accusation": 2, "accuracy": 2, "accredited": 2, "accord": 2, "accompany": 2, "accommodated": 2, "accomac": 2, "access": 2, "academy": 2, "abundantly": 2, "absurdity": 2, "absent": 2, "abridgement": 2, "abolitionize": 2, "ablest": 2, "96,096,922.09": 2, "901,125,674.86": 2, "895,796,630.65": 2, "867,276": 2, "865,234,087.86": 2, "7th": 2, "700,000": 2, "7,211": 2, "581,813": 2, "55,094": 2, "500": 2, "5,329,044.21": 2, "470": 2, "433,643": 2, "384,984": 2, "36.45": 2, "32,679": 2, "3,982,011": 2, "28th": 2, "27th": 2, "274,567": 2, "25th": 2, "1930": 2, "1920": 2, "1847": 2, "1844": 2, "1842": 2, "1832": 2, "1831": 2, "1830": 2, "1829": 2, "1827": 2, "1825": 2, "1818": 2, "1816": 2, "1815": 2, "1814": 2, "1813": 2, "1811": 2, "181,086,635.07": 2, "1809": 2, "1806": 2, "1804": 2, "18,842,558.71": 2, "1794": 2, "1790": 2, "1789": 2, "1784": 2, "1777": 2, "15th": 2, "13,606,759.11": 2, "12th": 2, "117": 2, "103,208,415": 2, "100,000": 2, "1,456,514": 2, "yedo": 1, "yards": 1, "wrought": 1, "wrongs": 1, "wronged": 1, "wringing": 1, "wrested": 1, "wrench": 1, "wrangling": 1, "worship": 1, "woo": 1, "wont": 1, "wogley": 1, "wofully": 1, "withdrew": 1, "withdrawn": 1, "withdrawing": 1, "withdrawal": 1, "winnebago": 1, "windy": 1, "win": 1, "willingness": 1, "williamson": 1, "williams": 1, "willfully": 1, "wilful": 1, "wilcox": 1, "widow": 1, "wider": 1, "wickedness": 1, "whisper": 1, "whining": 1, "wherewith": 1, "whereabouts": 1, "whatsoever": 1, "welfare": 1, "welcome": 1, "weights": 1, "weigh": 1, "weeks": 1, "weary": 1, "weapon": 1, "waves": 1, "wavering": 1, "waterloo": 1, "wastes": 1, "washingtonian": 1, "warring": 1, "warred": 1, "warrant": 1, "wards": 1, "wanting": 1, "wandered": 1, "walk": 1, "waiving": 1, "waived": 1, "waited": 1, "vow": 1, "voss": 1, "vol": 1, "vociferate": 1, "visiting": 1, "visited": 1, "vindictive": 1, "vindication": 1, "vindicated": 1, "vindicate": 1, "villified": 1, "vigorous": 1, "viewing": 1, "vicious": 1, "vet": 1, "versa": 1, "verge": 1, "venturing": 1, "ventures": 1, "venezuela": 1, "vehemently": 1, "vaunt": 1, "varying": 1, "varieties": 1, "varied": 1, "valuation": 1, "validity": 1, "vagrant": 1, "vacant": 1, "utters": 1, "utmost": 1, "usual": 1, "usefulness": 1, "usefully": 1, "urgently": 1, "urgent": 1, "upheld": 1, "up.this": 1, "unwisely": 1, "unwillingly": 1, "unwelcome": 1, "untruth": 1, "untried": 1, "unsubstantial": 1, "unsound": 1, "unsecured": 1, "unsay": 1, "unsaid": 1, "unsafe": 1, "unreservedly": 1, "unrequited": 1, "unrepealed": 1, "unprofitable": 1, "unpleasant": 1, "unphilosophical": 1, "unoccupied": 1, "unmindful": 1, "unmade": 1, "unlimited": 1, "unkept": 1, "unjustly": 1, "unitarian": 1, "unintentional": 1, "unintended": 1, "uninhabitable": 1, "uninfluenced": 1, "unimportant": 1, "unhappily": 1, "ungrateful": 1, "ungenerous": 1, "unfrequently": 1, "unfettered": 1, "unexpected": 1, "unexhausted": 1, "unexecuted": 1, "unequivocally": 1, "undue": 1, "undoubted": 1, "undo": 1, "undeveloped": 1, "undertook": 1, "undergone": 1, "undefined": 1, "unconstitutionally": 1, "uncompromisingly": 1, "unchangeably": 1, "unchangeable": 1, "unbroken": 1, "unavowed": 1, "unanswerable": 1, "ult": 1, "tyrannical": 1, "tycoon": 1, "tybee": 1, "turner": 1, "turmoil": 1, "tumbled": 1, "tuesday": 1, "truthfulness": 1, "trump": 1, "truckee": 1, "trotting": 1, "tropics": 1, "trio": 1, "trifle": 1, "tributary": 1, "tribune": 1, "tribunals": 1, "tribe": 1, "triangles": 1, "triangle": 1, "tremble": 1, "treats": 1, "treacherous": 1, "traversed": 1, "traveling": 1, "travel": 1, "transpiring": 1, "transient": 1, "transfer": 1, "transactions": 1, "transaction": 1, "traitorous": 1, "traitor": 1, "traders": 1, "tracts": 1, "towns": 1, "tours": 1, "tortured": 1, "tordenskiold": 1, "toothless": 1, "tools": 1, "tones": 1, "toll": 1, "tolerating": 1, "tolerably": 1, "tolerable": 1, "titles": 1, "tiptop": 1, "timely": 1, "timber": 1, "tilt": 1, "tier": 1, "tie": 1, "ticket": 1, "thrusting": 1, "thronging": 1, "throats": 1, "throat": 1, "thing.this": 1, "thieves": 1, "thief": 1, "thickness": 1, "thickly": 1, "thereon": 1, "therefor": 1, "theory": 1, "theme": 1, "theirs": 1, "theater": 1, "texan": 1, "tests": 1, "testing": 1, "territorially": 1, "terribly": 1, "terrebonne": 1, "terminus": 1, "termination": 1, "tens": 1, "tennesseeans": 1, "tenderness": 1, "tendered": 1, "telegraphs": 1, "telegraphic": 1, "teems": 1, "taunting": 1, "talked": 1, "talent": 1, "tact": 1, "table": 1, "sword": 1, "sweeter": 1, "sweden": 1, "sweat": 1, "swears": 1, "sway": 1, "swarm": 1, "swallow": 1, "sustains": 1, "suspicions": 1, "suspicion": 1, "suspend": 1, "surviving": 1, "surveyors": 1, "surveying": 1, "survey": 1, "surround": 1, "surrendering": 1, "surprised": 1, "surpassing": 1, "surgeon": 1, "surest": 1, "supplying": 1, "supplemental": 1, "supplant": 1, "superiority": 1, "sunk": 1, "sung": 1, "sundry": 1, "sumner": 1, "sultan": 1, "sugarcane": 1, "succeeds": 1, "subtract": 1, "subtle": 1, "substitution": 1, "subordinate": 1, "subjugation": 1, "subjected": 1, "subdued": 1, "style": 1, "stuart": 1, "struggling": 1, "stroke": 1, "strictness": 1, "stretching": 1, "strengthen": 1, "streams": 1, "stream": 1, "straw": 1, "strangers": 1, "straits": 1, "strained": 1, "stormy": 1, "stopping": 1, "stolen": 1, "stirring": 1, "stirred": 1, "stir": 1, "stimulate": 1, "stick": 1, "stepped": 1, "steamships": 1, "steamship": 1, "steamer": 1, "stayed": 1, "stationary": 1, "starvation": 1, "stale": 1, "stain": 1, "stade": 1, "squadron": 1, "spots": 1, "spoil": 1, "spit": 1, "spirited": 1, "specs": 1, "specifications": 1, "sparingly": 1, "spares": 1, "spare": 1, "south.have": 1, "soup": 1, "sophistical": 1, "sons": 1, "solution": 1, "solicitude": 1, "solicited": 1, "solicit": 1, "sojourners": 1, "softly": 1, "sneers": 1, "sneered": 1, "snake": 1, "smiles": 1, "slower": 1, "sloping": 1, "slavery.but": 1, "slaver": 1, "slandered": 1, "skipped": 1, "skip": 1, "skeptical": 1, "sitana": 1, "sister": 1, "sioux": 1, "sinners": 1, "singular": 1, "sin": 1, "simultaneous": 1, "simplifying": 1, "silliman": 1, "sierra": 1, "sic": 1, "shred": 1, "shouts": 1, "shoulder": 1, "shortly": 1, "shorter": 1, "shores": 1, "shops": 1, "shoot": 1, "shook": 1, "shirking": 1, "shirk": 1, "shipwreck": 1, "shifting": 1, "shields": 1, "sherman's": 1, "sheer": 1, "sharp": 1, "shaping": 1, "shameful": 1, "shakes": 1, "shake": 1, "severing": 1, "severance": 1, "severally": 1, "servile": 1, "services": 1, "serpent's": 1, "seriatim": 1, "sensitive": 1, "seniors": 1, "senatorship": 1, "senator's": 1, "senate.this": 1, "senate's": 1, "semi": 1, "selfishly": 1, "selecting": 1, "select": 1, "sees": 1, "seeds": 1, "seceder": 1, "season": 1, "searched": 1, "scurrilous": 1, "scruples": 1, "scrip": 1, "screw": 1, "scout": 1, "scourge": 1, "scot": 1, "scientific": 1, "sciences": 1, "science": 1, "scheldt": 1, "scatter": 1, "scared": 1, "scare": 1, "scapegrace": 1, "saviour": 1, "savannah": 1, "satisfying": 1, "satan": 1, "sanitary": 1, "sand": 1, "sanctioning": 1, "sanctifying": 1, "sameness": 1, "salvador": 1, "sailing": 1, "safeguards": 1, "safeguard": 1, "sadness": 1, "sacrificed": 1, "sacramento": 1, "sabine": 1, "rutledge": 1, "russian": 1, "rumors": 1, "rulers": 1, "ruinous": 1, "ruffle": 1, "ruffians": 1, "royal": 1, "rousing": 1, "roundly": 1, "rounded": 1, "rome": 1, "robbing": 1, "robber": 1, "rob": 1, "roar": 1, "roads": 1, "risk": 1, "rinciple": 1, "rigorous": 1, "rigor": 1, "rigidly": 1, "righteously": 1, "rifles": 1, "rider": 1, "riddle": 1, "richmond": 1, "richly": 1, "richest": 1, "richer": 1, "rewritten": 1, "revives": 1, "revision": 1, "revised": 1, "reverses": 1, "reverence": 1, "retiring": 1, "retire": 1, "retard": 1, "retaining": 1, "resulted": 1, "rests": 1, "restrictions": 1, "restitution": 1, "responsibilities": 1, "response": 1, "respectability": 1, "resource": 1, "resorting": 1, "resolves": 1, "resigning": 1, "resignation": 1, "resign": 1, "residue": 1, "resident": 1, "resided": 1, "reserving": 1, "reservation": 1, "resentment": 1, "rescind": 1, "requote": 1, "requests": 1, "requesting": 1, "reputed": 1, "reputation": 1, "reptiles": 1, "reproduce": 1, "repressed": 1, "represents": 1, "repossession": 1, "reporting": 1, "replying": 1, "replies": 1, "replenishing": 1, "repentance": 1, "repelled": 1, "repaired": 1, "reorganizing": 1, "reorganize": 1, "renwick": 1, "renounced": 1, "renegade": 1, "remotest": 1, "remorseless": 1, "remodeling": 1, "remnant": 1, "remissions": 1, "reminding": 1, "remedied": 1, "remakrs": 1, "remake": 1, "relying": 1, "rely": 1, "reluctantly": 1, "reluctant": 1, "relinquishment": 1, "relieved": 1, "relax": 1, "relatively": 1, "relating": 1, "relates": 1, "related": 1, "rejects": 1, "reiterates": 1, "reiterated": 1, "reiterate": 1, "reintroducing": 1, "reimbursements": 1, "reimbursement": 1, "reimbursed": 1, "regulated": 1, "registered": 1, "regenerated": 1, "regards": 1, "regaining": 1, "refunding": 1, "refund": 1, "refuge": 1, "reform": 1, "reflections": 1, "reflect": 1, "reestablished": 1, "reestablish": 1, "reenslave": 1, "reenforcements": 1, "reenforced": 1, "reenforce": 1, "reduces": 1, "redressed": 1, "redemptions": 1, "redeemed": 1, "recurring": 1, "recurrence": 1, "recurred": 1, "recruits": 1, "recording": 1, "reconsideration": 1, "reconcile": 1, "recollections": 1, "recollecting": 1, "reclamations": 1, "reclamation": 1, "reckoning": 1, "reckoned": 1, "reciprocity": 1, "reciprocates": 1, "receivability": 1, "receipt": 1, "recede": 1, "recapture": 1, "recanted": 1, "recantation": 1, "recalled": 1, "rebutting": 1, "reattempted": 1, "reat": 1, "reassert": 1, "reassembled": 1, "reargument": 1, "rear": 1, "reality": 1, "readopted": 1, "readopt": 1, "readiness": 1, "readers": 1, "ray": 1, "ravages": 1, "rationally": 1, "rascality": 1, "rank": 1, "ranges": 1, "ranged": 1, "random": 1, "rambling": 1, "rallies": 1, "railways": 1, "railway": 1, "radically": 1, "radical": 1, "radiate": 1, "quoting": 1, "quota": 1, "quits": 1, "quieted": 1, "quibbles": 1, "quibble": 1, "quest": 1, "quartermaster": 1, "quarreling": 1, "quarreled": 1, "quantities": 1, "puzzled": 1, "puzzle": 1, "pursuing": 1, "pursue": 1, "pursuant": 1, "purchaser": 1, "pur": 1, "punctually": 1, "pugets": 1, "publish": 1, "proximity": 1, "provost": 1, "provokes": 1, "provoked": 1, "provisioning": 1, "prototype": 1, "protecting": 1, "prostituted": 1, "prosecuting": 1, "prosecute": 1, "proposes": 1, "proportionately": 1, "propagandism": 1, "pronouncing": 1, "pronounces": 1, "promulgated": 1, "promptitude": 1, "promotive": 1, "promoted": 1, "prominently": 1, "prominence": 1, "projects": 1, "prohibitions": 1, "progresses": 1, "profoundly": 1, "profits": 1, "proffered": 1, "professions": 1, "professed": 1, "producers": 1, "prodded": 1, "procuring": 1, "processes": 1, "privateering": 1, "prisoners": 1, "princeton": 1, "princess": 1, "primarily": 1, "prices": 1, "prevention": 1, "prevailing": 1, "pretends": 1, "presumptuous": 1, "presumption": 1, "presumed": 1, "pressure": 1, "presidents": 1, "presided": 1, "presbyterian": 1, "prerequisite": 1, "preponderating": 1, "prepetuity": 1, "prepares": 1, "preparation": 1, "premium": 1, "premature": 1, "prejudices": 1, "preference": 1, "preferable": 1, "prefer": 1, "preeminently": 1, "prediction": 1, "predetermination": 1, "predecessor": 1, "precision": 1, "precipitate": 1, "precepts": 1, "precautionary": 1, "precaution": 1, "prairie": 1, "practicing": 1, "powerless": 1, "powerfully": 1, "poverty": 1, "pot": 1, "posts": 1, "postmasters": 1, "posterity": 1, "possessions": 1, "possesses": 1, "portugal": 1, "portsmouth": 1, "polls": 1, "politely": 1, "polite": 1, "plighting": 1, "plighted": 1, "plied": 1, "plenitude": 1, "pleasing": 1, "pleasantly": 1, "pleasant": 1, "plaster": 1, "plaquemines": 1, "plants": 1, "plainer": 1, "pity": 1, "pitchers": 1, "pistol": 1, "pins": 1, "pill": 1, "pile": 1, "pierce": 1, "physicians": 1, "phoenix": 1, "philosophical": 1, "philanthropic": 1, "philadelphia": 1, "petting": 1, "petitions": 1, "petition": 1, "perverted": 1, "perversions": 1, "perversion": 1, "perverse": 1, "peruse": 1, "perthshire": 1, "pertains": 1, "pertain": 1, "persuade": 1, "perspicuity": 1, "persistently": 1, "persistent": 1, "persevere": 1, "perplexities": 1, "perpetually": 1, "perpetration": 1, "permanency": 1, "perjured": 1, "periodically": 1, "performing": 1, "performances": 1, "perfection": 1, "percent": 1, "pensacola": 1, "penniless": 1, "peninsular": 1, "pendency": 1, "pays": 1, "paymaster": 1, "payable": 1, "pause": 1, "patting": 1, "pattern": 1, "patronized": 1, "patriots": 1, "patrick": 1, "patiently": 1, "paths": 1, "pastor": 1, "passeth": 1, "passages": 1, "pasha": 1, "parting": 1, "particulars": 1, "parishes": 1, "paris": 1, "parchment": 1, "paramount": 1, "paraguay": 1, "parading": 1, "parades": 1, "par": 1, "pan": 1, "palliated": 1, "painfully": 1, "paens": 1, "oysters": 1, "overwhelmingly": 1, "overwhelmed": 1, "overspreading": 1, "overslaughing": 1, "overruled": 1, "overreach": 1, "overlook": 1, "overland": 1, "overflowed": 1, "overcome": 1, "outnumbering": 1, "outlying": 1, "outlay": 1, "outlaws": 1, "outlandish": 1, "outgrown": 1, "outbreak": 1, "orsini's": 1, "orphan": 1, "orlando": 1, "originator": 1, "originate": 1, "oriental": 1, "ordnance": 1, "ordinay": 1, "ordinances": 1, "ordered": 1, "ordaining": 1, "oracle": 1, "oppression": 1, "opponent": 1, "opens": 1, "omnipotent": 1, "omnipotence": 1, "omissions": 1, "omaha": 1, "oft": 1, "officce": 1, "offering": 1, "of____for": 1, "occurrence": 1, "occupations": 1, "occasioned": 1, "occasional": 1, "obviously": 1, "obviate": 1, "obtuse": 1, "obstructions": 1, "obstructed": 1, "obstinate": 1, "obstacle": 1, "observe": 1, "observations": 1, "observation": 1, "obscures": 1, "obnoxious": 1, "obliges": 1, "oblige": 1, "obeying": 1, "o'clock": 1, "numerical": 1, "nullifying": 1, "nullify": 1, "novelty": 1, "notifying": 1, "notified": 1, "noteworthy": 1, "noted": 1, "norwegian": 1, "northwest": 1, "northeastern": 1, "nods": 1, "nicholasville": 1, "nice": 1, "netherlands": 1, "net": 1, "nest": 1, "neil": 1, "neighbors": 1, "neighboring": 1, "negotiations": 1, "neglected": 1, "negatively": 1, "negations": 1, "needless": 1, "neck": 1, "nebr": 1, "navigable": 1, "naturalizations": 1, "naturalization": 1, "nationality": 1, "napier": 1, "naperville": 1, "mystic": 1, "mysterious": 1, "mutters": 1, "mustered": 1, "murderers": 1, "murderer": 1, "munitions": 1, "moving": 1, "movers": 1, "movable": 1, "mouse": 1, "mourning": 1, "mounting": 1, "moulds": 1, "morrow": 1, "morris": 1, "morning": 1, "morals": 1, "morality": 1, "montgomery": 1, "montana": 1, "monroe": 1, "monopoly": 1, "monday": 1, "monarchy": 1, "monarchies": 1, "molding": 1, "modern": 1, "moderator": 1, "moderately": 1, "models": 1, "mobs": 1, "mob": 1, "missing": 1, "misrepresents": 1, "misrepresentations": 1, "misrepresentation": 1, "misled": 1, "mislead": 1, "misconstruction": 1, "mischievous": 1, "misapprehensions": 1, "misapprehension": 1, "mints": 1, "ministers": 1, "migrate": 1, "mifflin": 1, "midshipmen": 1, "methods": 1, "methodist": 1, "merged": 1, "merchants": 1, "mental": 1, "menacing": 1, "menaces": 1, "memorial": 1, "memoranda": 1, "melioration": 1, "medium": 1, "mechanics": 1, "meantime": 1, "meanly": 1, "mcdonough": 1, "mcdonell": 1, "maturely": 1, "mature": 1, "mattoon": 1, "matteson": 1, "mathematically": 1, "mates": 1, "mason": 1, "mary": 1, "marshall": 1, "marshal": 1, "marry": 1, "manufactured": 1, "manufacture": 1, "manly": 1, "manifold": 1, "manifests": 1, "manifestations": 1, "majorities": 1, "majesty": 1, "magnificent": 1, "magnanimously": 1, "magnanimous": 1, "magnanimity": 1, "magical": 1, "madisonian": 1, "madison": 1, "macy": 1, "macomb": 1, "machiney": 1, "macedonian": 1, "lumped": 1, "lumber": 1, "lugged": 1, "low": 1, "lovejoy's": 1, "louisianians": 1, "losing": 1, "london": 1, "localized": 1, "localize": 1, "lobby": 1, "loath": 1, "litigation": 1, "literature": 1, "lists": 1, "listener": 1, "liquidation": 1, "lincon's": 1, "limiting": 1, "lima": 1, "likewise": 1, "lighten": 1, "lift": 1, "lick": 1, "liberian": 1, "liberating": 1, "liberality": 1, "liable": 1, "liability": 1, "leveled": 1, "letters": 1, "lessening": 1, "lessened": 1, "lessen": 1, "legged": 1, "legation": 1, "legality": 1, "leg": 1, "lebanon": 1, "leaven": 1, "learning": 1, "lawyer's": 1, "lawgiver": 1, "lawfulness": 1, "lash": 1, "landing": 1, "lafourche": 1, "lacks": 1, "lacked": 1, "laboring": 1, "laboratory": 1, "knoxville": 1, "kite": 1, "kinds": 1, "kindest": 1, "killing": 1, "killed": 1, "kill": 1, "kicked": 1, "kelley": 1, "justification": 1, "justifiably": 1, "justices": 1, "jurisprudence": 1, "judiciary": 1, "judicially": 1, "jostle": 1, "jonathan": 1, "johnson": 1, "johns": 1, "joel": 1, "jeopardy": 1, "jehu": 1, "jeffersonian": 1, "jarred": 1, "jake": 1, "jackson's": 1, "items": 1, "italy": 1, "it.possibly": 1, "isolated": 1, "isn't": 1, "isles": 1, "islands": 1, "isaac": 1, "irruption": 1, "irrevocably": 1, "irrevocable": 1, "irrespective": 1, "ironclad": 1, "iota": 1, "ions": 1, "involves": 1, "involuntary": 1, "invoking": 1, "invocations": 1, "invites": 1, "invitation": 1, "inviolate": 1, "inviolable": 1, "investing": 1, "investigated": 1, "invent": 1, "invariable": 1, "introductory": 1, "intrinsic": 1, "intricacy": 1, "interview": 1, "intervening": 1, "interruptions": 1, "interruption": 1, "interrogated": 1, "interpreted": 1, "interposed": 1, "intermingled": 1, "intermediate": 1, "inter": 1, "intentions": 1, "intentionally": 1, "intends": 1, "intellect": 1, "insurrectionist": 1, "insulting": 1, "insubordination": 1, "instrumentality": 1, "instrumental": 1, "instruct": 1, "instituting": 1, "instinctive": 1, "instinct": 1, "instant": 1, "installed": 1, "inspiring": 1, "inspection": 1, "inspect": 1, "insomuch": 1, "insignificance": 1, "insidious": 1, "inserting": 1, "insecurity": 1, "inquiring": 1, "innovation": 1, "injuriously": 1, "initial": 1, "iniquitous": 1, "inimical": 1, "inhibited": 1, "inhabited": 1, "infringement": 1, "infringed": 1, "informs": 1, "informal": 1, "influenced": 1, "inflicting": 1, "infers": 1, "inexplicable": 1, "inexpedient": 1, "inequality": 1, "industriously": 1, "indulgence": 1, "induces": 1, "indorsing": 1, "indorses": 1, "indorsements": 1, "indiscriminately": 1, "indifferently": 1, "indicted": 1, "indemnification": 1, "indebted": 1, "incursions": 1, "incumbent's": 1, "incorporating": 1, "incorporate": 1, "inconveniences": 1, "inconvenience": 1, "inconstant": 1, "inconsiderately": 1, "inconsiderable": 1, "incomes": 1, "inclosures": 1, "incidentally": 1, "incapable": 1, "inaugurated": 1, "inadmissible": 1, "inadequacy": 1, "inactive": 1, "impute": 1, "imputation": 1, "impunity": 1, "impugning": 1, "impudently": 1, "impropriety": 1, "improbably": 1, "imprisonment": 1, "impressively": 1, "impressive": 1, "impressions": 1, "impressing": 1, "impress": 1, "imposing": 1, "imposes": 1, "imported": 1, "import": 1, "implying": 1, "imploring": 1, "implies": 1, "impliedly": 1, "implicate": 1, "imperial": 1, "imperatively": 1, "impending": 1, "impeded": 1, "impeaches": 1, "impatience": 1, "impartiality": 1, "impartial": 1, "impair": 1, "immoral": 1, "imbroglio": 1, "imagine": 1, "imaginary": 1, "illicit": 1, "illiberally": 1, "illiberal": 1, "illegal": 1, "ignore": 1, "identity": 1, "ideal": 1, "hypocritical": 1, "hypercritical": 1, "hurried": 1, "hurraed": 1, "hurra": 1, "hunted": 1, "hunt": 1, "hungry": 1, "humbuggery": 1, "hugh": 1, "huge": 1, "hudsons": 1, "hoyne": 1, "hour's": 1, "horsman": 1, "horrid": 1, "hopeless": 1, "hopefully": 1, "hopeful": 1, "hoosier": 1, "hoops": 1, "hoop": 1, "honorably": 1, "honduras": 1, "homesteads": 1, "homeopathic": 1, "hollow": 1, "holden": 1, "hitchcock": 1, "historical": 1, "hise": 1, "hires": 1, "hirers": 1, "hireling": 1, "hinted": 1, "hint": 1, "highwayman": 1, "hibernian": 1, "hertofore": 1, "herrington": 1, "heroic": 1, "hereunto": 1, "heretical": 1, "hereinbefore": 1, "hereinafter": 1, "hereditary": 1, "henceforward": 1, "helpless": 1, "helper's": 1, "heated": 1, "hearty": 1, "hearthstone": 1, "hears": 1, "hazards": 1, "hawking": 1, "hauled": 1, "hatteras": 1, "hats": 1, "hasty": 1, "harpers": 1, "harmless": 1, "harbors": 1, "harbored": 1, "harangue": 1, "happily": 1, "hans": 1, "hangs": 1, "handle": 1, "handed": 1, "handbills": 1, "hampshire": 1, "hamilton": 1, "halls": 1, "hallow": 1, "hallooed": 1, "hail": 1, "hackneyed": 1, "habitual": 1, "habits": 1, "gur": 1, "gunboat": 1, "gun": 1, "gulf": 1, "guise": 1, "guile": 1, "guides": 1, "guaranteed": 1, "grown": 1, "groping": 1, "grievances": 1, "grievance": 1, "greatness": 1, "gravity": 1, "gravest": 1, "graves": 1, "gratified": 1, "grasses": 1, "grasped": 1, "grapple": 1, "granada": 1, "grains": 1, "grades": 1, "grade": 1, "gracious": 1, "grab": 1, "gouverneur": 1, "gosport": 1, "goods": 1, "globes": 1, "glee": 1, "gingerbread": 1, "gesticulated": 1, "geometry": 1, "genuineness": 1, "generality": 1, "gathered": 1, "garbling": 1, "garbled": 1, "gap": 1, "gainsay": 1, "gaining": 1, "furious": 1, "fund": 1, "fullest": 1, "fulfilled": 1, "fuel": 1, "frontiers": 1, "frightened": 1, "frighten": 1, "friday": 1, "freesoilers": 1, "freedmen": 1, "free.i": 1, "fraudulently": 1, "fraudulent": 1, "frauds": 1, "framework": 1, "foundry": 1, "fostered": 1, "fortyeight": 1, "fortifications": 1, "forthwith": 1, "forthcoming": 1, "forsaken": 1, "forsake": 1, "formula": 1, "formidably": 1, "formidable": 1, "formality": 1, "forfeitures": 1, "foretell": 1, "foresight": 1, "foreshadowings": 1, "foreseen": 1, "foreigner": 1, "forego": 1, "ford": 1, "forcing": 1, "forcible": 1, "foolish": 1, "fondly": 1, "fondest": 1, "fond": 1, "follies": 1, "fog": 1, "fluid": 1, "flowing": 1, "flow": 1, "flimsy": 1, "flee": 1, "fled": 1, "flaunt": 1, "flattery": 1, "flatter": 1, "flagrant": 1, "fixing": 1, "fisherman's": 1, "fish": 1, "firmess": 1, "fillibustering": 1, "filed": 1, "fiery": 1, "fide": 1, "fertile": 1, "ferocity": 1, "fernandina": 1, "females": 1, "fellowship": 1, "fellows": 1, "felicitations": 1, "felicitation": 1, "felicitates": 1, "fees": 1, "feed": 1, "feat": 1, "fearlessly": 1, "fayette": 1, "favorite": 1, "favorers": 1, "fashioned": 1, "farnsworth": 1, "farms": 1, "farcical": 1, "fantastically": 1, "fantastical": 1, "fantastic": 1, "fancies": 1, "families": 1, "falter": 1, "falsifying": 1, "falsify": 1, "falsely": 1, "falsehoods": 1, "fallacies": 1, "fails": 1, "failing": 1, "factious": 1, "factions": 1, "faction": 1, "facilitating": 1, "facilitated": 1, "fabrication": 1, "eyes": 1, "extravagance": 1, "extraneous": 1, "extorted": 1, "extinguishing": 1, "expressions": 1, "expounder": 1, "exposition": 1, "exposed": 1, "exploration": 1, "explanation": 1, "explained": 1, "expire": 1, "expense": 1, "expeditions": 1, "expedite": 1, "expectancy": 1, "expansion": 1, "expanded": 1, "exigency": 1, "exhorted": 1, "exhortation": 1, "exhibition": 1, "exhausting": 1, "exertions": 1, "exerting": 1, "exempts": 1, "exemption": 1, "exempted": 1, "execute": 1, "excusing": 1, "excitement": 1, "excite": 1, "exceptionally": 1, "excellence": 1, "exceeds": 1, "exceed": 1, "exasperated": 1, "examines": 1, "exaggerate": 1, "evince": 1, "eventually": 1, "evading": 1, "evaded": 1, "evacuation": 1, "estoppel": 1, "estimating": 1, "estimable": 1, "esteemed": 1, "esteem": 1, "establishments": 1, "esq": 1, "espoused": 1, "escapes": 1, "erroneously": 1, "equitable": 1, "equipping": 1, "equaled": 1, "epithets": 1, "ephemeral": 1, "enunciated": 1, "enumerate": 1, "enthusiast": 1, "enters": 1, "ensue": 1, "enrolling": 1, "enjoyment": 1, "enjoying": 1, "enjoyed": 1, "enjoins": 1, "enjoined": 1, "enjoin": 1, "enhancing": 1, "enhanced": 1, "engrosses": 1, "engraft": 1, "engineered": 1, "engineer": 1, "energetic": 1, "endures": 1, "endurance": 1, "endowments": 1, "endless": 1, "endeavors": 1, "endeavored": 1, "endeavor": 1, "encumber": 1, "encouraged": 1, "encounter": 1, "encircling": 1, "enacts": 1, "emoluments": 1, "emissaries": 1, "emigrate": 1, "emergency": 1, "embroil": 1, "embodying": 1, "embody": 1, "embodied": 1, "embarrassing": 1, "embarrass": 1, "emancipate": 1, "emanates": 1, "elizabeth": 1, "eliminates": 1, "elijah": 1, "elicit": 1, "elevated": 1, "elevate": 1, "electioneering": 1, "elderly": 1, "elaborate": 1, "effectual": 1, "effectively": 1, "educate": 1, "editors": 1, "editorial": 1, "edited": 1, "edges": 1, "economically": 1, "echoed": 1, "echo": 1, "easiest": 1, "earnings": 1, "earned": 1, "earn": 1, "eagerness": 1, "dwindled": 1, "dwelling": 1, "dungeons": 1, "drugging": 1, "drives": 1, "dreaming": 1, "drayman's": 1, "draws": 1, "drawing": 1, "drawback": 1, "drama": 1, "dragged": 1, "drafted": 1, "doubts": 1, "donnelly": 1, "dominions": 1, "domingo": 1, "dominating": 1, "dominated": 1, "domicile": 1, "dolores": 1, "dog": 1, "dodging": 1, "dockyards": 1, "dixon's": 1, "diverted": 1, "diversion": 1, "ditto": 1, "disturbances": 1, "distrust": 1, "distribute": 1, "distinctness": 1, "distances": 1, "dissolve": 1, "dissevered": 1, "disruption": 1, "disrupting": 1, "disrespectful": 1, "disregards": 1, "disquiet": 1, "disputed": 1, "disproving": 1, "disproportionate": 1, "dispositions": 1, "disposing": 1, "displeased": 1, "displayed": 1, "display": 1, "disperse": 1, "dispense": 1, "dispatched": 1, "dismember": 1, "disloyalty": 1, "disintegration": 1, "dishonorable": 1, "dishonor": 1, "disgusting": 1, "disgust": 1, "disguise": 1, "disgraced": 1, "disenthralled": 1, "disenthrall": 1, "disease": 1, "discrepancy": 1, "discovers": 1, "discounts": 1, "discordant": 1, "discontented": 1, "disclaim": 1, "discharge": 1, "disbursement": 1, "disbursed": 1, "disavowed": 1, "disapprobation": 1, "disappearance": 1, "disadvantages": 1, "disabling": 1, "disabled": 1, "disabilities": 1, "directness": 1, "dinners": 1, "din": 1, "diminish": 1, "diligent": 1, "dignitaries": 1, "dignified": 1, "diffused": 1, "dictates": 1, "dictated": 1, "devotes": 1, "devolving": 1, "devolved": 1, "devise": 1, "develops": 1, "develop": 1, "detract": 1, "detected": 1, "detect": 1, "detained": 1, "detaching": 1, "detach": 1, "destructiveness": 1, "despite": 1, "despised": 1, "desperately": 1, "designedly": 1, "designating": 1, "designates": 1, "deserves": 1, "deserts": 1, "deserter": 1, "description": 1, "descending": 1, "descendant": 1, "derogation": 1, "deputies": 1, "depriving": 1, "depressing": 1, "depredations": 1, "deprecate": 1, "depots": 1, "depot": 1, "deposited": 1, "deported": 1, "depopulated": 1, "departures": 1, "departing": 1, "depart": 1, "denunciation": 1, "dense": 1, "demonstrations": 1, "demonstration": 1, "delivering": 1, "delights": 1, "delighted": 1, "delight": 1, "delicacy": 1, "deliberations": 1, "deliberation": 1, "delegations": 1, "delaying": 1, "dehumanize": 1, "degenerate": 1, "definitive": 1, "defines": 1, "define": 1, "deficiencies": 1, "defenses": 1, "defence": 1, "defective": 1, "defection": 1, "deepening": 1, "deduct": 1, "deduced": 1, "decrepit": 1, "decrees": 1, "decoction": 1, "declaratory": 1, "decisive": 1, "deceiving": 1, "decease": 1, "debtor": 1, "debauching": 1, "debates": 1, "debater": 1, "debated": 1, "dearly": 1, "dayton": 1, "davis": 1, "daughters": 1, "dates": 1, "darker": 1, "danville": 1, "daniel": 1, "dangers": 1, "damaged": 1, "daily": 1, "d.l": 1, "d'affaires": 1, "cuttings": 1, "custody": 1, "curiosity": 1, "cumberland": 1, "culture": 1, "culprits": 1, "crushed": 1, "crush": 1, "cruising": 1, "cruelty": 1, "crowded": 1, "crossing": 1, "crossed": 1, "cross": 1, "crops": 1, "crooked": 1, "criticizing": 1, "criticised": 1, "crimes": 1, "creeping": 1, "creditable": 1, "credibility": 1, "crawling": 1, "cozy": 1, "covers": 1, "courteously": 1, "counting": 1, "countervailing": 1, "counter": 1, "counsels": 1, "counsel": 1, "cosily": 1, "corruption": 1, "corroborative": 1, "corresponds": 1, "correlative": 1, "copies": 1, "cooperation": 1, "cooper": 1, "cool": 1, "convinces": 1, "convey": 1, "convertible": 1, "conversions": 1, "conversation": 1, "conventional": 1, "convened": 1, "controlled": 1, "contrived": 1, "contribute": 1, "contrabands": 1, "contraband": 1, "continuous": 1, "continual": 1, "contingency": 1, "contiguous": 1, "contests": 1, "contentions": 1, "contemporary": 1, "contemplating": 1, "consummating": 1, "consummated": 1, "consulting": 1, "consulted": 1, "consulates": 1, "consulate": 1, "constructed": 1, "constraint": 1, "constitutionality": 1, "conspirators": 1, "consolidated": 1, "consolations": 1, "consisting": 1, "considerations": 1, "consecrated": 1, "consecrate": 1, "conscientiously": 1, "conscience": 1, "connects": 1, "connections": 1, "conklingmy": 1, "conjuncture": 1, "conjecture": 1, "congratulation": 1, "confront": 1, "conforming": 1, "conform": 1, "conflicts": 1, "confirmed": 1, "confining": 1, "confidential": 1, "confession": 1, "confessedly": 1, "confers": 1, "conferring": 1, "confab": 1, "concrete": 1, "concourse": 1, "concoction": 1, "concluding": 1, "conciliatory": 1, "conceives": 1, "conceded": 1, "concede": 1, "computed": 1, "computations": 1, "compressed": 1, "comprehensive": 1, "compounded": 1, "complimenting": 1, "compliment": 1, "complied": 1, "complex": 1, "completing": 1, "complaining": 1, "competency": 1, "compensate": 1, "compel": 1, "comparative": 1, "communicated": 1, "communicate": 1, "commonly": 1, "commodity": 1, "committees": 1, "committals": 1, "commissions": 1, "commissioned": 1, "commissary": 1, "commenting": 1, "commands": 1, "comity": 1, "combining": 1, "combated": 1, "columns": 1, "columbus": 1, "colonists": 1, "colonel": 1, "collisions": 1, "cold": 1, "coinciding": 1, "coincident": 1, "cody": 1, "code": 1, "cock": 1, "cob": 1, "coated": 1, "coal": 1, "coadjutors": 1, "clutches": 1, "clung": 1, "clothe": 1, "closes": 1, "clings": 1, "climes": 1, "clever": 1, "clerkship": 1, "clerks": 1, "clemency": 1, "clearness": 1, "clearer": 1, "clean": 1, "clayon": 1, "citizen's": 1, "cities": 1, "cite": 1, "circumstantially": 1, "circles": 1, "cinnabar": 1, "churches": 1, "christianity": 1, "christian": 1, "chorus": 1, "choosing": 1, "chilean": 1, "chiefs": 1, "chiefest": 1, "chesapeake": 1, "cherokees": 1, "chemical": 1, "checks": 1, "checked": 1, "cheat": 1, "cheaper": 1, "chary": 1, "charity": 1, "charitable": 1, "characterize": 1, "chancery": 1, "challenge": 1, "chairman": 1, "certified": 1, "certificates": 1, "cereals": 1, "centuries": 1, "censuring": 1, "cautious": 1, "caution": 1, "category": 1, "catechised": 1, "catechise": 1, "casualties": 1, "casey": 1, "case.before": 1, "carlin's": 1, "caring": 1, "cargoes": 1, "captures": 1, "capture": 1, "capricious": 1, "capitalists": 1, "capacities": 1, "canes": 1, "candidly": 1, "canals": 1, "canal": 1, "canadian": 1, "camps": 1, "campaigns": 1, "camp": 1, "calumniated": 1, "calmness": 1, "calm": 1, "calculations": 1, "caged": 1, "cadets": 1, "bursting": 1, "buried": 1, "bureaus": 1, "builds": 1, "building": 1, "bugaboos": 1, "browns": 1, "brooks's": 1, "broods": 1, "brilliantly": 1, "brilliant": 1, "brigades": 1, "brig": 1, "breeding": 1, "breadth": 1, "brazilian": 1, "branded": 1, "boy": 1, "bounty": 1, "bountiful": 1, "bounded": 1, "bosses": 1, "boon": 1, "bondsman's": 1, "bona": 1, "bombarded": 1, "bolster": 1, "bolivia": 1, "boiling": 1, "bluffing": 1, "blount": 1, "blind": 1, "blast": 1, "bigler's": 1, "bigger": 1, "bid": 1, "betrayal": 1, "bestride": 1, "bespattered": 1, "besieged": 1, "beset": 1, "beseeching": 1, "bernard": 1, "berkley": 1, "berating": 1, "beneficially": 1, "bend": 1, "belongs.now": 1, "believers": 1, "belgians": 1, "belabored": 1, "behrings": 1, "beheve": 1, "beginners": 1, "befitting": 1, "befell": 1, "bed": 1, "beau": 1, "bearings": 1, "be.then": 1, "bayonets": 1, "basing": 1, "barrier": 1, "bargaining": 1, "barefacedness": 1, "barbary": 1, "baptiste": 1, "banished": 1, "band": 1, "baker": 1, "backwards": 1, "awfully": 1, "awards": 1, "award": 1, "awaiting": 1, "avowedly": 1, "avocations": 1, "aves": 1, "averting": 1, "aversion": 1, "averaging": 1, "avenue": 1, "availed": 1, "auxiliary": 1, "austria": 1, "auspicious": 1, "augusta": 1, "augmented.in": 1, "audited": 1, "audiences": 1, "audacity": 1, "attributes": 1, "attorney": 1, "attending": 1, "attendance": 1, "attainment": 1, "attainable": 1, "attacking": 1, "attached": 1, "asylum": 1, "asunder": 1, "assurances": 1, "assumptions": 1, "associates": 1, "assign": 1, "assiduous": 1, "assemble": 1, "assemblages": 1, "assassination": 1, "assailant": 1, "asiatic": 1, "asia": 1, "ashmun's": 1, "ashmun": 1, "ascribes": 1, "ascertained": 1, "ascension": 1, "ascendancy": 1, "artisans": 1, "artificial": 1, "arsenals": 1, "arrays": 1, "arraying": 1, "arranging": 1, "arrangements": 1, "arno": 1, "armory": 1, "armistice": 1, "armature": 1, "armament": 1, "arizona": 1, "aristocracy": 1, "argentine": 1, "arduous": 1, "arbitrament": 1, "aqueduct": 1, "aptitudes": 1, "aptest": 1, "approximated": 1, "approaches": 1, "apprehended": 1, "appreciation": 1, "apposite": 1, "apportioned": 1, "appointing": 1, "applying": 1, "applicable": 1, "appliances": 1, "apple": 1, "applaud": 1, "appended": 1, "appearing": 1, "appeals": 1, "apiece": 1, "anxiously": 1, "anticipations": 1, "anticipation": 1, "antecedent": 1, "annunciation": 1, "annuity": 1, "announcement": 1, "announced": 1, "annihilate": 1, "ann": 1, "animal": 1, "anderson's": 1, "analyze": 1, "amuse": 1, "amounting": 1, "amnesty": 1, "ammunition": 1, "amid": 1, "amicably": 1, "ambitious": 1, "ambitions": 1, "alternate": 1, "altering": 1, "altercations": 1, "altercation": 1, "allows": 1, "allotted": 1, "alleging": 1, "alleghanies": 1, "allege": 1, "allaying": 1, "allay": 1, "alienage": 1, "alexander": 1, "alarms": 1, "alarm": 1, "ain't": 1, "aim": 1, "aids": 1, "ahead": 1, "agravate": 1, "aggression": 1, "aggravate": 1, "agency": 1, "africans": 1, "afloat": 1, "affords": 1, "affording": 1, "affliction": 1, "affixed": 1, "affirmations": 1, "affections": 1, "affectionate": 1, "affecting": 1, "advertisement": 1, "adventitious": 1, "advantageously": 1, "advances": 1, "admiralty": 1, "adjusting": 1, "adjusted": 1, "adjudications": 1, "adjudicate": 1, "adhere": 1, "adequate": 1, "adducing": 1, "adds": 1, "adapt": 1, "actively": 1, "acknowledgment": 1, "acknowledges": 1, "achieve": 1, "accusations": 1, "accurately": 1, "accumulations": 1, "accumulates": 1, "accounts": 1, "accounted": 1, "accomplishment": 1, "accomplishing": 1, "accomplished": 1, "accomplish": 1, "accompanied": 1, "accommodations": 1, "accommodation": 1, "acclamation": 1, "accidental": 1, "accession": 1, "accepts": 1, "acceptable": 1, "accents": 1, "accelerate": 1, "abstracts": 1, "abstaining": 1, "abstain": 1, "absorbs": 1, "absorbed": 1, "absence": 1, "abrogation": 1, "abrogating": 1, "abridgment": 1, "abridged": 1, "aboriginal": 1, "abolitionizing": 1, "ably": 1, "abideth": 1, "abhors": 1, "abhorrence": 1, "abandonment": 1, "985.52": 1, "98,239,733.09": 1, "96,739,905.73": 1, "931,787.64": 1, "90,000": 1, "9,638,131": 1, "9,383,000": 1, "9,049,296.40": 1, "8th": 1, "884,076,646.57": 1, "86,835,900.27": 1, "85,733,292.77": 1, "84,578,834.47": 1, "800": 1, "8,683,000": 1, "8,349,296.40": 1, "8,299,820.90": 1, "793": 1, "776,682,361.57": 1, "76,677,872": 1, "75,000,000": 1, "75,000": 1, "720,039,039.79": 1, "714,709,995.58": 1, "712": 1, "7,517,930.97": 1, "7,500": 1, "7,239,814": 1, "690,791,842.97": 1, "69,059,642.40": 1, "678,007.21": 1, "671": 1, "63,211,105.27": 1, "623,443,929.13": 1, "600,000,000": 1, "6,151,405": 1, "6,000": 1, "599,298,600.83": 1, "588,333.29": 1, "588": 1, "584": 1, "583,885,247.06": 1, "570,841,700.25": 1, "56,967,216": 1, "53,685,421.69": 1, "529,692,460.50": 1, "52,257,065.80": 1, "510,396": 1, "51,000": 1, "50,000,000": 1, "50,000": 1, "5,939.009.29": 1, "5,656,705.49": 1, "5,304,937": 1, "5,000": 1, "49,056,397.62": 1, "487,788,324.97": 1, "475,648.96": 1, "474,744,778.16": 1, "47,511,448.10": 1, "42,674,569.69": 1, "42,427": 1, "42,323,341": 1, "400,000,000": 1, "400,000": 1, "40,000": 1, "4,610": 1, "4,557,462.71": 1, "4,551,966.98": 1, "4,504,616.92": 1, "4,292,776.18": 1, "4,221,342": 1, "4,216,520.79": 1, "4,015,773": 1, "394,368,407.36": 1, "37,640,787.95": 1, "35.87": 1, "35.58": 1, "35.02": 1, "34.60": 1, "34,000": 1, "33.49": 1, "33,762": 1, "324": 1, "32.67": 1, "31,443,790": 1, "31,000,000": 1, "300": 1, "3,995,097.31": 1, "3,929,827": 1, "3,870,222": 1, "3,841,549": 1, "3,800,000": 1, "3,750,000": 1, "3,145,000": 1, "3,046,615.35": 1, "3,000": 1, "271": 1, "27,505,599.46": 1, "267": 1, "251,680,914": 1, "250,000,000": 1, "25,433": 1, "248": 1, "24,729,846.51": 1, "238,647,262.35": 1, "23,253,922.08": 1, "23,191,876": 1, "22,767": 1, "22,198": 1, "217,186,000": 1, "206,652.42": 1, "200,000,000": 1, "200,000": 1, "20,000": 1, "2,963,000": 1, "2,645,722.19": 1, "2,481,000": 1, "2,257,065.80": 1, "2,112,814.57": 1, "1925": 1, "1910": 1, "1890": 1, "1880": 1, "1870": 1, "1865": 1, "186,984,335": 1, "1859": 1, "1855": 1, "1840": 1, "1778": 1, "1774": 1, "17,069,453": 1, "167,617.17": 1, "167": 1, "161,911": 1, "16,770": 1, "157": 1, "152,203.77": 1, "150,417.25": 1, "145,551": 1, "14,396,250.51": 1, "14,129,771.50": 1, "13th": 1, "138,918,526": 1, "137,476.26": 1, "136,077.95": 1, "133,000,000": 1, "133": 1, "13,190,324.45": 1, "13,043,546.81": 1, "13,000,000": 1, "12,866,020": 1, "12,644,786.20": 1, "12,528,000": 1, "12,438,253.78": 1, "12,000": 1, "11th": 1, "113": 1, "11,314,206.84": 1, "11,163,789.59": 1, "11,125,364.13": 1, "109,741,134.10": 1, "102,532,509.27": 1, "102,316,152.99": 1, "100,000,000": 1, "10,000,000": 1, "1,795,331.73": 1, "1,740,690,489.49": 1, "1,538,614": 1, "1,500": 1, "1,485,103.61": 1, "1,470,018": 1, "1,430": 1, "1,394,796,007.62": 1, "1,379": 1, "1,339,710.35": 1, "1,298,056,101.89": 1, "1,019,446": 1, "1,000,000": 1, "1,000": 1}

# ── Frequency thresholds (calibrated to this 63k-word corpus) ──────────
# RARE   : rawFreq 1–14   → ideal discriminator; weight 15–20
# MODERATE: rawFreq 15–50  → acceptable; weight 8–14
# COMMON  : rawFreq >50    → broad term; keep weight ≤ 8
# ABSENT  : rawFreq 0      → not in corpus; flag for review

def term_freq(term):
    return VOYANT_FREQ.get(term.lower(), 0)

def classify_keyword(term, weight):
    freq = term_freq(term)
    if freq == 0:                        return "ABSENT  ⚠"
    if freq <= 14 and weight >= 15:      return "STRONG  ✓"
    if freq <= 50 and weight >= 8:       return "OK      ✓"
    if freq > 50  and weight >= 15:      return "WEAK    ⚠"
    return "OK      ✓"

# Quick sanity-check on representative terms
checks = [
    ("Harper",          20), ("Ferry",         20), ("Brown",         20),
    ("colonization",    13), ("jurisprudence",   1), ("perpetuity",     8),
    ("suffrage",         2), ("negro",          69), ("liberty",       69),
    ("union",          334), ("slavery",        636),
]
print(f"TERM                     FREQ  WEIGHT  QUALITY")
print("-"*52)
for term, w in checks:
    f = term_freq(term)
    q = classify_keyword(term, w)
    print(f"  {term:<20} {f:>6}  {w:>6}  {q}")


In [ ]:
# 0.3  Load corpus and Nicolay dataset
def load_json(path):
    with open(path) as f: return json.load(f)

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(l) for l in f if l.strip()]

corpus     = load_json("lincoln_speech_corpus_repaired.json")
nicolay_ft = load_jsonl("nicolay_finetuning_data.jsonl")

# Corpus lookups
corpus_by_id    = {item["text_id"]: item for item in corpus}
adjacency_index = {
    (item["source_file"], item["position"]): item
    for item in corpus
    if item.get("source_file") is not None and item.get("position") is not None
}

def get_chunk(text_id):
    return corpus_by_id.get(text_id)

def get_neighbor_context(item, window=1):
    sf, pos, total = item.get("source_file"), item.get("position"), item.get("total_in_source")
    if sf is None or pos is None:
        return item.get("full_text","")
    parts = []
    for offset in range(-window, window+1):
        np_ = pos + offset
        if np_ < 0 or (total is not None and np_ >= total): continue
        neighbor = adjacency_index.get((sf, np_))
        if neighbor: parts.append(neighbor.get("full_text",""))
    return "\n\n".join(filter(None, parts))

print(f"Corpus chunks : {{len(corpus)}}")
print(f"Nicolay FT    : {{len(nicolay_ft)}} examples")


In [ ]:
# 0.4  Hay dataset (pre-loaded and schema-fixed)
# All 51 examples validated; 4 schema bugs repaired:
#   Line 1  : weighted_keywords_japan → weighted_keywords
#   Line 2  : duplicate 'Consecrate' key removed; text_keywords quoted
#   Line 9  : trailing comma in weighted_keywords removed
#   Line 33 : missing comma after year_keywords added
#
# NOTE ON EMBEDDED DATA:
# The full Hay fine-tuning dataset (51 examples) is embedded directly below as a
# Python literal rather than loaded from an external JSONL file. This design choice
# makes the notebook self-contained for peer review and reproducibility — readers
# can inspect the complete training data without needing access to separate files.
# The schema shown here is the Hay output format: initial_answer (HyDE hypothesis),
# weighted_keywords (corpus-rare terms with retrieval weights), year_keywords
# (temporal discriminators), and text_keywords (speech-specific surface terms).

HAY_FT = [{"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How does Liberia factor into Lincoln’s speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln may have referenced Liberia in his speeches in the context of discussing the colonization of freed African Americans. Liberia, a West African country, was established by the American Colonization Society as a destination for free African Americans and formerly enslaved people. Lincoln's views on colonization and the idea of sending African Americans to Liberia could have been a topic in some of his speeches, especially considering the debates around slavery and race relations during his presidency.\", \"weighted_keywords\": {\"Liberia\": 20, \"Colonization\": 8, \"Emancipation\": 7, \"Slavery\": 6, \"Repatriation\": 5}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was the relationship between Japan and the U.S. during the Civil War, and what can Lincoln's speeches tell us about this relationship?"}, {"role": "assistant", "content": "{\"initial_answer\": \"During the American Civil War, the U.S. was largely focused on internal conflict, but it also continued to engage with foreign nations, including Japan. The relationship was relatively new, as Commodore Matthew Perry had only recently opened Japan to the West with the Kanagawa Treaty of 1854. As such, any mentions of Japan in Lincoln\\u2019s speeches would likely have pertained to diplomacy or trade as opposed to direct involvement in the Civil War. It\\u2019s possible that Lincoln may have referenced Japan in the context of broader discussions about international relations or the importance of maintaining and establishing ties with foreign nations during a period when national unity was a critical issue.\", \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null, \"weighted_keywords\": {\"Japan\": 10, \"Diplomacy\": 8, \"Trade\": 7, \"Treaty\": 6, \"Foreign relations\": 5, \"Unity\": 4}}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln use rhetorical devices in his Gettysburg Address?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In his Gettysburg Address, Abraham Lincoln employed a range of rhetorical devices with notable effectiveness. The speech is characterized by its conciseness and brevity, making it both short and impactful. Lincoln skillfully used repetition for emphasis, as seen in the phrase 'we cannot dedicate, we cannot consecrate, we cannot hallow,' which not only underscores his point but also adds a rhythmic quality. Parallelism is another device used, enhancing the speech's poetic nature and reinforcing ideas. Lincoln's allusion to the Declaration of Independence, particularly through the phrase 'all men are created equal,' subtly reminds the audience of America's foundational principles. He also used antithesis, juxtaposing contrasting ideas for memorable impact, as in 'The world will little note, nor long remember what we say here, but it can never forget what they did here.' The speech is rich in pathos, aiming to unite and inspire a grieving nation.\", \"weighted_keywords\": {\"Dedicate\": 10, \"Consecrate\": 10, \"Hallow\": 10, \"Proposition\": 10}, \"year_keywords\": [1863], \"text_keywords\": [\"Gettysburg Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln justify the Civil War in his speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's two inaugural addresses reflect a significant shift in tone and focus, mirroring the evolving circumstances of his presidency. In his First Inaugural Address, Lincoln's tone was conciliatory, aimed at averting the impending crisis of the Civil War. He focused on the legality of secession, the preservation of the Union, and reassured the Southern states that he had no intention to abolish slavery where it existed. By his Second Inaugural Address, the tone had shifted to one of somber reflection and moral introspection. The focus was on the underlying causes of the Civil War, particularly slavery, and the necessity of seeing the war through to preserve the Union. Lincoln also expressed a desire for reconciliation and healing, emphasizing 'malice toward none, with charity for all.'\", \"weighted_keywords\": {\"Union\": 20, \"Secession\": 15, \"Slavery\": 10, \"Reconciliation\": 8, \"Legality\": 7, \"Morality\": 6}, \"year_keywords\": [1861, 1865], \"text_keywords\": [\"First Inaugural Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Compare Lincoln's inaugural addresses. How did his tone and focus change?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's two inaugural addresses reflect a significant shift in tone and focus, mirroring the evolving circumstances of his presidency. In his First Inaugural Address, Lincoln's tone was conciliatory, aimed at averting the impending crisis of the Civil War. He focused on the legality of secession, the preservation of the Union, and reassured the Southern states that he had no intention to abolish slavery where it existed. By his Second Inaugural Address, the tone had shifted to one of somber reflection and moral introspection. The focus was on the underlying causes of the Civil War, particularly slavery, and the necessity of seeing the war through to preserve the Union. Lincoln also expressed a desire for reconciliation and healing, emphasizing 'malice toward none, with charity for all.'\", \"weighted_keywords\": {\"Union\": 20, \"Secession\": 15, \"Slavery\": 10, \"Reconciliation\": 8, \"Legality\": 7, \"Morality\": 6}, \"year_keywords\": [1861, 1865], \"text_keywords\": [\"First Inaugural Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Identify an example of pathos in Lincoln's speeches."}, {"role": "assistant", "content": "{\"initial_answer\": \"An example of pathos in Abraham Lincoln's speeches is evident in his Gettysburg Address. In this speech, Lincoln appeals to the emotions of his audience by commemorating the sacrifices of those who fought in the Battle of Gettysburg. He invokes a sense of national grief and honor, highlighting the tragedy of the Civil War while also inspiring hope for the nation's future. Lincoln's use of emotive language and his reference to the principles of liberty and equality for which the soldiers sacrificed their lives are aimed at stirring the audience's emotions, fostering a sense of unity and purpose in the midst of war.\", \"weighted_keywords\": {\"Gettysburg\": 20, \"Sacrifice\": 15, \"Grief\": 10, \"Honor\": 8, \"Liberty\": 7, \"Equality\": 6}, \"year_keywords\": [1863], \"text_keywords\": [\"Gettysburg Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln use ethos and logos in his arguments?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln used ethos and logos effectively in his arguments. For ethos, his credibility and moral authority were emphasized through references to the Constitution, American founding principles, and his commitment to democracy and equality. His First and Second Inaugural Addresses and the Gettysburg Address are prime examples where he demonstrated these ethical appeals. As for logos, Lincoln's logical reasoning was evident in his structured arguments, legal justifications, and historical context. His Cooper Union Address and debates with Stephen Douglas showcase his logical argumentation, focusing on constitutional law and political analysis.\", \"weighted_keywords\": {\"Constitution\": 20, \"Democracy\": 15, \"Equality\": 10, \"Legal\": 8, \"History\": 7, \"Reasoning\": 6}, \"year_keywords\": null, \"text_keywords\": [\"First Inaugural Address\", \"Second Inaugural Address\", \"Gettysburg Address\", \"Cooper Union Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln's speeches address the issue of national unity?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's speeches frequently addressed the issue of national unity, particularly in the context of the Civil War. He emphasized the importance of preserving the Union and the principles upon which it was founded. In his First Inaugural Address, Lincoln appealed directly to the seceded states, urging them to remain part of the Union and emphasizing the shared history and values that bound the nation together. His Gettysburg Address is another key example, where he spoke of a 'new birth of freedom' to unite the nation under the values of liberty and equality. Throughout his presidency, Lincoln consistently called for reconciliation and unity, seeing the Civil War not only as a conflict to be won but as an opportunity to bring the nation closer to its founding ideals.\", \"weighted_keywords\": {\"Union\": 20, \"Reconciliation\": 15, \"Liberty\": 10, \"Equality\": 8, \"Freedom\": 7, \"Secession\": 6}, \"year_keywords\": [1861, 1863], \"text_keywords\": [\"First Inaugural Address\", \"Gettysburg Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What does Lincoln's first inaugural address reveal about his hopes for the nation?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In his first inaugural address, Abraham Lincoln expressed a deep hope for preserving the Union and averting the impending crisis of the Civil War. He articulated a vision of national unity, emphasizing the shared history and common values that bound the American people together. Lincoln sought to reassure the Southern states that he had no intention to interfere with slavery where it existed, reflecting his hope for peace and reconciliation. He also stressed the importance of the rule of law and the constitutional process, highlighting his belief in a government of the people, by the people, and for the people. Overall, his address conveyed a message of hope for a resolution to the nation's divisions and a commitment to maintaining the Union at all costs.\", \"weighted_keywords\": {\"Hope\": 20, \"Union\": 18, \"Reconciliation\": 15, \"Slavery\": 10, \"Constitution\": 8, \"Peace\": 7, \"Law\": 6, \"Government\": 5}, \"year_keywords\": [1861], \"text_keywords\": [\"First Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln express his personal struggles and emotions in his speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In his speeches, Abraham Lincoln often expressed his personal struggles and emotions, particularly in relation to the Civil War and the divided state of the nation. In the Gettysburg Address, he reflects on the immense human cost of the war, expressing grief and the hope that the dead shall not have died in vain. His Second Inaugural Address is marked by a tone of reconciliation and moral reflection, stressing 'malice toward none' and 'charity for all'. The House Divided Speech reveals his concern about the growing divisions within the country, prophetically warning that 'a house divided against itself cannot stand.' These speeches collectively showcase Lincoln's ability to convey deep emotional and personal reflections amidst national turmoil.\", \"weighted_keywords\": {\"consecrated\": 10, \"charity\": 8, \"divided\": 7, \"malice\": 6, \"stand\": 5}, \"year_keywords\": [1858, 1863, 1865], \"text_keywords\": [\"House Divided Speech\", \"Gettysburg Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln express his vision for America's future in his speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's vision for America's future was eloquently expressed in several key speeches. In the Gettysburg Address, he spoke of ensuring that the nation 'shall not perish from the earth,' emphasizing the enduring nature of democracy. His Second Inaugural Address called for the nation to 'bind up the nation's wounds' in a time of healing and reconciliation. The House Divided Speech, with its stark warning that the nation could not 'cease' to be divided, conveyed a vision of a country that must 'endure' as united.\", \"weighted_keywords\": {\"perish\": 10, \"bind\": 8, \"wounds\": 7, \"cease\": 6, \"endure\": 5}, \"year_keywords\": [1858, 1863, 1865], \"text_keywords\": [\"House Divided Speech\", \"Gettysburg Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Discuss Lincoln’s use of allusion in his speeches."}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln effectively used allusion in his speeches to reinforce his messages. In the Gettysburg Address, he alluded to the principles of 'liberty' as enshrined in the Declaration of Independence. His Second Inaugural Address is noted for its biblical allusions, often invoking 'God' to emphasize divine providence. The 'house' in his House Divided Speech is a biblical reference, highlighting the nation's internal conflict. In his First Inaugural Address, Lincoln alluded to 'constitution' principles, reaffirming his commitment to American founding ideals. The Speech at Peoria included allusions to the 'compromise' of the Missouri Compromise, referencing a critical moment in American history.\", \"weighted_keywords\": {\"liberty\": 10, \"God\": 9, \"house\": 8, \"constitution\": 7, \"compromise\": 6}, \"year_keywords\": [1854, 1858, 1861, 1863], \"text_keywords\": [\"At Peoria\", \"House Divided Speech\", \"First Inaugural Address\", \"Gettysburg Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What historical events influenced the content of Lincoln's speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Several historical events significantly influenced the content of Abraham Lincoln's speeches. The American Civil War was a central theme, often referred to as 'war' in his addresses. The 'slavery' issue, heightened by the Kansas-Nebraska Act, featured prominently in his earlier speeches. The Dred Scott Decision's impact on 'citizenship' was another critical topic. Lincoln's own 'election' in 1860 and the subsequent secession of Southern states shaped much of his presidential rhetoric. Additionally, the 'emancipation' of slaves following his famous proclamation was a key subject in his later speeches.\", \"weighted_keywords\": {\"war\": 10, \"slavery\": 9, \"citizenship\": 8, \"election\": 7, \"emancipation\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Discuss the evolution of Lincoln's speech style throughout his presidency."}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's speech style evolved throughout his presidency, marked by distinct phases and topics. In his First Inaugural Address, he addressed the nation's crisis with a focus on 'secession'. The Emancipation Proclamation marked a shift to a more decisive tone, prominently featuring the term 'proclimation'. The Gettysburg Address demonstrated a concise and poetic style, embodied in the word 'dedicate'. By his Second Inaugural Address, Lincoln had adopted a more reflective and reconciliatory tone, as suggested by the term 'bind'.\", \"weighted_keywords\": {\"secession\": 10, \"freedom\": 9, \"dedicate\": 8, \"bind\": 7, \"proclamation\": 6}, \"year_keywords\": [1861, 1863, 1865], \"text_keywords\": [\"First Inaugural Address\", \"Emancipation Proclamation\", \"Gettysburg Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln's views on the Missouri Compromise reflect his broader political philosophy?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's views on the Missouri Compromise significantly influenced his broader political philosophy. 'Missouri' and the 'compromise' are key terms reflecting his direct engagement with this legislation. The term 'slavery' is central to understanding his stance on the expansion of slavery. 'Division', as an alternative to 'union', captures the divisive nature of slavery in the nation. The term 'federal' highlights his interpretation of federal versus state authority, an alternative to the high-frequency 'constitution'. Lincoln's 'morality' on slavery evolved in response to these political issues, indicating a complex and developing philosophical stance.\", \"weighted_keywords\": {\"Missouri\": 10, \"compromise\": 9, \"slavery\": 8, \"division\": 7, \"federal\": 6, \"morality\": 5}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What were Lincoln’s views on gradual emancipation and its feasibility?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln\\u2019s views on gradual emancipation and its feasibility reflect a complex and evolving stance. The concept of 'gradual' emancipation was key in his approach, considering a slow process of ending slavery. 'Emancipation' itself was a central theme, with Lincoln weighing its broader implications. He also pondered 'compensation' for slave owners, an element of the gradual emancipation plan. 'Colonization' was another aspect of his early views, suggesting resettlement for freed slaves. Additionally, the 'practical' aspects, including legal and logistical issues, were significant in his contemplation of emancipation's feasibility.\", \"weighted_keywords\": {\"gradual\": 10, \"emancipation\": 9, \"compensation\": 8, \"colonization\": 7, \"practical\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln justify the challenges of reinforcing and resupplying Fort Sumter?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's justification for the challenges of reinforcing and resupplying Fort Sumter revolved around pivotal themes. Using terms like 'Fort' and 'Sumter', he addressed the specific military situation. His speeches and communications often included language about the 'crisis' at the fort, the need for 'defense', and the federal 'authority' over such installations. The year 1861 was crucial, as it was when the Fort Sumter crisis unfolded, shaping much of his early presidential decisions.\", \"weighted_keywords\": {\"Fort\": 10, \"Sumter\": 9, \"crisis\": 8, \"defense\": 7, \"authority\": 6}, \"year_keywords\": [1861], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Lincoln's perspective on the Civil War's impact on international affairs?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's views on the Civil War's impact on international affairs focused on several key areas. His approach to 'diplomacy' was pivotal, navigating complex international relations during the conflict. The 'foreign' policy aspects, including how the war influenced America's standing abroad, were of significant concern. The impact on 'trade', especially with the naval blockades, had international repercussions. 'Europe's perspective and potential involvement were carefully monitored. Finally, 'slavery' as an issue had a profound effect on international sentiment, influencing foreign views on the conflict.\", \"weighted_keywords\": {\"diplomacy\": 10, \"foreign\": 9, \"trade\": 8, \"Europe\": 7, \"slavery\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": [\"Annual Message\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln approach the issue of the Dred Scott decision and its potential implications?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's approach to the Dred Scott decision and its implications was multifaceted. He referenced 'Dred' in discussing the specific case, critically analyzing its consequences. Lincoln's focus on the 'constitution' was evident in his legal critique of the decision's interpretation. The issue of 'citizenship' and the rights of African Americans was central to his argument. He also considered the decision's impact on 'federal' versus state authority. Additionally, his moral and ethical stance on 'slavery' was reflected in his response to the decision's reinforcement of slavery.\", \"weighted_keywords\": {\"Dred\": 10, \"constitution\": 9, \"citizenship\": 8, \"federal\": 7, \"slavery\": 6}, \"year_keywords\": [1857, 1858, 1859, 1860], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What strategies did Lincoln propose for the Republican party to unite against the expansion of slavery?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln proposed several strategies for the Republican party to unite against the expansion of slavery. His emphasis on 'slavery' and its moral and political implications was a core part of the party's stance. He used the term 'Republican' to rally the party around a unified approach. His 'strategy' involved a mix of political mobilization and persuasive argumentation. The issue of 'territory' was crucial, focusing on preventing the spread of slavery into new areas. Lincoln also often invoked the 'constitution' in framing the legal basis for opposing slavery's expansion.\", \"weighted_keywords\": {\"slavery\": 10, \"Republican\": 9, \"strategy\": 8, \"territory\": 7, \"constitution\": 6}, \"year_keywords\": [1858, 1859, 1860], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln argue about the founders' understanding of the Constitution regarding federal control of slavery in the territories?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's arguments about the founders' understanding of the Constitution in relation to federal control of slavery in the territories were multifaceted. He often referred to the founding 'fathers' to ground his arguments in historical context. His interpretation of the 'constitution' was central to his stance on federal authority over slavery. The term 'federal' was used to discuss the extent of federal power in this context. The issue of slavery in the 'territories' was a key point of debate, and Lincoln used 'history' to bolster his arguments about the founders' intentions.\", \"weighted_keywords\": {\"fathers\": 10, \"constitution\": 9, \"federal\": 8, \"territories\": 7, \"history\": 6}, \"year_keywords\": [1858, 1859, 1860, 1861], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln describe the financial state of the United States during the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In addressing the financial state of the United States during the Civil War, Abraham Lincoln likely discussed various economic challenges and strategies. The war imposed a significant financial burden, necessitating innovative fiscal policies. Lincoln's administration introduced measures like the first income tax and the issuance of paper currency. Discussions about the 'dollar' during this era would have been pivotal, as it represented the country's struggle to maintain economic stability amidst rising costs and the need for effective currency to fund the war effort. Lincoln's focus would have included managing national debt, ensuring the functionality of the treasury, and implementing taxes to support war expenses. The term 'dollar' reflects the core of these economic discussions, being the fundamental unit of currency and a symbol of the nation's financial health and policies.\", \"weighted_keywords\": {\"economy\": 10, \"dollar\": 9, \"tax\": 8, \"debt\": 7, \"treasury\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "In what ways did Lincoln highlight the contributions of African Americans in the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"During the Civil War, Abraham Lincoln's acknowledgment of African Americans' contributions would have centered significantly on their roles as soldiers and their quest for freedom. The use of terms like 'negro' and 'black' in his speeches would be indicative of the period's language and context, reflecting on the participation of African Americans in the war. Lincoln likely highlighted their service in the Union Army, which became increasingly prominent following the Emancipation Proclamation. This historic document not only declared the freedom of slaves in the Confederacy but also set the stage for African Americans to actively participate as soldiers in the war, fighting for their emancipation and the broader cause of liberty. Lincoln's speeches during this time would have woven these themes together, recognizing the bravery and significance of African American soldiers in the struggle for union and freedom.\", \"weighted_keywords\": {\"negro\": 10, \"soldier\": 9, \"black\": 8, \"emancipation\": 7, \"freedom\": 6}, \"year_keywords\": [1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How can a nation effectively address internal divisions and dissenting opinions while striving for growth and unity?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's approach to addressing internal divisions and dissenting opinions, particularly during the Civil War, can be gleaned from his speeches. He emphasized the importance of unity and the 'better angels of our nature' in his First Inaugural Address. His speeches often called for reconciliation and understanding among the divided states. Lincoln stressed the idea of a united nation, 'conceived in Liberty, and dedicated to the proposition that all men are created equal,' especially in his Gettysburg Address. He believed in a government 'of the people, by the people, for the people,' underscoring the need for collective effort in resolving internal conflicts while pursuing growth and unity. His speeches provide insights into his views on managing dissent and fostering national unity in the face of profound division.\", \"weighted_keywords\": {\"Angels\": 10, \"Conceived\": 9, \"Proposition\": 8, \"Dedicated\": 7, \"Perish\": 6}, \"year_keywords\": [1861, 1863], \"text_keywords\": [\"First Inaugural Address\", \"Gettysburg Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What are the best practices for cloud computing in modern software engineering?"}, {"role": "assistant", "content": "{\"initial_answer\": \"This query is outside the scope of Abraham Lincoln's speeches and historical context. Please provide a query related to Abraham Lincoln or the historical period of his presidency.\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Lincoln thing about stuff?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Your query is unclear. Could you please provide more specific details or context related to Abraham Lincoln's speeches or his presidency?\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Abraham Lincoln’s opinion on the use of social media for political campaigns?"}, {"role": "assistant", "content": "{\"initial_answer\": \"This query is outside the scope of Abraham Lincoln's speeches and historical context. Please provide a query related to Abraham Lincoln or the historical period of his presidency.\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What did Lincoln say about Henry Clay's role in American history, especially regarding slavery and the Union?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln often spoke highly of Henry Clay, particularly regarding Clay's efforts to address contentious issues like slavery and the Union. Lincoln admired Clay's ability to navigate complex political landscapes, emphasizing his role in critical compromises that aimed to balance divergent interests and preserve national unity.\", \"weighted_keywords\": {\"Henry\": 20, \"Clay\": 20, \"Compromise\": 8, \"Slavery\": 7, \"Union\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Lincoln's stance on the Kansas-Nebraska Act and its impact on national unity?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln viewed the Kansas-Nebraska Act as a significant threat to national unity. He argued that the Act undermined the Missouri Compromise, which was essential in maintaining a balance between free and slave states. Lincoln believed that restoring the Missouri Compromise was crucial for preserving national harmony and preventing the spread of slavery to new territories.\", \"weighted_keywords\": {\"Kansas\": 10, \"Nebraska\": 10, \"Compromise\": 8, \"Missouri\": 7, \"slavery\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln address the issue of negotiating with the Confederacy during the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In addressing the Civil War, Lincoln was clear about the challenges of negotiating with insurgents. He often articulated that such negotiations were complex due to the insurgents' demands, which he viewed as incompatible with the preservation of the Union. However, he also acknowledged the potential for reconciliation and emphasized the importance of reuniting the nation.\", \"weighted_keywords\": {\"Rebel\": 10, \"Insurgent\": 9, \"Compromise\": 8, \"Negotiation\": 7, \"Confederacy\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln propose to address the issue of slavery in his Second Annual Message?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In his Second Annual Message, Lincoln proposed a plan of compensated emancipation as a means to address the issue of slavery. He considered this plan to be both just and economical, presenting it as a pragmatic solution to preserve the Union and pave the way for the country's future prosperity and growth.\", \"weighted_keywords\": {\"Emancipation\": 10, \"Compensation\": 9, \"slavery\": 8, \"Union\": 7, \"Economical\": 6}, \"year_keywords\": null, \"text_keywords\": [\"Second Annual Message\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Lincoln's view of Stephen Douglas's stance on the Kansas-Nebraska Act?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln critiqued Stephen Douglas's approach to the Kansas-Nebraska Act, particularly highlighting Douglas's failure to recognize the moral dimensions of the slavery issue. He argued that Douglas viewed the matter as merely a political or legal question, overlooking the profound moral concerns held by the majority. Lincoln emphasized the need to acknowledge and address these moral aspects, especially in the context of national legislation like the Kansas-Nebraska Act.\", \"weighted_keywords\": {\"Douglas\": 20, \"Kansas\": 9, \"Nebraska\": 8, \"Legislation\": 7, \"Slavery\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln defend the Emancipation Proclamation?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln defended the Emancipation Proclamation by emphasizing its role as a crucial military measure during the Civil War. He argued that it was a necessary step to weaken the Confederacy's war effort by depriving it of labor and adding manpower to the Union cause. Lincoln also highlighted its constitutional validity, framing it as an exercise of his presidential war powers. Beyond its strategic importance, Lincoln saw the proclamation as a moral imperative, a step towards justice and the eventual abolition of slavery.\", \"weighted_keywords\": {\"Emancipation\": 10, \"Proclamation\": 9, \"Military\": 8, \"Constitution\": 7, \"Abolition\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln address the military occupation of Confederate territory?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln's approach to the military occupation of Confederate territory was closely intertwined with his broader plans for reconstruction and the reintegration of insurgent states into the Union. He focused on the administration of these occupied areas, aiming to establish a framework that would allow for a smooth postwar transition. His policies were designed to balance the immediate needs of occupation with the long-term goal of restoring civil governance and national unity.\", \"weighted_keywords\": {\"Reconstruction\": 10, \"Adminstration\": 9, \"Insurgent\": 8, \"Pardon\": 7, \"Occupy\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln regard immigration as a social issue?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln viewed immigration positively, emphasizing the importance of welcoming new settlers and supporting their naturalization. He saw the United States as a land of opportunity for emigrants, advocating policies like the Homestead Act that encouraged immigration and settlement. Lincoln appreciated the contributions of immigrants to American society and the economy.\", \"weighted_keywords\": {\"Immigration\": 10, \"Naturalization\": 9, \"Settlers\": 8, \"Homestead\": 7, \"Emigrant\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did neutral nations like England and France impact the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Neutral nations like England and France had a complex impact on the American Civil War. While officially maintaining neutrality, their economic and diplomatic actions indirectly influenced the conflict. England's industrial need for cotton led to a form of economic dependency on the Confederacy, while France's geopolitical interests shaped its diplomatic stance. Both nations' decisions regarding trade, recognition, and diplomacy played crucial roles in the dynamics of the Civil War.\", \"weighted_keywords\": {\"Britain\": 20, \"France\": 20, \"Neutral\": 8, \"Diplomacy\": 7, \"Intervention\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Lincoln's relationship to the Supreme Court during the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"During the Civil War, Lincoln's engagement with the Supreme Court was marked by significant legal and constitutional challenges. The judiciary, especially the Supreme Court, played a crucial role in interpreting the Constitution in the context of wartime measures. Key issues included the suspension of habeas corpus and the balance of power between the executive and judicial branches. Lincoln's actions, often driven by the urgencies of war, led to notable legal disputes and set precedents in American constitutional law.\", \"weighted_keywords\": {\"Supreme\": 20, \"Court\": 20, \"Judiciary\": 8, \"Constitution\": 7, \"Habeas\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How important were border states like Kentucky, Maryland, and Missouri during the Civil War?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln viewed the border states of Kentucky, Maryland, and Missouri as strategically crucial during the Civil War. He recognized their political and symbolic significance in maintaining the Union and preventing further secession. Lincoln's policies and actions reflected the importance he placed on these states, particularly in balancing the delicate issue of slavery while trying to keep them aligned with the Union. The loyalty of these states was vital in the context of geography, resources, and political influence.\", \"weighted_keywords\": {\"Kentucky\": 20, \"Maryland\": 20, \"Missouri\": 20, \"Border\": 8, \"States\": 8}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln explain the importance of the naval blockade against the Confederacy?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln emphasized the naval blockade as a critical strategy in the Civil War, focusing on its role in intercepting and disrupting Confederate trade and supply lines. The blockade was designed to prevent the shipment of goods, particularly cotton, to foreign markets and to block the import of essential supplies and military equipment. By deploying naval forces to intercept ships attempting to breach the blockade, Lincoln aimed to weaken the economic and military capabilities of the Confederacy, thereby exerting significant pressure on their war effort.\", \"weighted_keywords\": {\"Blockade\": 10, \"Naval\": 9, \"Intercept\": 8, \"Trade\": 7, \"Ship\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln's speeches communicate his strategies for post-war reconstruction?"}, {"role": "assistant", "content": "{\"initial_answer\": \"In his speeches, Lincoln outlined a vision for post-war reconstruction focused on reconciliation and reintegration of the insurgent states. He emphasized a reconstruction policy marked by 'malice toward none' and 'charity for all,' signaling his desire for a compassionate and inclusive approach to reunifying the nation. Lincoln's speeches often highlighted the importance of taking an oath of allegiance as a step toward reconstruction, aiming to bring the Southern states back into the Union with minimal punishment and maximal healing.\", \"weighted_keywords\": {\"Reconstruction\": 20, \"Malice\": 9, \"Charity\": 8, \"Insurgent\": 7, \"Oath\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What did Lincoln's speeches reveal about his evolving stance on slavery and emancipation?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln's stance on slavery and emancipation evolved significantly, as seen in key speeches. In the 'Peoria Speech,' he argued against the expansion of slavery. The 'House Divided' speech highlighted the unsustainable nature of a nation divided by slavery. The 'Emancipation Proclamation' marked a definitive shift towards abolition. These speeches reflect a journey from opposing slavery's expansion to actively working towards its end.\", \"weighted_keywords\": {\"Divided\": 10, \"Emancipation\": 9, \"Proclamation\": 8, \"Moral\": 7, \"Liberty\": 6}, \"year_keywords\": null, \"text_keywords\": [\"At Peoria\", \"House Divided\", \"Emancipation Proclamation\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln use his speeches to rally public support for the Union war effort?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln used his oratory skills in key speeches to galvanize public support for the Union war effort. His 'First Inaugural Address' focused on the enduring nature of the Union, the 'Gettysburg Address' underscored the sacrifice for a nation 'conceived in liberty,' and the 'Second Inaugural Address' sought to heal the nation's wounds with 'malice toward none.' These speeches played a pivotal role in strengthening the Union's resolve and moral purpose during the Civil War.\", \"weighted_keywords\": {\"Passion\": 10, \"Strained\": 9, \"Gettysburg\": 8, \"Charity\": 7, \"Malice\": 6}, \"year_keywords\": [1861, 1863, 1865], \"text_keywords\": [\"First Inaugural Address\", \"Gettysburg Address\", \"Second Inaugural Address\"]}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did George Washington think about the institution of slavery?"}, {"role": "assistant", "content": "{\"initial_answer\": \"This query is outside the scope of Abraham Lincoln's speeches and historical context. Please provide a query related to Abraham Lincoln or the historical period of his presidency.\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "Tell me about the thing about the story."}, {"role": "assistant", "content": "{\"initial_answer\": \"Your query is unclear. Could you please provide more specific details or context related to Abraham Lincoln's speeches or his presidency?\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Abraham Lincoln’s opinion on the use of solar power for renewable energy?"}, {"role": "assistant", "content": "{\"initial_answer\": \"This query is outside the scope of Abraham Lincoln's speeches and historical context. Please provide a query related to Abraham Lincoln or the historical period of his presidency.\", \"weighted_keywords\": {}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln regard John Brown's raid on Harper's Ferry?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln's response to John Brown's raid on Harper's Ferry in Virginia was one of clear disapproval. While empathizing with the anti-slavery sentiment, Lincoln condemned the violent methods employed by Brown. He viewed the raid as an act of insurrection that could further inflame tensions between the North and South. Lincoln stressed the importance of addressing slavery within the legal framework of the Constitution, rather than through violent means. His reaction to the Harper's Ferry raid underscored his commitment to lawful methods in the pursuit of societal change.\", \"weighted_keywords\": {\"Harper\": 20, \"Ferry\": 20, \"Brown\": 20, \"Virginia\": 8, \"Raid\": 7}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln regard democracy as a form of government?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln deeply esteemed democracy as the cornerstone of American government. He viewed it as a system where the will of the people played a central role in shaping the republic. In his views, democracy was more than just a political system; it was a manifestation of popular sovereignty and a reflection of the nation's commitment to the principles of liberty and equality. His speeches, including the Gettysburg Address, often highlighted the concept of 'government of the people, by the people, for the people,' emphasizing the fundamental role of democratic governance in ensuring a just and representative society.\", \"weighted_keywords\": {\"Democracy\": 20, \"Republic\": 7, \"Government\": 6, \"People\": 5, \"Gettysburg\": 4}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln oversee American monetary policy during his adminstration? "}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln's administration was marked by pivotal shifts in American monetary policy, crucial for financing the Civil War and stabilizing the economy. He oversaw the introduction of a national paper currency, diverging from the traditional reliance on specie (gold and silver). The Legal Tender Act authorized the issuance of 'greenbacks,' a significant move away from specie-backed currencies. Additionally, the National Banking Act under Lincoln's guidance reformed the banking system, aiming to establish a stable, standardized currency across states. The Treasury Department, under his administration, played a critical role in these financial reforms.\", \"weighted_keywords\": {\"Currency\": 10, \"Specie\": 9, \"Banking\": 8, \"Treasury\": 7, \"Finance\": 6}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln's presidency contribute to the construction of the Transcontinental Railroad?"}, {"role": "assistant", "content": "{\"initial_answer\": \"During Lincoln's presidency, the groundwork for the Transcontinental Railroad was significantly advanced. Lincoln, understanding the strategic importance of a railroad spanning the continent, supported key legislation to make this ambitious project a reality. The Pacific Railway Act of 1862, signed by Lincoln, was pivotal. It provided federal government support for the construction of the railroad, including land grants and financing. This act facilitated the building of the railroad, connecting the eastern and western parts of the United States, and greatly influencing the nation's economic and social development.\", \"weighted_keywords\": {\"Transcontinental\": 20, \"Railroad\": 20, \"Pacific\": 8, \"Railway\": 7, \"Construction\": 6}, \"year_keywords\": [1861, 1862, 1863, 1864, 1865], \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "How did Lincoln invoke the language of religion in his speeches?"}, {"role": "assistant", "content": "{\"initial_answer\": \"Lincoln's speeches often incorporated the language of religion, using references to God, providence, and biblical themes to imbue his messages with moral and spiritual depth. He invoked divine guidance in the nation's affairs, especially during the Civil War, suggesting that events were part of a greater divine plan. This religious rhetoric was evident in speeches like his Second Inaugural Address, where he interpreted the war in the context of God's will and used biblical references to appeal to a broader moral and holy cause. Lincoln's use of religious language helped frame the Union's struggle in a context that transcended political objectives, aligning it with a higher moral purpose.\", \"weighted_keywords\": {\"God\": 9, \"Providence\": 8, \"Divine\": 7, \"Bible\": 6, \"Holy\": 5}, \"year_keywords\": null, \"text_keywords\": null}"}]}, {"messages": [{"role": "system", "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."}, {"role": "user", "content": "What was Lincoln's vision for African Americans after the Civil War"}, {"role": "assistant", "content": "{\"initial_answer\": \"Abraham Lincoln's vision for African Americans post-Civil War was progressive for his time, focusing on their integration into American society as free citizens. He believed in the possibility of a United States where former slaves could enjoy the full benefits of freedom, including economic and social opportunities. Lincoln's views on the specifics of integration and rights varied over time, but he consistently supported the idea of emancipation and was beginning to explore the concept of voting rights for African Americans, particularly those who had served in the Union Army, by the end of his life.\", \"weighted_keywords\": {\"Lincoln\": 10, \"Speech\": 8, \"Freedom\": 6, \"Democracy\": 4, \"Union\": 2}, \"year_keywords\": null, \"text_keywords\": null}"}]}]

print(f"Hay FT examples loaded: {len(HAY_FT)}")


In [ ]:
    # 0.5  Format primer — Line 45: Harper's Ferry (strongest example)
    # Chosen because: three corpus-rare proper nouns all at weight 20,
    # clear discrimination logic, no text_keywords/year_keywords clutter.

    HAY_PRIMER = {
  "messages": [
    {
      "role": "system",
      "content": "Generate a detailed and accurate initial response to the user's query, focusing on Abraham Lincoln's perspectives and historical context. After the initial response, identify and list a set of weighted keywords. These keywords should be unique and specifically tailored to the query, with an emphasis on low-frequency, single-word terms that are likely to appear in Lincoln's speeches and writings. Prioritize words that are significant yet less common, as they will be more effective for dynamic keyword searches against the Lincoln speech corpus. Additionally, if a particular speech or document by Lincoln is highly relevant to the query, please specify it. This approach will aid in narrowing down the search to the most pertinent texts. Ensure that the keywords and references are historically accurate and directly address the user's query."
    },
    {
      "role": "user",
      "content": "How did Lincoln regard John Brown's raid on Harper's Ferry?"
    },
    {
      "role": "assistant",
      "content": "{\"initial_answer\": \"Lincoln's response to John Brown's raid on Harper's Ferry in Virginia was one of clear disapproval. While empathizing with the anti-slavery sentiment, Lincoln condemned the violent methods employed by Brown. He viewed the raid as an act of insurrection that could further inflame tensions between the North and South. Lincoln stressed the importance of addressing slavery within the legal framework of the Constitution, rather than through violent means. His reaction to the Harper's Ferry raid underscored his commitment to lawful methods in the pursuit of societal change.\", \"weighted_keywords\": {\"Harper\": 20, \"Ferry\": 20, \"Brown\": 20, \"Virginia\": 8, \"Raid\": 7}, \"year_keywords\": null, \"text_keywords\": null}"
    }
  ]
}

    HAY_SYSTEM_PROMPT = HAY_PRIMER["messages"][0]["content"]
    HAY_PRIMER_OUTPUT = HAY_PRIMER["messages"][2]["content"]   # assistant turn

    print("=== HAY PRIMER QUERY ===")
    print(HAY_PRIMER["messages"][1]["content"])
    print("\n=== HAY PRIMER OUTPUT ===")
    print(HAY_PRIMER_OUTPUT)


In [ ]:
# 0.6  Contamination tracker
contamination_log = {"chunk_ids_used": [], "queries_used": [], "sections_run": []}

def log_contamination(section, chunk_ids=None, queries=None):
    contamination_log["sections_run"].append(section)
    if chunk_ids: contamination_log["chunk_ids_used"].extend(
        [c for c in chunk_ids if c and "XX" not in str(c)])
    if queries:   contamination_log["queries_used"].extend(queries)

# Output collectors
approved_hay     = []
approved_nicolay = []

def approve_hay(example_dict):
    approved_hay.append(example_dict)
    print(f"  ✓ Hay approved  (total: {len(approved_hay)})")

def approve_nicolay(example_dict):
    approved_nicolay.append(example_dict)
    print(f"  ✓ Nicolay approved  (total: {len(approved_nicolay)})")

print("Contamination tracker and output collectors ready.")


In [ ]:
# 0.7  Shared verification helper
def verify_hay_candidate(task, query, output, chunk_id=None):
    keywords = output.get("weighted_keywords") or {}
    print(f"\n{'='*62}")
    print(f"TASK   : {task}")
    print(f"QUERY  : {query[:90]}")
    ia = output.get("initial_answer","")
    print(f"IA     : {ia[:180]}")
    print(f"\n{'KEYWORD':<24} {'WEIGHT':>6}  {'CORPUS_FREQ':>12}  QUALITY")
    print("-"*62)
    all_good = True
    for term, weight in sorted(keywords.items(), key=lambda x: -x[1]):
        freq    = term_freq(term)
        quality = classify_keyword(term, weight)
        print(f"  {term:<22} {weight:>6}  {freq:>12}  {quality}")
        if "⚠" in quality: all_good = False
    print(f"\nYear keywords : {output.get('year_keywords')}")
    print(f"Text keywords : {output.get('text_keywords')}")
    if chunk_id and get_chunk(chunk_id):
        chunk_text = get_chunk(chunk_id).get("full_text","").lower()
        missing = [t for t in keywords if t.lower() not in chunk_text]
        if missing: print(f"\n⚠ TERMS NOT IN TARGET CHUNK: {missing}")
        else:        print(f"\n✓ All keywords present in target chunk")
    print(f"\nOVERALL: {'PASS' if all_good else 'NEEDS REVIEW'}")
    return all_good

def wrap_hay_ft(query, output, task="", section="", edits="", chunk_id=""):
    """Wrap a generated Hay output in OpenAI fine-tuning message format."""
    return {
        "messages": [
            {"role": "system",    "content": HAY_SYSTEM_PROMPT},
            {"role": "user",      "content": query},
            {"role": "assistant", "content": json.dumps(output)},
        ],
        "_meta": {"task": task, "section": section, "edits": edits, "chunk_id": chunk_id},
    }

print("Helpers ready.")


---
## Section 1 — Hay: Synthesis Discrimination

**Gap (dev log #72, failure modes #5, #55, #59):** For thematic synthesis queries,
Hay generates high-frequency broad terms ("Liberty," "Law," "God") that fire widely
but fail to surface the *specific* passages most relevant to the query.

**What new examples must model:**
- Corpus-rare secondary discriminators weighted *higher* than the broad thematic anchor
- `initial_answer` framed as a directional hypothesis, not a confident survey

**Confirmed rare discriminators from Voyant (corpus = 63,489 words):**

| Query type | Rare discriminator | rawFreq | Target weight |
|---|---|---|---|
| Divine providence | `almighty` | 5 | 18 |
| Divine providence | `offense` | 3 | 18 |
| Divine providence | `malice` | 3 | 16 |
| Liberty / law | `jurisprudence` | 1 | 20 |
| Liberty / law | `perpetuity` | 8 | 18 |
| Liberty / law | `proposition` | 50 | 10 |
| Constitution / slavery | `framers` | 12 | 18 |
| Constitution / slavery | `insurgent` | 13 | 16 |
| Constitution / slavery | `pardon` | 5 | 18 |

**Note:** `bondsman` and `thirty-nine` do not appear in Voyant — likely Voyant
lemmatized or excluded them. Use `almighty`/`offense`/`malice` for the divine
providence cluster instead.


In [ ]:
# 1.1  Find target chunks for each synthesis query type
TARGET_TERMS = {
    "divine_providence":    ["almighty", "offense", "malice", "charity"],
    "liberty_law":          ["jurisprudence", "perpetuity", "proposition"],
    "constitution_slavery": ["framers", "insurgent", "pardon"],
}

for task, terms in TARGET_TERMS.items():
    print(f"\n=== {task} ===")
    for term in terms:
        hits = [
            f"{item['text_id']} ({item.get('source','')[:45]})"
            for item in corpus
            if re.search(r'\b' + re.escape(term) + r'\b',
                         item.get("full_text",""), re.IGNORECASE)
        ]
        print(f"  '{term}': {hits[:4]}")


In [ ]:
# 1.2  Set anchor chunks after reviewing 1.1 output
# Fill in the text_id values that contain the most relevant content.

SYNTHESIS_CHUNKS = {
    "divine_providence":    "Text #: XX",   # <-- Second Inaugural cluster
    "liberty_law":          "Text #: XX",   # <-- First Inaugural / Cooper Union
    "constitution_slavery": "Text #: XX",   # <-- Cooper Union cluster
}

# Preview each chunk
for task, tid in SYNTHESIS_CHUNKS.items():
    chunk = get_chunk(tid)
    if chunk:
        print(f"\n=== {task} | {tid} ===")
        print(chunk["full_text"][:400])
    else:
        print(f"\n⚠ NOT FOUND: {tid} — update SYNTHESIS_CHUNKS")


In [ ]:
# 1.3  Queries for synthesis discrimination
# Edit wording here before running generation.

SYNTHESIS_QUERIES = {
    "divine_providence":
        "How does Lincoln invoke the concept of divine providence and moral judgment across his speeches?",
    "liberty_law":
        "How does Lincoln frame the relationship between liberty and legal order in his speeches?",
    "constitution_slavery":
        "How does Lincoln use constitutional reasoning to address the question of slavery?",
}

SYNTHESIS_RARE_TERMS = {
    "divine_providence":    ["almighty", "offense", "malice"],
    "liberty_law":          ["jurisprudence", "perpetuity"],
    "constitution_slavery": ["framers", "insurgent", "pardon"],
}

SYNTHESIS_GENERATION_SYSTEM = """\
You are building fine-tuning examples for Hay, a RAG retrieval model for Lincoln's speeches.
Hay outputs a JSON object with exactly these four keys:
  - initial_answer  : 2-4 sentence directional hypothesis (NOT a confident survey)
  - weighted_keywords : dict of {term: weight} where weight is 1-20
  - year_keywords   : list of relevant years as integers, or null
  - text_keywords   : list of specific speech titles, or null

CRITICAL KEYWORD RULES for this batch (thematic synthesis queries):
1. The corpus is 63,489 words. Terms with rawFreq > 50 are COMMON and must be
   weighted no higher than 8, even if thematically central.
2. Terms with rawFreq < 15 are RARE and should be weighted 15-20.
   These are your primary discriminators — they surface the specific passage.
3. You MUST include at least one rare discriminator alongside the thematic anchor.
   Example: for a divine providence query, 'almighty' (freq=5) at weight 18
   should outweigh 'God' (freq=unknown-but-common) at weight 6.
4. initial_answer must begin with hedging language:
   'My initial hypothesis is...' or 'Based on general knowledge, I believe...'
   It should NOT assert specific facts — those must come from retrieval.

Output valid JSON only. No text outside the JSON object.

FORMAT EXAMPLE (follow this schema exactly):
"""


In [ ]:
# 1.4  Generate synthesis discrimination candidates

synthesis_candidates = {}

for task, query in SYNTHESIS_QUERIES.items():
    chunk = get_chunk(SYNTHESIS_CHUNKS.get(task,""))
    if not chunk:
        print(f"SKIP {task}: chunk not set or not found"); continue

    rare_hint = ", ".join(f"'{t}' (freq={term_freq(t)})"
                          for t in SYNTHESIS_RARE_TERMS[task])
    prompt = [
        {"role": "system", "content": SYNTHESIS_GENERATION_SYSTEM + HAY_PRIMER_OUTPUT},
        {"role": "user",   "content":
            f"Generate a Hay fine-tuning example for this synthesis query.\n\n"
            f"Query: {query}\n\n"
            f"Grounding passage (your keywords must be able to retrieve this):\n"
            f"---\n{chunk['full_text'][:1400]}\n---\n\n"
            f"REQUIRED rare discriminators (weight >= 15): {rare_hint}\n"
            f"These are confirmed present in the corpus and rare enough to discriminate.\n"
            f"Do NOT weight broad terms like 'Lincoln','God','Liberty','Law','Union' above 8."
        },
    ]
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL, messages=prompt, temperature=0.4, max_tokens=600)
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        synthesis_candidates[task] = {
            "query": query, "output": parsed, "chunk_id": SYNTHESIS_CHUNKS[task]}
        kw_preview = sorted(parsed.get("weighted_keywords",{}).items(), key=lambda x:-x[1])[:4]
        print(f"  OK  top_kw={kw_preview}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}\n  raw: {raw[:300]}")

log_contamination("S1_synthesis", chunk_ids=list(SYNTHESIS_CHUNKS.values()),
                  queries=list(SYNTHESIS_QUERIES.values()))


In [ ]:
# 1.5  Verify synthesis candidates

for task, data in synthesis_candidates.items():
    verify_hay_candidate(task, data["query"], data["output"], data["chunk_id"])


In [ ]:
# 1.6  Approve — set flags and optional edit notes

APPROVE_S1 = {
    "divine_providence":    True,
    "liberty_law":          True,
    "constitution_slavery": True,
}
EDITS_S1 = {
    "divine_providence":    "",   # e.g. "raised 'almighty' from 12 to 18"
    "liberty_law":          "",
    "constitution_slavery": "",
}

for task, approve in APPROVE_S1.items():
    if not approve or task not in synthesis_candidates: continue
    data = synthesis_candidates[task]
    approve_hay(wrap_hay_ft(data["query"], data["output"],
                            task=task, section="S1",
                            edits=EDITS_S1[task], chunk_id=data["chunk_id"]))

print(f"\nApproved Hay examples so far: {len(approved_hay)}")


---
## Section 2 — Hay: HyDE Hypothesis Framing

**Gap (dev log #73, failure mode 1 — Q7):** Hay's `initial_answer` propagates
hallucinated specifics (fabricated vote tallies, figures) that Nicolay then
confirms rather than interrogates against retrieved evidence.

**What new examples must model:**
- `initial_answer` explicitly hedged: never states specific numbers or dates as facts
- Closes with: "Retrieval should confirm these details against the corpus text."
- Keywords remain precise and corpus-rare — the hedge is in the *answer*, not the keywords

**Target query types:** numerical/statistical facts; specific event-date attributions;
cross-speech comparative claims where exact attribution matters.


In [ ]:
# 2.1  Find chunks with verifiable specific data (numbers, figures, dates)
# These become the ground-truth that initial_answer must NOT pre-empt.

DATA_TERMS = ["33,762", "000", "revenue", "tonnage", "appropriat",
              "postmaster", "coinage", "treasury", "percent"]

for term in DATA_TERMS:
    hits = [item["text_id"] for item in corpus
            if term.lower() in item.get("full_text","").lower()]
    if hits: print(f"  '{term}': {hits[:5]}")


In [ ]:
# 2.2  Set anchor chunks and customize query text
# Choose chunks where a naive LLM might guess a plausible-but-wrong specific figure.

HYPOTHESIS_CHUNKS = {
    "factual_statistical":  "Text #: XX",   # <-- chunk with numeric data
    "factual_event_date":   "Text #: XX",   # <-- chunk where speech/date attribution matters
    "factual_cross_speech": "Text #: XX",   # <-- chunk where cross-speech comparison is needed
}

HYPOTHESIS_QUERIES = {
    "factual_statistical":
        "What specific figures did Lincoln report regarding [TOPIC] in his Annual Messages?",
    "factual_event_date":
        "In which specific speech and year did Lincoln first address [TOPIC]?",
    "factual_cross_speech":
        "How did Lincoln's specific claims about [TOPIC] change between his speeches?",
}
# Customize the bracketed [TOPIC] placeholders before running generation.

for task, tid in HYPOTHESIS_CHUNKS.items():
    chunk = get_chunk(tid)
    if chunk: print(f"\n{task} | {tid}\n{chunk['full_text'][:350]}")
    else:     print(f"\n⚠ NOT FOUND: {tid}")


In [ ]:
# 2.3  Generate hypothesis framing candidates

HYPOTHESIS_SYSTEM = """\
You are building fine-tuning examples for Hay, a RAG retrieval model.

The CRITICAL requirement for this batch: initial_answer must be an explicit
retrieval hypothesis — NOT a confident factual claim.

Rules for initial_answer:
1. Begin with: 'My initial hypothesis is...', 'Based on general knowledge, I believe...',
   or 'Retrieval should determine whether...'
2. NEVER state specific numbers, vote tallies, dollar amounts, or dates as facts.
   Write 'some specific figure Lincoln reported' or 'a date I cannot confirm without
   retrieval' rather than guessing a value.
3. End with: 'Retrieval should confirm these details against the corpus text.'

weighted_keywords must still be precise and corpus-aware — rare proper nouns and
specific terms from the target passage, weighted by rarity (rare = higher weight).
Output valid JSON only.

FORMAT EXAMPLE:
"""

HEDGE_PHRASES = [
    "my initial hypothesis", "i believe", "i tentatively", "retrieval should",
    "based on general", "based on my", "may have", "cannot confirm",
    "some specific figure", "some figure", "a date", "unclear",
]

hypothesis_candidates = {}

for task, query in HYPOTHESIS_QUERIES.items():
    chunk = get_chunk(HYPOTHESIS_CHUNKS.get(task,""))
    if not chunk: print(f"SKIP {task}"); continue
    prompt = [
        {"role": "system", "content": HYPOTHESIS_SYSTEM + HAY_PRIMER_OUTPUT},
        {"role": "user",   "content":
            f"Query: {query}\n\nGrounding passage:\n---\n{chunk['full_text'][:1400]}\n---\n\n"
            f"Remember: initial_answer must NOT guess specific numbers. "
            f"Keywords should be corpus-rare proper nouns and specific terms."
        },
    ]
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL, messages=prompt, temperature=0.4, max_tokens=600)
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        ia = parsed.get("initial_answer","")
        found_hedges = [p for p in HEDGE_PHRASES if p in ia.lower()]
        hypothesis_candidates[task] = {
            "query": query, "output": parsed, "chunk_id": HYPOTHESIS_CHUNKS[task]}
        print(f"  OK  hedges_found={found_hedges if found_hedges else '⚠ NONE'}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}")

log_contamination("S2_hypothesis", chunk_ids=list(HYPOTHESIS_CHUNKS.values()),
                  queries=list(HYPOTHESIS_QUERIES.values()))


In [ ]:
# 2.4  Verify hypothesis framing candidates

for task, data in hypothesis_candidates.items():
    verify_hay_candidate(task, data["query"], data["output"], data["chunk_id"])
    # Extra check: no specific numbers in initial_answer
    ia = data["output"].get("initial_answer","")
    numbers = re.findall(r'\b\d{2,}\b', ia)
    if numbers: print(f"  ⚠ NUMBERS IN INITIAL_ANSWER: {numbers} — review before approving")
    else:       print(f"  ✓ No specific numbers in initial_answer")


In [ ]:
# 2.5  Approve hypothesis framing examples

APPROVE_S2 = {k: True for k in hypothesis_candidates}
EDITS_S2   = {k: ""   for k in hypothesis_candidates}

for task, approve in APPROVE_S2.items():
    if not approve or task not in hypothesis_candidates: continue
    data = hypothesis_candidates[task]
    approve_hay(wrap_hay_ft(data["query"], data["output"],
                            task=task, section="S2",
                            edits=EDITS_S2.get(task,""), chunk_id=data["chunk_id"]))

print(f"\nApproved Hay examples so far: {len(approved_hay)}")


---
## Section 3 — Hay + Nicolay: Race/Citizenship Sensitivity

**Gap (dev log #1, #60, failure mode 6 — Q-new scored 0/4):**
The system conflates anti-slavery with Black citizenship/equality advocacy,
adopting the uncritical "Great Emancipator" progressive narrative.

**Confirmed rare discriminators (Voyant):**

| Term | rawFreq | Note |
|---|---|---|
| `suffrage` | 2 | Direct citizenship signal |
| `citizenship` | 2 | Direct citizenship signal |
| `colonization` | 13 | Lincoln's early emigration advocacy |
| `inferior` | 11 | Lincoln-Douglas debate framing |
| `colored` | 18 | Period-appropriate descriptor |
| `negro` | 69 | Common — weight ≤ 8 |

**Corpus limitation:** Lincoln-Douglas debates are not in the corpus. Examples
must work with what's available (Peoria, Cooper Union, Conkling letter, Annual
Messages) while the Nicolay example explicitly flags this gap.


In [ ]:
# 3.1  Find citizenship-relevant chunks in current corpus

CITIZENSHIP_TERMS = ["colonization", "suffrage", "inferior", "equality",
                     "citizenship", "negro", "colored", "emancipat"]

print("Citizenship-relevant chunks:\n")
for term in CITIZENSHIP_TERMS:
    hits = [
        f"{item['text_id']} ({item.get('source','')[:50]})"
        for item in corpus
        if re.search(r'\b' + re.escape(term) + r'\b',
                     item.get("full_text",""), re.IGNORECASE)
    ]
    if hits: print(f"  '{term}' (freq={term_freq(term)}): {hits[:4]}")


In [ ]:
# 3.2  Set anchor chunks for Hay citizenship examples

CITIZENSHIP_HAY_CHUNKS = {
    "colonization_early":  "Text #: XX",   # Lincoln's colonization advocacy
    "conditional_service": "Text #: XX",   # Conkling letter: Black soldiers
    "late_war_equality":   "Text #: XX",   # late Annual Message or last address
}

CITIZENSHIP_HAY_QUERIES = {
    "colonization_early":
        "What were Lincoln's early views on colonization and the status of free Black Americans?",
    "conditional_service":
        "How did Lincoln connect African American military service to questions of citizenship?",
    "late_war_equality":
        "How did Lincoln's statements about African American rights evolve toward the end of the Civil War?",
}

for task, tid in CITIZENSHIP_HAY_CHUNKS.items():
    chunk = get_chunk(tid)
    if chunk: print(f"\n{task} | {tid}\n{chunk['full_text'][:400]}")
    else:     print(f"\n⚠ NOT FOUND: {tid}")


In [ ]:
# 3.3  Generate Hay citizenship examples

CITIZENSHIP_HAY_SYSTEM = """\
You are building fine-tuning examples for Hay, a RAG retrieval model for Lincoln's speeches.

This batch covers Lincoln's evolving views on race, citizenship, and equality —
a historiographically sensitive topic requiring careful scholarly framing.

Rules for initial_answer:
1. Do NOT adopt the 'Great Emancipator' progressive narrative uncritically.
2. Acknowledge Lincoln's positions were evolving and politically constrained.
3. Distinguish between: (a) opposition to slavery's *extension*, (b) early
   support for colonization/emigration, (c) conditional late-war statements
   on limited Black suffrage. These are historically distinct positions.
4. Frame as hypothesis: 'My initial assessment is that Lincoln's views were
   complex and evolved under political pressure...'

Rules for weighted_keywords (confirmed Voyant frequencies):
- 'colonization' rawFreq=13  → weight 18  (rare, precise)
- 'suffrage'     rawFreq=2   → weight 20  (very rare, highly precise)
- 'citizenship'  rawFreq=2   → weight 20
- 'inferior'     rawFreq=11  → weight 16
- 'colored'      rawFreq=18  → weight 14
- 'negro'        rawFreq=69  → weight ≤ 8  (common)
- 'emancipation' rawFreq=high → weight ≤ 8  (too broad for citizenship queries)

Output valid JSON only. FORMAT EXAMPLE:
"""

citizenship_hay_candidates = {}

for task, query in CITIZENSHIP_HAY_QUERIES.items():
    chunk = get_chunk(CITIZENSHIP_HAY_CHUNKS.get(task,""))
    if not chunk: print(f"SKIP {task}"); continue
    prompt = [
        {"role": "system", "content": CITIZENSHIP_HAY_SYSTEM + HAY_PRIMER_OUTPUT},
        {"role": "user",   "content":
            f"Query: {query}\n\nGrounding passage:\n---\n{chunk['full_text'][:1400]}\n---\n\n"
            f"Prioritize: 'suffrage'(2), 'citizenship'(2), 'colonization'(13), "
            f"'inferior'(11), 'colored'(18) at weights 14-20.\n"
            f"Do NOT weight 'emancipation', 'freedom', 'slavery', 'negro' above 8."
        },
    ]
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL, messages=prompt, temperature=0.4, max_tokens=600)
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        citizenship_hay_candidates[task] = {
            "query": query, "output": parsed, "chunk_id": CITIZENSHIP_HAY_CHUNKS[task]}
        kw_preview = sorted(parsed.get("weighted_keywords",{}).items(), key=lambda x:-x[1])[:4]
        print(f"  OK  top_kw={kw_preview}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}")

log_contamination("S3_citizenship_hay",
                  chunk_ids=list(CITIZENSHIP_HAY_CHUNKS.values()),
                  queries=list(CITIZENSHIP_HAY_QUERIES.values()))


In [ ]:
# 3.4  Verify and approve Hay citizenship examples

for task, data in citizenship_hay_candidates.items():
    verify_hay_candidate(task, data["query"], data["output"], data["chunk_id"])

APPROVE_S3_HAY = {k: True for k in citizenship_hay_candidates}

for task, approve in APPROVE_S3_HAY.items():
    if not approve or task not in citizenship_hay_candidates: continue
    data = citizenship_hay_candidates[task]
    approve_hay(wrap_hay_ft(data["query"], data["output"],
                            task=task, section="S3_hay", chunk_id=data["chunk_id"]))

print(f"\nApproved Hay examples so far: {len(approved_hay)}")


### Section 3b — Nicolay: Race/Citizenship Chain-of-Thought

Generate a Nicolay example that corrects the Q-new failure pattern:
- Received a plausible but overclaiming `initial_answer` (anti-slavery = equality framing)
- Retrieved citizenship-relevant passages
- Must explicitly distinguish what the corpus *does* and *does not* support


In [ ]:
# 3.5  Build synthetic Nicolay input for citizenship query

def build_nicolay_input(query, initial_answer, match_chunks):
    matches = []
    for i, chunk in enumerate(match_chunks[:3], 1):
        matches.append(
            f"Match {i}: Text ID - {chunk['text_id']}, "
            f"Source - {chunk.get('source','')}, "
            f"Summary - {chunk.get('summary','')}, "
            f"Full Text (select the most relevant passage to quote directly) - "
            f"{chunk.get('full_text','')[:900]}, "
            f"Relevance Score - 0.82"
        )
    return (f"User Query: {query}\n\n"
            f"Initial Answer: {initial_answer}\n\n"
            + "\n\n".join(matches))

Q_NEW_QUERY = (
    "How did Lincoln's views on African American citizenship and racial "
    "equality evolve across his speeches?"
)

# Deliberately overclaiming initial_answer — mirrors Q-new failure for Nicolay to correct
Q_NEW_INITIAL = (
    "My initial hypothesis is that Lincoln's views on African American citizenship "
    "progressively evolved toward greater equality throughout his career, moving from "
    "cautious anti-slavery positions to broader advocacy for civil rights. Retrieval "
    "should confirm whether the corpus supports this trajectory."
)

# Fill in after running 3.1 — pick 3 best citizenship-relevant chunk IDs
Q_NEW_MATCH_IDS = ["Text #: XX", "Text #: XX", "Text #: XX"]

q_new_chunks = [get_chunk(tid) for tid in Q_NEW_MATCH_IDS if get_chunk(tid)]
q_new_input  = build_nicolay_input(Q_NEW_QUERY, Q_NEW_INITIAL, q_new_chunks)

print("=== NICOLAY INPUT PREVIEW (first 600 chars) ===")
print(q_new_input[:600])


In [ ]:
# 3.6  Generate Nicolay citizenship CoT example

NICOLAY_PRIMER_OUTPUT = nicolay_ft[0]["messages"][-1]["content"]   # adjust index if needed
NICOLAY_SYSTEM_PROMPT = nicolay_ft[0]["messages"][0]["content"]

CITIZENSHIP_NICOLAY_SYSTEM = """\
You are generating a fine-tuning example for Nicolay, a historical RAG synthesis model.

This example must model HISTORIOGRAPHICALLY NUANCED reasoning about Lincoln's
views on race and citizenship. Output a JSON object with these sections:
User Query Analysis, Initial Answer Review, Match Analysis (Match 1/2/3),
Meta Analysis, FinalAnswer (with Text and References), Model Feedback.

CRITICAL REQUIREMENTS:

1. Initial Answer Review must NOT validate the initial_answer's progressive framing.
   It must explicitly note:
   - 'Anti-slavery advocacy is not equivalent to advocacy for Black citizenship or equality'
   - Lincoln's early colonization/emigration support (1850s) must be acknowledged
   - Late-war statements on suffrage were conditional (literacy/military service),
     not a universal equality position

2. Match Analysis must distinguish what each passage actually says from what the
   query implies. For each match, state explicitly whether it is evidence of:
   (a) anti-slavery sentiment, (b) colonization advocacy, (c) conditional suffrage,
   or (d) positive citizenship claims — these are historically distinct.

3. FinalAnswer must:
   - Quote directly from retrieved passages with correct citations
   - Characterize Lincoln's position as evolving and contested, not linear
   - Note the absence of Lincoln-Douglas debate corpus as a significant gap

4. Model Feedback must flag: 'The Lincoln-Douglas debates (1858) are the primary
   missing evidence for this query and are not currently in the corpus.'

Output valid JSON only. Follow the schema of the format example exactly.
"""

print("Generating Nicolay citizenship example...")
resp = client.chat.completions.create(
    model=GENERATION_MODEL,
    messages=[
        {"role": "system", "content":
            CITIZENSHIP_NICOLAY_SYSTEM + "\n\nFORMAT EXAMPLE:\n" + NICOLAY_PRIMER_OUTPUT[:2500]},
        {"role": "user", "content": q_new_input},
    ],
    temperature=0.3, max_tokens=2800,
)
raw_nicolay_citizenship = resp.choices[0].message.content.strip()
try:
    parsed_nicolay_citizenship = json.loads(raw_nicolay_citizenship)
    print("JSON parse: OK")
    fa = parsed_nicolay_citizenship.get("FinalAnswer",{}).get("Text","")[:250]
    print(f"FinalAnswer preview:\n{fa}")
except json.JSONDecodeError as e:
    print(f"JSON ERROR: {e}")
    parsed_nicolay_citizenship = None

log_contamination("S3_citizenship_nicolay", chunk_ids=Q_NEW_MATCH_IDS, queries=[Q_NEW_QUERY])


In [ ]:
# 3.7  Verify Nicolay citizenship example — manual checklist

if parsed_nicolay_citizenship:
    print("=== INITIAL ANSWER REVIEW ===")
    print(json.dumps(parsed_nicolay_citizenship.get("Initial Answer Review",{}), indent=2)[:700])
    print("\n=== MATCH 1 ANALYSIS ===")
    print(json.dumps(parsed_nicolay_citizenship.get("Match Analysis",{}).get("Match 1",{}), indent=2)[:500])
    print("\n=== FINAL ANSWER ===")
    print(parsed_nicolay_citizenship.get("FinalAnswer",{}).get("Text","")[:500])
    print("\n=== MODEL FEEDBACK ===")
    print(json.dumps(parsed_nicolay_citizenship.get("Model Feedback",{}), indent=2)[:400])

print("\n--- REVIEW CHECKLIST ---")
for item in [
    "[ ] IAR critiques 'progressive evolution' framing explicitly",
    "[ ] Anti-slavery ≠ citizenship distinction stated in IAR or Match Analysis",
    "[ ] Colonization/emigration period acknowledged",
    "[ ] Late-war suffrage framed as conditional (not universal)",
    "[ ] FinalAnswer quotes accurate — verify against corpus text",
    "[ ] Lincoln-Douglas gap noted in Model Feedback",
]: print(item)


In [ ]:
# 3.8  Approve Nicolay citizenship example

APPROVE_S3_NICOLAY = True   # set False if checklist items failed
EDITS_S3_NICOLAY   = ""

if APPROVE_S3_NICOLAY and parsed_nicolay_citizenship:
    approve_nicolay({
        "messages": [
            {"role": "system",    "content": NICOLAY_SYSTEM_PROMPT},
            {"role": "user",      "content": q_new_input},
            {"role": "assistant", "content": json.dumps(parsed_nicolay_citizenship)},
        ],
        "_meta": {"task": "citizenship_q_new", "section": "S3_nicolay",
                  "edits": EDITS_S3_NICOLAY, "chunk_ids": Q_NEW_MATCH_IDS},
    })

print(f"Approved Nicolay examples so far: {len(approved_nicolay)}")


---
## Section 4 — Hay: Refusal Recalibration

**Gap (dev log #44, failure mode 2 — Q0 Russia):** Hay's refusal heuristic
over-fires on plausible-but-surprising corpus content (Russia, Cuba, China,
prize money). The existing dataset already has four corrective examples drafted
in `hay_new_examples.jsonl`; the fixed dataset incorporates them.

This section generates 2-3 additional positive examples for the same pattern
category — diplomatic / administrative Annual Message content that might trigger
a false refusal — and adds a quick audit of how many refusal examples currently
exist vs. positive examples.


In [ ]:
# 4.1  Audit current refusal/positive balance in dataset

refusal_count  = 0
positive_count = 0

for ex in HAY_FT:
    asst = next(m for m in ex["messages"] if m["role"]=="assistant")
    parsed = json.loads(asst["content"])
    kw = parsed.get("weighted_keywords") or {}
    if not kw:
        refusal_count += 1
    else:
        positive_count += 1

print(f"Dataset balance:")
print(f"  Positive (keywords present): {positive_count}")
print(f"  Refusal  (no keywords):      {refusal_count}")
print(f"  Ratio: 1 refusal per {positive_count//refusal_count if refusal_count else '?'} positive")

# List refusal examples for reference
print("\nRefusal examples (no keywords):")
for i, ex in enumerate(HAY_FT):
    asst = next(m for m in ex["messages"] if m["role"]=="assistant")
    parsed = json.loads(asst["content"])
    if not (parsed.get("weighted_keywords") or {}):
        user_q = next(m["content"] for m in ex["messages"] if m["role"]=="user")
        print(f"  Line {i:2d}: {user_q[:70]}")


In [ ]:
# 4.2  Find plausible-but-surprising corpus content for positive examples

SURPRISING_TERMS = ["liberia", "mexico", "nicaragua", "prize", "consular",
                    "tonnage", "coinage", "postmaster", "nicaragua", "cuba"]

print("Surprising-but-genuine corpus content:\n")
for term in SURPRISING_TERMS:
    hits = [
        f"{item['text_id']} ({item.get('source','')[:45]})"
        for item in corpus
        if re.search(r'\b' + re.escape(term) + r'\b',
                     item.get("full_text",""), re.IGNORECASE)
    ]
    if hits: print(f"  '{term}': {hits[:3]}")


In [ ]:
# 4.3  Set chunks and generate — pick 2 topics from 4.2 output

REFUSAL_CHUNKS = {
    "surprising_1": "Text #: XX",   # <-- fill in
    "surprising_2": "Text #: XX",   # <-- fill in
}

REFUSAL_QUERIES = {
    "surprising_1": "Does Lincoln discuss [TOPIC] in any of his speeches?",
    "surprising_2": "What did Lincoln say about [TOPIC] in his Annual Messages?",
}
# Replace [TOPIC] with the actual topic before running.

REFUSAL_SYSTEM = """\
You are building fine-tuning examples for Hay. This batch targets queries that
SEEM out-of-scope but have genuine corpus answers.

The model must ATTEMPT RETRIEVAL, not refuse. Rules:
- weighted_keywords MUST be non-empty; include the surprising topic term at weight 15-20
- initial_answer should note the topic sounds unexpected but assert the corpus may
  contain references: 'While this may seem an unusual topic for Lincoln's speeches,
  Annual Messages and diplomatic addresses may contain relevant references...'
- NEVER produce empty weighted_keywords for these queries

Output valid JSON only. FORMAT EXAMPLE:
"""

refusal_candidates = {}

for task, query in REFUSAL_QUERIES.items():
    chunk = get_chunk(REFUSAL_CHUNKS.get(task,""))
    if not chunk: print(f"SKIP {task}"); continue
    prompt = [
        {"role": "system", "content": REFUSAL_SYSTEM + HAY_PRIMER_OUTPUT},
        {"role": "user",   "content":
            f"Query: {query}\n\nGrounding passage:\n---\n{chunk['full_text'][:1200]}\n---"},
    ]
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL, messages=prompt, temperature=0.4, max_tokens=500)
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        kw = parsed.get("weighted_keywords") or {}
        refusal_candidates[task] = {"query": query, "output": parsed, "chunk_id": REFUSAL_CHUNKS[task]}
        print(f"  OK  keywords={'PRESENT' if kw else '⚠ EMPTY — do not approve'}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}")

log_contamination("S4_refusal", chunk_ids=list(REFUSAL_CHUNKS.values()),
                  queries=list(REFUSAL_QUERIES.values()))


In [ ]:
# 4.4  Verify and approve refusal recalibration examples

for task, data in refusal_candidates.items():
    verify_hay_candidate(task, data["query"], data["output"], data["chunk_id"])
    if not (data["output"].get("weighted_keywords") or {}):
        print("  ⚠ EMPTY KEYWORDS — do not approve this example")

APPROVE_S4 = {k: True for k in refusal_candidates}

for task, approve in APPROVE_S4.items():
    if not approve or task not in refusal_candidates: continue
    data = refusal_candidates[task]
    if not (data["output"].get("weighted_keywords") or {}):
        print(f"  SKIPPED {task}: empty keywords"); continue
    approve_hay(wrap_hay_ft(data["query"], data["output"],
                            task=task, section="S4", chunk_id=data["chunk_id"]))

print(f"\nApproved Hay examples so far: {len(approved_hay)}")


---
## Section 5 — Nicolay: Hypothesis Testing (Corpus > Initial Answer)

**Gap (dev log #73, failure mode 1 — Q7 voter figures):**
Nicolay confirms Hay's `initial_answer` rather than correcting it when
retrieved evidence contradicts it.

**What new examples must model:**
1. `Initial Answer Review` identifies a specific claim in `initial_answer` as
   unverified and flags it for corpus check
2. `Match Analysis` for the key match explicitly notes contradiction:
   *"The corpus text contradicts the initial answer on this point"*
3. `FinalAnswer` uses the corpus-sourced figure, not the Hay seed
4. The CoT states clearly: *"corpus evidence takes priority over initial answer"*

**Strategy:** Construct inputs with a *deliberately wrong* specific figure in
`initial_answer` and a Match 1 that contains the correct answer.


In [ ]:
# 5.1  Build Q7-type synthetic input (voter figures — known failure case)

chunk_77 = get_chunk("Text #: 77")
if chunk_77:
    print("Chunk 77 (Fourth Annual Message):")
    print(chunk_77["full_text"][:600])
else:
    print("⚠ Text #: 77 not found — update chunk ID")


In [ ]:
# 5.2  Build additional factual-contradiction scenario
# After running 5.1, pick a second chunk with verifiable specific data.

HT_SCENARIOS = [
    {
        "task":          "voter_figures_q7",
        "query":         "Lincoln noted in his speeches how many voters from Kansas and Nevada "
                         "participated in the 1864 presidential election. What was this number?",
        "wrong_initial": ("My initial hypothesis is that Lincoln reported approximately 10,000 "
                          "voters from Kansas and 1,000 from Nevada — retrieval should verify "
                          "these figures against the actual corpus text."),
        "target_id":     "Text #: 77",
    },
    {
        "task":          "factual_2",
        "query":         "What specific figure did Lincoln report regarding [TOPIC]?",   # customize
        "wrong_initial": ("My initial hypothesis is that Lincoln reported approximately "
                          "[WRONG_FIGURE] — retrieval should confirm this against the corpus."),
        "target_id":     "Text #: XX",   # <-- fill in
    },
]

ht_inputs = {}
for sc in HT_SCENARIOS:
    chunk = get_chunk(sc["target_id"])
    if not chunk: print(f"SKIP {sc['task']}: chunk not found"); continue
    # Build 3-match input: target chunk + up to 2 neighbors
    sf, pos = chunk.get("source_file"), chunk.get("position",0)
    match_chunks = [chunk]
    for offset in [1,-1]:
        neighbor = adjacency_index.get((sf, pos+offset))
        if neighbor and len(match_chunks) < 3:
            match_chunks.append(neighbor)
    ht_inputs[sc["task"]] = {
        "query":   sc["query"],
        "initial": sc["wrong_initial"],
        "input":   build_nicolay_input(sc["query"], sc["wrong_initial"], match_chunks),
        "chunk_id":sc["target_id"],
    }
    print(f"Built input: {sc['task']}  ({len(match_chunks)} matches)")


In [ ]:
# 5.3  Generate Nicolay hypothesis-testing CoT examples

HT_SYSTEM = """\
You are generating a fine-tuning example for Nicolay, a historical RAG model.
This example must model HYPOTHESIS TESTING: using corpus evidence to correct
a wrong initial answer.

CRITICAL REQUIREMENTS:

1. Initial Answer Review MUST identify the specific uncertain claim and flag it:
   'The initial answer states [X], but this specific figure requires corpus verification.'

2. Match Analysis for Match 1 MUST explicitly compare corpus text to initial answer:
   'The corpus text states [Y], which contradicts/differs from the initial answer's [X].
    Corpus evidence takes priority.'

3. FinalAnswer MUST:
   - Use the corpus figure, NOT the initial answer's figure
   - Include a direct verbatim quote from Match 1 with citation
   - Not repeat the wrong figure from the initial answer anywhere

4. Model Feedback should note: 'The initial_answer field is a retrieval hypothesis
   and should always be compared against FinalAnswer — discrepancies indicate
   the initial answer was corrected by corpus evidence.'

Output valid JSON only. Follow format example exactly.
"""

CORRECTION_PHRASES = ["contradicts","corrects","differs from","corpus evidence",
                      "takes priority","corpus text states","initial answer states"]

ht_candidates = {}

for task, data in ht_inputs.items():
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content":
                HT_SYSTEM + "\n\nFORMAT EXAMPLE:\n" + NICOLAY_PRIMER_OUTPUT[:2500]},
            {"role": "user", "content": data["input"]},
        ],
        temperature=0.3, max_tokens=2800,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        ht_candidates[task] = {**data, "output": parsed}
        # Quick check
        full_text = json.dumps(parsed).lower()
        found = [p for p in CORRECTION_PHRASES if p in full_text]
        print(f"  OK  correction_phrases_found={found if found else '⚠ NONE'}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}")

log_contamination("S5_hypothesis_testing",
                  chunk_ids=[d["chunk_id"] for d in ht_inputs.values()],
                  queries=[d["query"] for d in ht_inputs.values()])


In [ ]:
# 5.4  Verify hypothesis testing examples

for task, data in ht_candidates.items():
    output = data.get("output",{})
    print(f"\n{'='*62}\nTASK: {task}")
    iar = json.dumps(output.get("Initial Answer Review",{}))
    print(f"IAR (200 chars): {iar[:200]}")
    m1  = output.get("Match Analysis",{}).get("Match 1",{})
    print(f"Match 1 relevance : {m1.get('Relevance Assessment','?')}")
    ra  = m1.get("Relevance Assessment","")
    fa  = output.get("FinalAnswer",{}).get("Text","")
    print(f"FinalAnswer (200): {fa[:200]}")

    # Check wrong figure does NOT appear in FinalAnswer
    # (specific to Q7: figures 10,000 and 1,000 should be absent)
    wrong_figures = ["10,000", "1,000 from nevada", "10000", "1000"]
    found_wrong = [f for f in wrong_figures if f.lower() in fa.lower()]
    if found_wrong: print(f"⚠ WRONG FIGURE IN FINAL ANSWER: {found_wrong}")
    else:           print("✓ Wrong figure not repeated in FinalAnswer")

    print("\nChecklist:")
    print("[ ] IAR identifies specific uncertain claim")
    print("[ ] Match 1 analysis explicitly contradicts/corrects initial answer")
    print("[ ] FinalAnswer uses corpus figure with direct quote")
    print("[ ] Model Feedback notes initial_answer is hypothesis")


In [ ]:
# 5.5  Approve hypothesis testing examples

APPROVE_S5 = {k: True for k in ht_candidates}

for task, approve in APPROVE_S5.items():
    if not approve or task not in ht_candidates: continue
    data = ht_candidates[task]
    approve_nicolay({
        "messages": [
            {"role": "system",    "content": NICOLAY_SYSTEM_PROMPT},
            {"role": "user",      "content": data["input"]},
            {"role": "assistant", "content": json.dumps(data["output"])},
        ],
        "_meta": {"task": task, "section": "S5", "chunk_id": data["chunk_id"]},
    })

print(f"Approved Nicolay examples so far: {len(approved_nicolay)}")


---
## Section 6 — Nicolay: Single-Document Selective Citation

**Gap (failure mode 4 — scope drift):** For queries targeting a specific speech,
Nicolay incorporates all three retrieved matches even when Matches 2-3 are from
different speeches, producing a career survey instead of a focused analysis.

**What new examples must model:**
- `Match Analysis` explicitly downgrades off-target matches:
  *"This passage is from the First Inaugural, but the query asks about the
  Second Inaugural — Low relevance"*
- `Meta Analysis` names the single-document strategy explicitly
- `FinalAnswer` cites only the on-target match(es)


In [ ]:
# 6.1  Build single-document scope inputs
# Match 1 = target speech; Matches 2-3 = plausible but off-target

SCOPE_SCENARIOS = [
    {
        "task":       "second_inaugural_allusions",
        "query":      "What biblical allusions does Lincoln use in the Second Inaugural Address?",
        "target_id":  "Text #: XX",   # Second Inaugural chunk — fill in
        "off_ids":    ["Text #: XX", "Text #: XX"],  # First Inaugural, Peoria — fill in
    },
    {
        "task":       "gettysburg_structure",
        "query":      "How is the Gettysburg Address structured as a rhetorical argument?",
        "target_id":  "Text #: XX",   # Gettysburg chunk — fill in
        "off_ids":    ["Text #: XX", "Text #: XX"],  # Cooper Union, House Divided
    },
]

scope_inputs = {}
for sc in SCOPE_SCENARIOS:
    target = get_chunk(sc["target_id"])
    if not target: print(f"SKIP {sc['task']}: target chunk not found"); continue
    off_chunks = [get_chunk(tid) for tid in sc["off_ids"] if get_chunk(tid)]
    match_chunks = [target] + off_chunks[:2]
    scope_inputs[sc["task"]] = {
        "query":     sc["query"],
        "input":     build_nicolay_input(
            sc["query"],
            "My initial hypothesis addresses a specific rhetorical feature of the target speech. "
            "Retrieval should identify the relevant passages in that speech specifically.",
            match_chunks,
        ),
        "target_id": sc["target_id"],
        "off_ids":   sc["off_ids"],
    }
    print(f"Built: {sc['task']}  target={sc['target_id']}  off={sc['off_ids']}")


In [ ]:
# 6.2  Generate scope-correction Nicolay examples

SCOPE_SYSTEM = """\
You are generating a fine-tuning example for Nicolay, a historical RAG model.
This example must model SELECTIVE CITATION for a single-document query.

CRITICAL REQUIREMENTS:

1. Match Analysis MUST explicitly note when a match is from the wrong speech:
   'This passage is from [Speech X], but the query specifically asks about [Speech Y].
    Relevance: Low — this is from a different document than queried.'

2. Meta Analysis must identify this as a single-document task and state the strategy:
   'Focus on Match 1 only (correct document). Matches 2 and 3 are from different
    speeches and will not be cited in the final answer.'

3. FinalAnswer must cite ONLY the on-target match. It may briefly acknowledge
   that other passages were retrieved but were from different speeches.

4. Do NOT synthesize across all three matches simply because all three were retrieved.
   Correct behavior is recognizing when only one match is actually relevant.

Output valid JSON only. Follow format example exactly.
"""

scope_candidates = {}

for task, data in scope_inputs.items():
    print(f"Generating: {task} ...")
    resp = client.chat.completions.create(
        model=GENERATION_MODEL,
        messages=[
            {"role": "system", "content":
                SCOPE_SYSTEM + "\n\nFORMAT EXAMPLE:\n" + NICOLAY_PRIMER_OUTPUT[:2500]},
            {"role": "user", "content": data["input"]},
        ],
        temperature=0.3, max_tokens=2800,
    )
    raw = resp.choices[0].message.content.strip()
    try:
        parsed = json.loads(raw)
        scope_candidates[task] = {**data, "output": parsed}
        m2 = parsed.get("Match Analysis",{}).get("Match 2",{}).get("Relevance Assessment","?")
        m3 = parsed.get("Match Analysis",{}).get("Match 3",{}).get("Relevance Assessment","?")
        ok = "Low" in str(m2) and "Low" in str(m3)
        print(f"  OK  M2={m2}  M3={m3}  {'✓' if ok else '⚠ off-target not downgraded'}")
    except json.JSONDecodeError as e:
        print(f"  JSON ERROR: {e}")

log_contamination("S6_scope",
                  chunk_ids=[d["target_id"] for d in scope_inputs.values()],
                  queries=[d["query"] for d in scope_inputs.values()])


In [ ]:
# 6.3  Verify and approve scope examples

for task, data in scope_candidates.items():
    output = data.get("output",{})
    print(f"\n=== {task} ===")
    for mk in ["Match 1","Match 2","Match 3"]:
        m = output.get("Match Analysis",{}).get(mk,{})
        print(f"  {mk}: relevance={m.get('Relevance Assessment','?')} | source={m.get('Source','?')[:50]}")
    meta = output.get("Meta Analysis",{}).get("Strategy for Response Composition","")
    print(f"  Meta: {str(meta)[:180]}")
    fa = output.get("FinalAnswer",{}).get("Text","")[:200]
    print(f"  FA  : {fa}")
    refs = output.get("FinalAnswer",{}).get("References",[])
    print(f"  Refs: {refs}  (should only include target speech)")

APPROVE_S6 = {k: True for k in scope_candidates}

for task, approve in APPROVE_S6.items():
    if not approve or task not in scope_candidates: continue
    data = scope_candidates[task]
    approve_nicolay({
        "messages": [
            {"role": "system",    "content": NICOLAY_SYSTEM_PROMPT},
            {"role": "user",      "content": data["input"]},
            {"role": "assistant", "content": json.dumps(data["output"])},
        ],
        "_meta": {"task": task, "section": "S6", "chunk_id": data["target_id"]},
    })

print(f"\nApproved Nicolay examples so far: {len(approved_nicolay)}")


---
## Section 7 — Export & Contamination Report

In [ ]:
# 7.1  Summary counts by section

def count_by_section(approved_list):
    counts = {}
    for ex in approved_list:
        sec = ex.get("_meta",{}).get("section","?")
        counts[sec] = counts.get(sec,0) + 1
    return counts

print("=== APPROVED EXAMPLE COUNTS ===")
print(f"  Hay total    : {len(approved_hay)}")
print(f"  Nicolay total: {len(approved_nicolay)}")
print(f"\n  Hay by section    : {count_by_section(approved_hay)}")
print(f"  Nicolay by section: {count_by_section(approved_nicolay)}")
print(f"\n  Hay final total (existing + new): {len(HAY_FT)} + {len(approved_hay)} = {len(HAY_FT)+len(approved_hay)}")
print(f"  Nicolay final total: {len(nicolay_ft)} + {len(approved_nicolay)} = {len(nicolay_ft)+len(approved_nicolay)}")


In [ ]:
# 7.2  Schema validation before export

def validate_jsonl(examples, label):
    REQUIRED = {"initial_answer","weighted_keywords","year_keywords","text_keywords"}
    errors = []
    for i, ex in enumerate(examples):
        msgs = ex.get("messages",[])
        roles = [m.get("role") for m in msgs]
        if roles != ["system","user","assistant"]:
            errors.append(f"{label} line {i}: bad roles {roles}")
        asst = next((m for m in msgs if m["role"]=="assistant"), None)
        if asst:
            try:
                p = json.loads(asst["content"])
            except json.JSONDecodeError as e:
                errors.append(f"{label} line {i}: JSON error — {e}")
    return errors

# Validate new Hay examples (require 4-key schema)
strip = lambda exs: [{"messages": ex["messages"]} for ex in exs]

hay_errors = validate_jsonl(strip(approved_hay), "Hay-new")
nic_errors = []   # Nicolay schema varies — spot-check manually

if hay_errors:
    print("⚠ HAY VALIDATION ERRORS:")
    for e in hay_errors: print(f"  {e}")
else:
    print(f"✓ All {len(approved_hay)} new Hay examples valid")

print(f"✓ {len(approved_nicolay)} new Nicolay examples (validate manually via checklist)")


In [ ]:
# 7.3  Write output files

def write_jsonl(examples, path):
    with open(path,"w") as f:
        for ex in examples:
            f.write(json.dumps(ex) + "\n")
    print(f"  Wrote {len(examples)} examples → {path}")

# New-only files (for review before merging)
write_jsonl(strip(approved_hay),     "hay_new_generated.jsonl")
write_jsonl(strip(approved_nicolay), "nicolay_new_generated.jsonl")

# Merged files (ready for fine-tuning upload)
hay_merged     = HAY_FT + strip(approved_hay)
nicolay_merged = nicolay_ft + strip(approved_nicolay)

write_jsonl(hay_merged,     "hay_finetuning_data_v2.jsonl")   # corrected from hays_ typo
write_jsonl(nicolay_merged, "nicolay_finetuning_data_v2.jsonl")

print(f"\nMerged Hay    : {len(HAY_FT)} existing + {len(approved_hay)} new = {len(hay_merged)}")
print(f"Merged Nicolay: {len(nicolay_ft)} existing + {len(approved_nicolay)} new = {len(nicolay_merged)}")


In [ ]:
# 7.4  Contamination report

print("=== CONTAMINATION REPORT ===")
print("These chunk IDs and queries were used in fine-tuning generation.")
print("Exclude them from the formal benchmark run.\n")

used_chunks = sorted(set(c for c in contamination_log["chunk_ids_used"] if "XX" not in str(c)))
print("Chunk IDs used:")
for cid in used_chunks:
    chunk = get_chunk(cid)
    src   = chunk.get("source","?")[:55] if chunk else "NOT FOUND"
    print(f"  {cid}  |  {src}")

print("\nQueries used:")
for q in contamination_log["queries_used"]:
    print(f"  - {q[:95]}")

print(f"\nSections run: {contamination_log['sections_run']}")

with open("contamination_log.json","w") as f:
    json.dump(contamination_log, f, indent=2)
print("\nSaved: contamination_log.json")


In [ ]:
# 7.5  Optional: upload to OpenAI for fine-tuning
# Uncomment when ready. Review merged files and contamination report first.

# with open("hay_finetuning_data_v2.jsonl","rb") as f:  # corrected from hays_ typo
#     hay_upload = client.files.create(file=f, purpose="fine-tune")
#     print(f"Hay file uploaded: {hay_upload.id}")

# with open("nicolay_finetuning_data_v2.jsonl","rb") as f:
#     nic_upload = client.files.create(file=f, purpose="fine-tune")
#     print(f"Nicolay file uploaded: {nic_upload.id}")

# hay_job = client.fine_tuning.jobs.create(
#     training_file=hay_upload.id,
#     model="gpt-4o-mini-2024-07-18",
#     suffix="hay-gpt4o-v2",  # corrected from hays- typo
# )
# nic_job = client.fine_tuning.jobs.create(
#     training_file=nic_upload.id,
#     model="gpt-4o-mini-2024-07-18",
#     suffix="nicolay-gpt4o-v2",
# )
# print(f"Hay job   : {hay_job.id}")
# print(f"Nicolay job: {nic_job.id}")

print("Fine-tuning upload cells ready — uncomment to run.")
